# Burmese Automatic Speech Recognition Using Kaldi


This project focuses on developing a **Burmese Automatic Speech Recognition (ASR)** system using the **Kaldi Speech Recognition Toolkit**. The project implements a traditional GMM-HMM based ASR pipeline, including speech data preparation, MFCC feature extraction, pronunciation lexicon construction, acoustic model training, language model development, and speech decoding. Different acoustic and language models are compared using **Word Error Rate (WER)** and **Sentence Error Rate (SER)**. The best-performing completed system is the **Tri3 acoustic model with a Trigram language model**, achieving a WER of **96.06%** on the test dataset.


Automatic Speech Recognition (ASR) is a technology that converts spoken language into written text. In this project, we explore how ASR can be implemented for the **Burmese language** using Kaldi. The experiments investigate the effects of different acoustic models, including **Tri1, Tri2, and Tri3**, and different language models, including **Bigram and Trigram** models. The project also includes an fMLLR experiment to examine speaker adaptation. Through these experiments, we analyze the performance of each system and identify the best-performing configuration.

GitHub Repository: [ASR-Mini-Project](https://github.com/ThantSinTun009/ASR-Mini-Project/tree/main)


## Project Pipeline

The Burmese ASR system follows the pipeline below:

```text
Burmese Speech Dataset
        ↓
1. Data Preparation & Validation
        ↓
2. MFCC Feature Extraction
        ↓
3. Pronunciation Lexicon Preparation
        ↓
4. Language Model Construction
   ├── Unigram
   ├── Bigram
   └── Trigram
        ↓
5. Acoustic Model Training
   ├── Tri1
   ├── Tri2 (LDA + MLLT)
   └── Tri3 (SAT)
        ↓
6. HCLG Decoding Graph Construction
        ↓
7. Speech Decoding
        ↓
8. Model Evaluation
   ├── WER
   ├── SER
   └── Substitution / Deletion / Insertion
        ↓
9. Final Model Comparison
        ↓
Best System: Tri3 + Trigram
WER: 96.06%
```

### Pipeline Summary

> **Data → Features → Lexicon → Language Model → Acoustic Model → Decoding → Evaluation → Comparison**


In [ ]:
%pwd

'/home/thant_syn/kaldi/egs/burmese_asr'

In [ ]:
import os

PROJECT_DIR = os.getcwd()

print(PROJECT_DIR)

/home/thant_syn/kaldi/egs/burmese_asr


In [ ]:
!ls

ASR_Mini_Project.ipynb	conf  echo   recordings  utils
cmd.sh			data  local  steps


In [ ]:
!ls data

test  train  train_test.zip


In [ ]:
!ls recordings/

AungKhantMyat	MyintThuSoe	ThantSinTun  WaiYanHtetAung
HtooEaindraTin	SoeThandarTint	ThidaAye     recording.zip


## Data Preparation & Validation

In [ ]:
###

import os

recording_dir = "recordings"

for speaker in os.listdir(recording_dir):
    speaker_path = os.path.join(recording_dir, speaker)

    if os.path.isdir(speaker_path):
        wav_files = [
            f for f in os.listdir(speaker_path)
            if f.lower().endswith(".wav")
        ]

        print(f"{speaker}: {len(wav_files)} WAV files")

ThidaAye: 750 WAV files
HtooEaindraTin: 150 WAV files
MyintThuSoe: 750 WAV files
AungKhantMyat: 753 WAV files
WaiYanHtetAung: 750 WAV files
SoeThandarTint: 749 WAV files
ThantSinTun: 750 WAV files


In [ ]:
####

!ls -lh data/train

total 1.3M
-rw-r--r-- 1 thant_syn thant_syn 101K Sep 10 06:43 spk2utt
-rw-r--r-- 1 thant_syn thant_syn 198K Sep 10 06:58 text
-rw-r--r-- 1 thant_syn thant_syn 146K Sep 10 06:59 utt2spk
-rw-r--r-- 1 thant_syn thant_syn 393K Sep 10 07:00 wav.scp
-rw-r--r-- 1 thant_syn thant_syn 393K Sep 10 06:59 wav.scp.new


In [ ]:
!head -10 data/train/text

ThantSinTun_20260901_091117 ၀
ThantSinTun_20260901_091140 ၁
ThantSinTun_20260901_091159 ၂
ThantSinTun_20260901_091242 ၃
ThantSinTun_20260901_091305 ၄
ThantSinTun_20260901_091455 ၅
ThantSinTun_20260901_091530 ၆
ThantSinTun_20260901_091550 ၇
ThantSinTun_20260901_091609 ၈
ThantSinTun_20260901_091626 ၉


In [ ]:
!head -10 data/train/utt2spk

MyintThuSoe_Rec1_20260907_191704 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191712 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191719 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191726 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191733 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191741 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191835 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191844 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191851 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191858 MyintThuSoe_Rec1


In [ ]:
!head -10 data/train/wav.scp.new

ThantSinTun_20260901_091117 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091117.wav
ThantSinTun_20260901_091140 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091140.wav
ThantSinTun_20260901_091159 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091159.wav
ThantSinTun_20260901_091242 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091242.wav
ThantSinTun_20260901_091305 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091305.wav
ThantSinTun_20260901_091455 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091455.wav
ThantSinTun_20260901_091530 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091530.wav
ThantSinTun_20260901_091550 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091550.wav
ThantSinTun_20260901_091609 /hom

In [ ]:
# convert to standard Kaldi name

!cp data/train/wav.scp.new data/train/wav.scp
!cp data/test/wav.scp.new data/test/wav.scp

# !cp data/test/utt2spk.txt data/test/utt2spk
# !cp data/train/utt2spk.txt data/train/utt2spk

# !cp data/test/text.txt data/test/text
# !cp data/train/text.txt data/train/text

In [ ]:
!tree data/train

data/train
├── spk2utt
├── text
├── utt2spk
├── wav.scp
└── wav.scp.new

1 directory, 5 files


#### Generate spk2utt

In [ ]:
!rm data/train/spk2utt

In [ ]:
!utils/utt2spk_to_spk2utt.pl \
    data/train/utt2spk \
    > data/train/spk2utt

In [ ]:
!ls data/train

spk2utt  text  utt2spk	wav.scp  wav.scp.new


In [ ]:
!head data/train/utt2spk

MyintThuSoe_Rec1_20260907_191704 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191712 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191719 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191726 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191733 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191741 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191835 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191844 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191851 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191858 MyintThuSoe_Rec1


In [ ]:
!head data/train/spk2utt

MyintThuSoe_Rec1 MyintThuSoe_Rec1_20260907_191704 MyintThuSoe_Rec1_20260907_191712 MyintThuSoe_Rec1_20260907_191719 MyintThuSoe_Rec1_20260907_191726 MyintThuSoe_Rec1_20260907_191733 MyintThuSoe_Rec1_20260907_191741 MyintThuSoe_Rec1_20260907_191835 MyintThuSoe_Rec1_20260907_191844 MyintThuSoe_Rec1_20260907_191851 MyintThuSoe_Rec1_20260907_191858 MyintThuSoe_Rec1_20260907_191908 MyintThuSoe_Rec1_20260907_191917 MyintThuSoe_Rec1_20260907_191925 MyintThuSoe_Rec1_20260907_191934 MyintThuSoe_Rec1_20260907_191942 MyintThuSoe_Rec1_20260907_191950 MyintThuSoe_Rec1_20260907_191959 MyintThuSoe_Rec1_20260907_192007 MyintThuSoe_Rec1_20260907_192015 MyintThuSoe_Rec1_20260907_192023 MyintThuSoe_Rec1_20260907_192030 MyintThuSoe_Rec1_20260907_192037 MyintThuSoe_Rec1_20260907_192044 MyintThuSoe_Rec1_20260907_192051 MyintThuSoe_Rec1_20260907_192059 MyintThuSoe_Rec1_20260907_192107 MyintThuSoe_Rec1_20260907_192113 MyintThuSoe_Rec1_20260907_192121 MyintThuSoe_Rec1_20260907_192132 MyintThuSoe_Rec1_20260907_

In [ ]:
!utils/utt2spk_to_spk2utt.pl \
    data/test/utt2spk \
    > data/test/spk2utt

In [ ]:
!tree data/test

data/test
├── spk2utt
├── text
├── utt2spk
├── wav.scp
└── wav.scp.new

1 directory, 5 files


In [ ]:
!head -n 10 data/test/utt2spk

AungKhantMyat_Rec1_20260907_232253 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232313 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232326 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232345 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232406 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232432 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232450 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232502 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232539 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232610 AungKhantMyat_Rec1


In [ ]:
### validate

!utils/validate_data_dir.sh --no-feats --non-print data/train

utils/validate_data_dir.sh: spk2utt and utt2spk do not seem to match
Unable to flush stdout: Broken pipe
cat: write error: Broken pipe


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== FILE COUNTS ==="
wc -l data/train/utt2spk data/train/spk2utt

echo ""
echo "=== FIRST 10 utt2spk ==="
head -10 data/train/utt2spk

echo ""
echo "=== FIRST 10 spk2utt ==="
head -10 data/train/spk2utt

echo ""
echo "=== RECREATE utt2spk FROM spk2utt ==="
utils/spk2utt_to_utt2spk.pl data/train/spk2utt > /tmp/recreated_utt2spk

echo ""
echo "=== FIRST DIFFERENCES ==="
diff -u data/train/utt2spk /tmp/recreated_utt2spk | head -40 || true

echo ""
echo "=== SORTED COMPARISON ==="
diff -u \
  <(sort data/train/utt2spk) \
  <(sort /tmp/recreated_utt2spk) | head -40 || true

=== FILE COUNTS ===
  3057 data/train/utt2spk
    12 data/train/spk2utt
  3069 total

=== FIRST 10 utt2spk ===
MyintThuSoe_Rec1_20260907_191704 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191712 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191719 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191726 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191733 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191741 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191835 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191844 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191851 MyintThuSoe_Rec1
MyintThuSoe_Rec1_20260907_191858 MyintThuSoe_Rec1

=== FIRST 10 spk2utt ===
MyintThuSoe_Rec1 MyintThuSoe_Rec1_20260907_191704 MyintThuSoe_Rec1_20260907_191712 MyintThuSoe_Rec1_20260907_191719 MyintThuSoe_Rec1_20260907_191726 MyintThuSoe_Rec1_20260907_191733 MyintThuSoe_Rec1_20260907_191741 MyintThuSoe_Rec1_20260907_191835 MyintThuSoe_Rec1_20260907_191844 MyintThuSoe_Rec1_20260907_191851 MyintThuSoe_Rec1_20260907_191858 MyintThuSoe_Rec1

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

utils/spk2utt_to_utt2spk.pl data/train/spk2utt > /tmp/recreated_utt2spk

echo "=== COUNTS ==="
echo "utt2spk:"
wc -l < data/train/utt2spk
echo "recreated from spk2utt:"
wc -l < /tmp/recreated_utt2spk

echo ""
echo "=== UTTERANCES IN utt2spk BUT MISSING FROM spk2utt ==="
comm -23 \
  <(awk '{print $1}' data/train/utt2spk | sort) \
  <(awk '{print $1}' /tmp/recreated_utt2spk | sort) | head -100

echo ""
echo "=== NUMBER MISSING ==="
comm -23 \
  <(awk '{print $1}' data/train/utt2spk | sort) \
  <(awk '{print $1}' /tmp/recreated_utt2spk | sort) | wc -l

echo ""
echo "=== SPEAKERS IN utt2spk ==="
cut -d' ' -f2 data/train/utt2spk | sort | uniq -c

echo ""
echo "=== SPEAKERS IN spk2utt ==="
awk '{print $1}' data/train/spk2utt | sort | uniq -c

=== COUNTS ===
utt2spk:
3057
recreated from spk2utt:
3057

=== UTTERANCES IN utt2spk BUT MISSING FROM spk2utt ===

=== NUMBER MISSING ===
0

=== SPEAKERS IN utt2spk ===
    150 MyintThuSoe_Rec1
    150 MyintThuSoe_Rec2
    150 MyintThuSoe_Rec3
    150 MyintThuSoe_Rec4
    150 MyintThuSoe_Rec5
    150 SoeThandarTint_Rec1
    149 SoeThandarTint_Rec2
    150 SoeThandarTint_Rec3
    150 SoeThandarTint_Rec4
    150 SoeThandarTint_Rec5
    808 ThantSinTun
    750 WaiYanHtetAung

=== SPEAKERS IN spk2utt ===
      1 MyintThuSoe_Rec1
      1 MyintThuSoe_Rec2
      1 MyintThuSoe_Rec3
      1 MyintThuSoe_Rec4
      1 MyintThuSoe_Rec5
      1 SoeThandarTint_Rec1
      1 SoeThandarTint_Rec2
      1 SoeThandarTint_Rec3
      1 SoeThandarTint_Rec4
      1 SoeThandarTint_Rec5
      1 ThantSinTun
      1 WaiYanHtetAung


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== COUNTS ==="
wc -l data/train/utt2spk data/train/wav.scp data/train/text

echo ""
echo "=== utt2spk → wav.scp: MISSING FROM WAV ==="
comm -23 \
  <(awk '{print $1}' data/train/utt2spk | sort) \
  <(awk '{print $1}' data/train/wav.scp | sort)

echo ""
echo "=== wav.scp → utt2spk: EXTRA IN WAV ==="
comm -13 \
  <(awk '{print $1}' data/train/utt2spk | sort) \
  <(awk '{print $1}' data/train/wav.scp | sort)

echo ""
echo "=== utt2spk → text: MISSING FROM TEXT ==="
comm -23 \
  <(awk '{print $1}' data/train/utt2spk | sort) \
  <(awk '{print $1}' data/train/text | sort)

echo ""
echo "=== text → utt2spk: EXTRA IN TEXT ==="
comm -13 \
  <(awk '{print $1}' data/train/utt2spk | sort) \
  <(awk '{print $1}' data/train/text | sort)

=== COUNTS ===
  3057 data/train/utt2spk
  3059 data/train/wav.scp
  3060 data/train/text
  9176 total

=== utt2spk → wav.scp: MISSING FROM WAV ===

=== wav.scp → utt2spk: EXTRA IN WAV ===

/
/home/thant_syn/kaldi/egs/burmese_asr/recordings/

=== utt2spk → text: MISSING FROM TEXT ===

=== text → utt2spk: EXTRA IN TEXT ===





In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== LAST 10 LINES OF utt2spk ==="
tail -10 data/train/utt2spk

echo ""
echo "=== FIRST 10 LINES OF wav.scp ==="
head -10 data/train/wav.scp

echo ""
echo "=== LAST 10 LINES OF wav.scp ==="
tail -10 data/train/wav.scp

echo ""
echo "=== FIRST 10 LINES OF text ==="
head -10 data/train/text

echo ""
echo "=== LINES WITH EMPTY/INVALID FIRST FIELD IN TEXT ==="
awk 'NF < 2 {print "LINE " NR ": [" $0 "]"}' data/train/text

echo ""
echo "=== TEXT LINE COUNT ==="
wc -l data/train/text

=== LAST 10 LINES OF utt2spk ===
WaiYanHtetAung_20260904_142212 WaiYanHtetAung
WaiYanHtetAung_20260904_142217 WaiYanHtetAung
WaiYanHtetAung_20260904_142222 WaiYanHtetAung
WaiYanHtetAung_20260904_142227 WaiYanHtetAung
WaiYanHtetAung_20260904_142231 WaiYanHtetAung
WaiYanHtetAung_20260904_142237 WaiYanHtetAung
WaiYanHtetAung_20260904_142242 WaiYanHtetAung
WaiYanHtetAung_20260904_142247 WaiYanHtetAung
WaiYanHtetAung_20260904_142252 WaiYanHtetAung
WaiYanHtetAung_20260904_142257 WaiYanHtetAung

=== FIRST 10 LINES OF wav.scp ===
ThantSinTun_20260901_091117 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091117.wav
ThantSinTun_20260901_091140 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091140.wav
ThantSinTun_20260901_091159 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091159.wav
ThantSinTun_20260901_091242 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== Cleaning train data ==="

# Remove blank lines from text
sed -i '/^[[:space:]]*$/d' data/train/text

# Remove the two malformed path-only lines from wav.scp
sed -i '/^[[:space:]]*\/home\/thant_syn\/kaldi\/egs\/burmese_asr\/recordings\/[[:space:]]*$/d' data/train/wav.scp
sed -i '/^[[:space:]]*\/[[:space:]]*$/d' data/train/wav.scp

# Sort wav.scp and text by utterance ID
sort -k1,1 data/train/wav.scp -o data/train/wav.scp
sort -k1,1 data/train/text -o data/train/text

# Regenerate spk2utt from the current utt2spk
sort -k1,1 data/train/utt2spk -o data/train/utt2spk
utils/utt2spk_to_spk2utt.pl data/train/utt2spk > data/train/spk2utt

echo ""
echo "=== COUNTS ==="
wc -l data/train/utt2spk data/train/wav.scp data/train/text

echo ""
echo "=== VALIDATE TRAIN ==="
utils/validate_data_dir.sh --no-feats --non-print data/train

=== Cleaning train data ===

=== COUNTS ===
  3057 data/train/utt2spk
  3058 data/train/wav.scp
  3057 data/train/text
  9172 total

=== VALIDATE TRAIN ===
utils/validate_data_dir.sh: spk2utt and utt2spk do not seem to match


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\necho "=== Cleaning train data ==="\n\n# Remove blank lines from text\nsed -i \'/^[[:space:]]*$/d\' data/train/text\n\n# Remove the two malformed path-only lines from wav.scp\nsed -i \'/^[[:space:]]*\\/home\\/thant_syn\\/kaldi\\/egs\\/burmese_asr\\/recordings\\/[[:space:]]*$/d\' data/train/wav.scp\nsed -i \'/^[[:space:]]*\\/[[:space:]]*$/d\' data/train/wav.scp\n\n# Sort wav.scp and text by utterance ID\nsort -k1,1 data/train/wav.scp -o data/train/wav.scp\nsort -k1,1 data/train/text -o data/train/text\n\n# Regenerate spk2utt from the current utt2spk\nsort -k1,1 data/train/utt2spk -o data/train/utt2spk\nutils/utt2spk_to_spk2utt.pl data/train/utt2spk > data/train/spk2utt\n\necho ""\necho "=== COUNTS ==="\nwc -l data/train/utt2spk data/train/wav.scp data/train/text\n\necho ""\necho "=== VALIDATE TRAIN ==="\nutils/validate_data_dir.sh --no-feats --non-print data/train\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. utt2spk / spk2utt consistency ==="

utils/spk2utt_to_utt2spk.pl data/train/spk2utt > /tmp/recreated_utt2spk

echo "utt2spk lines:"
wc -l < data/train/utt2spk

echo "recreated lines:"
wc -l < /tmp/recreated_utt2spk

echo ""
echo "First difference:"
diff -u data/train/utt2spk /tmp/recreated_utt2spk | head -20 || true

echo ""
echo "=== 2. EXTRA WAV ENTRY ==="

comm -13 \
  <(awk '{print $1}' data/train/utt2spk | sort) \
  <(awk '{print $1}' data/train/wav.scp | sort)

echo ""
echo "=== 3. LAST 5 WAV ENTRIES ==="
tail -5 data/train/wav.scp

echo ""
echo "=== 4. SPEAKER/UTTERANCE CHECK ==="
awk '{print $2}' data/train/utt2spk | sort | uniq -c

=== 1. utt2spk / spk2utt consistency ===
utt2spk lines:
3057
recreated lines:
3057

First difference:
--- data/train/utt2spk	2026-09-10 07:08:54.452587125 +0000
+++ /tmp/recreated_utt2spk	2026-09-10 07:09:36.087789125 +0000
@@ -1,3057 +1,3057 @@
-MyintThuSoe_Rec1_20260907_191704 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191712 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191719 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191726 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191733 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191741 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191835 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191844 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191851 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191858 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191908 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191917 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191925 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191934 MyintThuSoe_Rec1
-MyintThuSoe_Rec1_20260907_191942 MyintT

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Rebuild utt2spk from spk2utt ==="
utils/spk2utt_to_utt2spk.pl data/train/spk2utt > data/train/utt2spk.new
mv data/train/utt2spk.new data/train/utt2spk

echo
echo "=== 2. Check exact consistency ==="
utils/spk2utt_to_utt2spk.pl data/train/spk2utt > /tmp/recreated_utt2spk

if cmp -s data/train/utt2spk /tmp/recreated_utt2spk; then
    echo "PASS: utt2spk and spk2utt match exactly."
else
    echo "FAIL: They still do not match."
    diff -u data/train/utt2spk /tmp/recreated_utt2spk | head -20
fi

echo
echo "=== 3. Check counts ==="
wc -l data/train/utt2spk data/train/spk2utt data/train/wav.scp data/train/text

echo
echo "=== 4. Validate TRAIN ==="
utils/validate_data_dir.sh --no-feats --non-print data/train

=== 1. Rebuild utt2spk from spk2utt ===

=== 2. Check exact consistency ===
PASS: utt2spk and spk2utt match exactly.

=== 3. Check counts ===
  3057 data/train/utt2spk
    12 data/train/spk2utt
  3058 data/train/wav.scp
  3057 data/train/text
  9184 total

=== 4. Validate TRAIN ===


utils/validate_text.pl: The line for utterance MyintThuSoe_Rec1_20260907_191704 contains CR (0x0D) character
utils/validate_text.pl: ERROR: text file 'data/train/text' contains disallowed UTF-8 whitespace character(s)


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\necho "=== 1. Rebuild utt2spk from spk2utt ==="\nutils/spk2utt_to_utt2spk.pl data/train/spk2utt > data/train/utt2spk.new\nmv data/train/utt2spk.new data/train/utt2spk\n\necho\necho "=== 2. Check exact consistency ==="\nutils/spk2utt_to_utt2spk.pl data/train/spk2utt > /tmp/recreated_utt2spk\n\nif cmp -s data/train/utt2spk /tmp/recreated_utt2spk; then\n    echo "PASS: utt2spk and spk2utt match exactly."\nelse\n    echo "FAIL: They still do not match."\n    diff -u data/train/utt2spk /tmp/recreated_utt2spk | head -20\nfi\n\necho\necho "=== 3. Check counts ==="\nwc -l data/train/utt2spk data/train/spk2utt data/train/wav.scp data/train/text\n\necho\necho "=== 4. Validate TRAIN ==="\nutils/validate_data_dir.sh --no-feats --non-print data/train\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Lines with empty/invalid utterance ID in wav.scp ==="
awk 'NF < 2 {printf "LINE %d: [%s]\n", NR, $0}' data/train/wav.scp

echo
echo "=== 2. All blank/whitespace-only lines ==="
grep -n '^[[:space:]]*$' data/train/wav.scp || true

echo
echo "=== 3. Lines whose first field is not a normal utterance ID ==="
awk '$1 !~ /^[A-Za-z0-9_]+$/ {printf "LINE %d: [%s]\n", NR, $0}' data/train/wav.scp

echo
echo "=== 4. Count ==="
wc -l data/train/wav.scp

=== 1. Lines with empty/invalid utterance ID in wav.scp ===
]INE 1: [

=== 2. All blank/whitespace-only lines ===
1:

=== 3. Lines whose first field is not a normal utterance ID ===
]INE 1: [

=== 4. Count ===
3058 data/train/wav.scp


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== Remove blank lines from wav.scp ==="
sed -i '/^[[:space:]]*$/d' data/train/wav.scp

echo
echo "=== Counts ==="
wc -l data/train/utt2spk data/train/spk2utt data/train/wav.scp data/train/text

echo
echo "=== Validate TRAIN ==="
utils/validate_data_dir.sh --no-feats --non-print data/train

=== Remove blank lines from wav.scp ===

=== Counts ===
  3057 data/train/utt2spk
    12 data/train/spk2utt
  3057 data/train/wav.scp
  3057 data/train/text
  9183 total

=== Validate TRAIN ===
utils/validate_data_dir.sh: Successfully validated data-directory data/train


✅ What we fixed

- Removed invalid blank lines from utt2spk

- Fixed utt2spk ↔ spk2utt ordering/consistency

- Removed Windows CR (0x0D) characters from text

- Removed empty lines from text

- Removed the extra blank line from wav.scp

- Revalidated the complete training data

> Your training data is ready for the next Kaldi stage. 🚀

In [ ]:
!cd ~/kaldi/egs/burmese_asr

!utils/validate_data_dir.sh --no-feats --non-print data/test

file data/test/utt2spk has invalid newline at -e line 1, <> line 1527.


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Fix utt2spk line endings ==="
sed -i 's/\r$//' data/test/utt2spk

# Remove blank/whitespace-only lines
sed -i '/^[[:space:]]*$/d' data/test/utt2spk

echo
echo "=== 2. Check last few lines ==="
tail -5 data/test/utt2spk

echo
echo "=== 3. Validate TEST ==="
utils/validate_data_dir.sh --no-feats --non-print data/test


=== 1. Fix utt2spk line endings ===

=== 2. Check last few lines ===
ThidaAye_20260906_123630 ThidaAye
ThidaAye_20260906_123636 ThidaAye
ThidaAye_20260906_123644 ThidaAye
ThidaAye_20260906_123652 ThidaAye
ThidaAye_20260906_123705 ThidaAye
=== 3. Validate TEST ===


file data/test/utt2spk has invalid newline at -e line 1, <> line 1527.


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\necho "=== 1. Fix utt2spk line endings ==="\nsed -i \'s/\\r$//\' data/test/utt2spk\n\n# Remove blank/whitespace-only lines\nsed -i \'/^[[:space:]]*$/d\' data/test/utt2spk\n\necho\necho "=== 2. Check last few lines ==="\ntail -5 data/test/utt2spk\n\necho\necho "=== 3. Validate TEST ==="\nutils/validate_data_dir.sh --no-feats --non-print data/test\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Ensure final newline in utt2spk ==="
python3 - <<'PY'
from pathlib import Path

p = Path("data/test/utt2spk")
data = p.read_bytes()

if not data.endswith(b'\n'):
    p.write_bytes(data + b'\n')
    print("Added missing final newline.")
else:
    print("Final newline already exists.")
PY

echo
echo "=== 2. Remove blank lines ==="
sed -i '/^[[:space:]]*$/d' data/test/utt2spk

echo
echo "=== 3. Ensure final newline again ==="
python3 - <<'PY'
from pathlib import Path

p = Path("data/test/utt2spk")
data = p.read_bytes()

if not data.endswith(b'\n'):
    p.write_bytes(data + b'\n')
    print("Added final newline.")
else:
    print("Final newline OK.")
PY

echo
echo "=== 4. Validate TEST ==="
utils/validate_data_dir.sh --no-feats --non-print data/test

=== 1. Ensure final newline in utt2spk ===
Added missing final newline.

=== 2. Remove blank lines ===

=== 3. Ensure final newline again ===
Final newline OK.

=== 4. Validate TEST ===
utils/validate_data_dir.sh: file data/test/utt2spk is not sorted or has duplicates


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\necho "=== 1. Ensure final newline in utt2spk ==="\npython3 - <<\'PY\'\nfrom pathlib import Path\n\np = Path("data/test/utt2spk")\ndata = p.read_bytes()\n\nif not data.endswith(b\'\\n\'):\n    p.write_bytes(data + b\'\\n\')\n    print("Added missing final newline.")\nelse:\n    print("Final newline already exists.")\nPY\n\necho\necho "=== 2. Remove blank lines ==="\nsed -i \'/^[[:space:]]*$/d\' data/test/utt2spk\n\necho\necho "=== 3. Ensure final newline again ==="\npython3 - <<\'PY\'\nfrom pathlib import Path\n\np = Path("data/test/utt2spk")\ndata = p.read_bytes()\n\nif not data.endswith(b\'\\n\'):\n    p.write_bytes(data + b\'\\n\')\n    print("Added final newline.")\nelse:\n    print("Final newline OK.")\nPY\n\necho\necho "=== 4. Validate TEST ==="\nutils/validate_data_dir.sh --no-feats --non-print data/test\n'' returned non-zero exit status 1.

### Debug Duplicated data

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Duplicate utterance IDs ==="
awk '{print $1}' data/test/utt2spk | sort | uniq -d

echo
echo "=== 2. Number of duplicate IDs ==="
awk '{print $1}' data/test/utt2spk | sort | uniq -d | wc -l

echo
echo "=== 3. Check whether utt2spk is sorted ==="
if sort -k1,1 -c data/test/utt2spk 2>/dev/null; then
    echo "PASS: utt2spk is sorted"
else
    echo "FAIL: utt2spk is NOT sorted"
fi

echo
echo "=== 4. Count lines ==="
wc -l data/test/utt2spk

echo
echo "=== 5. First 5 lines ==="
head -5 data/test/utt2spk

echo
echo "=== 6. Last 5 lines ==="
tail -5 data/test/utt2spk

=== 1. Duplicate utterance IDs ===
ThidaAye_20260905_102030

=== 2. Number of duplicate IDs ===
1

=== 3. Check whether utt2spk is sorted ===
PASS: utt2spk is sorted

=== 4. Count lines ===
1527 data/test/utt2spk

=== 5. First 5 lines ===
AungKhantMyat_Rec1_20260907_232253 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232313 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232326 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232345 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232406 AungKhantMyat_Rec1

=== 6. Last 5 lines ===
ThidaAye_20260906_123630 ThidaAye
ThidaAye_20260906_123636 ThidaAye
ThidaAye_20260906_123644 ThidaAye
ThidaAye_20260906_123652 ThidaAye
ThidaAye_20260906_123705 ThidaAye


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== Duplicate records ==="
grep -n '^ThidaAye_20260905_102030 ' data/test/utt2spk

=== Duplicate records ===
1044:ThidaAye_20260905_102030 ThidaAye
1045:ThidaAye_20260905_102030 ThidaAye


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== Remove duplicate utterance ID ==="
awk '!seen[$1]++' data/test/utt2spk > data/test/utt2spk.new
mv data/test/utt2spk.new data/test/utt2spk

echo
echo "=== Check duplicate ==="
awk '{print $1}' data/test/utt2spk | sort | uniq -d

echo
echo "=== Count ==="
wc -l data/test/utt2spk

echo
echo "=== Validate TEST ==="
utils/validate_data_dir.sh --no-feats --non-print data/test

=== Remove duplicate utterance ID ===

=== Check duplicate ===

=== Count ===
1526 data/test/utt2spk

=== Validate TEST ===
utils/validate_data_dir.sh: spk2utt and utt2spk do not seem to match


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\necho "=== Remove duplicate utterance ID ==="\nawk \'!seen[$1]++\' data/test/utt2spk > data/test/utt2spk.new\nmv data/test/utt2spk.new data/test/utt2spk\n\necho\necho "=== Check duplicate ==="\nawk \'{print $1}\' data/test/utt2spk | sort | uniq -d\n\necho\necho "=== Count ==="\nwc -l data/test/utt2spk\n\necho\necho "=== Validate TEST ==="\nutils/validate_data_dir.sh --no-feats --non-print data/test\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Rebuild spk2utt from corrected utt2spk ==="
utils/utt2spk_to_spk2utt.pl data/test/utt2spk > data/test/spk2utt

echo
echo "=== 2. Check exact consistency ==="
utils/spk2utt_to_utt2spk.pl data/test/spk2utt > /tmp/test_recreated_utt2spk

if cmp -s data/test/utt2spk /tmp/test_recreated_utt2spk; then
    echo "PASS: utt2spk and spk2utt match exactly."
else
    echo "FAIL: They still differ."
    diff -u data/test/utt2spk /tmp/test_recreated_utt2spk | head -20
fi

echo
echo "=== 3. Counts ==="
wc -l data/test/utt2spk data/test/spk2utt data/test/wav.scp data/test/text

echo
echo "=== 4. Validate TEST ==="
utils/validate_data_dir.sh --no-feats --non-print data/test

=== 1. Rebuild spk2utt from corrected utt2spk ===

=== 2. Check exact consistency ===
PASS: utt2spk and spk2utt match exactly.

=== 3. Counts ===
  1526 data/test/utt2spk
     6 data/test/spk2utt
  1526 data/test/wav.scp
  1522 data/test/text
  4580 total

=== 4. Validate TEST ===
utils/validate_data_dir.sh: file data/test/text is not sorted or has duplicates


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\necho "=== 1. Rebuild spk2utt from corrected utt2spk ==="\nutils/utt2spk_to_spk2utt.pl data/test/utt2spk > data/test/spk2utt\n\necho\necho "=== 2. Check exact consistency ==="\nutils/spk2utt_to_utt2spk.pl data/test/spk2utt > /tmp/test_recreated_utt2spk\n\nif cmp -s data/test/utt2spk /tmp/test_recreated_utt2spk; then\n    echo "PASS: utt2spk and spk2utt match exactly."\nelse\n    echo "FAIL: They still differ."\n    diff -u data/test/utt2spk /tmp/test_recreated_utt2spk | head -20\nfi\n\necho\necho "=== 3. Counts ==="\nwc -l data/test/utt2spk data/test/spk2utt data/test/wav.scp data/test/text\n\necho\necho "=== 4. Validate TEST ==="\nutils/validate_data_dir.sh --no-feats --non-print data/test\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Duplicate utterance IDs in text ==="
awk '{print $1}' data/test/text | sort | uniq -d

echo
echo "=== 2. Number of duplicates ==="
awk '{print $1}' data/test/text | sort | uniq -d | wc -l

echo
echo "=== 3. Check sorting ==="
if sort -k1,1 -c data/test/text 2>/dev/null; then
    echo "PASS: text is sorted"
else
    echo "FAIL: text is NOT sorted"
fi

echo
echo "=== 4. Utterances in utt2spk but missing from text ==="
comm -23 \
  <(awk '{print $1}' data/test/utt2spk | sort -u) \
  <(awk '{print $1}' data/test/text | sort -u)

echo
echo "=== 5. Number missing from text ==="
comm -23 \
  <(awk '{print $1}' data/test/utt2spk | sort -u) \
  <(awk '{print $1}' data/test/text | sort -u) | wc -l

echo
echo "=== 6. Utterances in text but missing from utt2spk ==="
comm -13 \
  <(awk '{print $1}' data/test/utt2spk | sort -u) \
  <(awk '{print $1}' data/test/text | sort -u)

echo
echo "=== 7. Number extra in text ==="
comm -13 \
  <(awk '{print $1}' data/test/utt2spk | sort -u) \
  <(awk '{print $1}' data/test/text | sort -u) | wc -l

=== 1. Duplicate utterance IDs in text ===
ThidaAye_20260905_102030

=== 2. Number of duplicates ===
1

=== 3. Check sorting ===
PASS: text is sorted

=== 4. Utterances in utt2spk but missing from text ===
ThidaAye_20260905_004336
ThidaAye_20260905_004415
ThidaAye_20260905_004439
ThidaAye_20260905_004505
ThidaAye_20260905_004745

=== 5. Number missing from text ===
5

=== 6. Utterances in text but missing from utt2spk ===

=== 7. Number extra in text ===
0


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Check the 5 missing utterances in recordings ==="
for id in \
  ThidaAye_20260905_004336 \
  ThidaAye_20260905_004415 \
  ThidaAye_20260905_004439 \
  ThidaAye_20260905_004505 \
  ThidaAye_20260905_004745
do
    echo "--- $id ---"
    grep -R -n -- "$id" data/ recordings/ 2>/dev/null | head -5 || true
done

echo
echo "=== 2. Show the duplicate text entries ==="
grep -n '^ThidaAye_20260905_102030 ' data/test/text

echo
echo "=== 3. Check whether the 5 IDs exist in wav.scp ==="
grep -E '^ThidaAye_20260905_(004336|004415|004439|004505|004745) ' data/test/wav.scp || true

=== 1. Check the 5 missing utterances in recordings ===
--- ThidaAye_20260905_004336 ---
data/test/spk2utt:6:ThidaAye ThidaAye_20260905_004312 ThidaAye_20260905_004336 ThidaAye_20260905_004404 ThidaAye_20260905_004415 ThidaAye_20260905_004429 ThidaAye_20260905_004439 ThidaAye_20260905_004455 ThidaAye_20260905_004505 ThidaAye_20260905_004517 ThidaAye_20260905_004528 ThidaAye_20260905_004549 ThidaAye_20260905_004628 ThidaAye_20260905_004705 ThidaAye_20260905_004716 ThidaAye_20260905_004741 ThidaAye_20260905_004745 ThidaAye_20260905_004758 ThidaAye_20260905_004811 ThidaAye_20260905_004831 ThidaAye_20260905_004842 ThidaAye_20260905_004854 ThidaAye_20260905_004908 ThidaAye_20260905_004923 ThidaAye_20260905_004937 ThidaAye_20260905_004949 ThidaAye_20260905_005003 ThidaAye_20260905_005019 ThidaAye_20260905_005045 ThidaAye_20260905_005125 ThidaAye_20260905_005134 ThidaAye_20260905_005147 ThidaAye_20260905_005200 ThidaAye_20260905_005229 ThidaAye_20260905_005240 ThidaAye_20260905_005252 ThidaAy

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

# The 5 utterances that have no transcription
MISSING="ThidaAye_20260905_004336|ThidaAye_20260905_004415|ThidaAye_20260905_004439|ThidaAye_20260905_004505|ThidaAye_20260905_004745"

echo "=== 1. Remove 5 untranscribed utterances ==="

for file in data/test/utt2spk data/test/wav.scp data/test/text; do
    grep -v -E "^(${MISSING})([[:space:]]|$)" "$file" > "${file}.new"
    mv "${file}.new" "$file"
done

# Remove the duplicate text entry by keeping only the first occurrence
awk '!seen[$1]++' data/test/text > data/test/text.new
mv data/test/text.new data/test/text

# Rebuild spk2utt from the corrected utt2spk
utils/utt2spk_to_spk2utt.pl data/test/utt2spk > data/test/spk2utt

echo
echo "=== 2. Sort files ==="
sort -k1,1 data/test/utt2spk -o data/test/utt2spk
sort -k1,1 data/test/wav.scp -o data/test/wav.scp
sort -k1,1 data/test/text -o data/test/text

# Rebuild spk2utt one final time after sorting
utils/utt2spk_to_spk2utt.pl data/test/utt2spk > data/test/spk2utt

echo
echo "=== 3. Counts ==="
wc -l data/test/utt2spk data/test/spk2utt data/test/wav.scp data/test/text

echo
echo "=== 4. Validate TEST ==="
utils/validate_data_dir.sh --no-feats --non-print data/test

=== 1. Remove 5 untranscribed utterances ===

=== 2. Sort files ===

=== 3. Counts ===
  1521 data/test/utt2spk
     6 data/test/spk2utt
  1521 data/test/wav.scp
  1521 data/test/text
  4569 total

=== 4. Validate TEST ===
utils/validate_data_dir.sh: file data/test/wav.scp is not sorted or has duplicates


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\n# The 5 utterances that have no transcription\nMISSING="ThidaAye_20260905_004336|ThidaAye_20260905_004415|ThidaAye_20260905_004439|ThidaAye_20260905_004505|ThidaAye_20260905_004745"\n\necho "=== 1. Remove 5 untranscribed utterances ==="\n\nfor file in data/test/utt2spk data/test/wav.scp data/test/text; do\n    grep -v -E "^(${MISSING})([[:space:]]|$)" "$file" > "${file}.new"\n    mv "${file}.new" "$file"\ndone\n\n# Remove the duplicate text entry by keeping only the first occurrence\nawk \'!seen[$1]++\' data/test/text > data/test/text.new\nmv data/test/text.new data/test/text\n\n# Rebuild spk2utt from the corrected utt2spk\nutils/utt2spk_to_spk2utt.pl data/test/utt2spk > data/test/spk2utt\n\necho\necho "=== 2. Sort files ==="\nsort -k1,1 data/test/utt2spk -o data/test/utt2spk\nsort -k1,1 data/test/wav.scp -o data/test/wav.scp\nsort -k1,1 data/test/text -o data/test/text\n\n# Rebuild spk2utt one final time after sorting\nutils

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Duplicate WAV utterance IDs ==="
awk '{print $1}' data/test/wav.scp | sort | uniq -d

echo
echo "=== 2. Number of duplicates ==="
awk '{print $1}' data/test/wav.scp | sort | uniq -d | wc -l

echo
echo "=== 3. Check sorting ==="
if sort -k1,1 -c data/test/wav.scp 2>/dev/null; then
    echo "PASS: wav.scp is sorted"
else
    echo "FAIL: wav.scp is NOT sorted"
fi

echo
echo "=== 4. Find suspicious lines ==="
awk 'NF != 2 {printf "LINE %d: [%s]\n", NR, $0}' data/test/wav.scp

echo
echo "=== 5. First 5 lines ==="
head -5 data/test/wav.scp

echo
echo "=== 6. Last 5 lines ==="
tail -5 data/test/wav.scp

=== 1. Duplicate WAV utterance IDs ===
ThidaAye_20260905_102030

=== 2. Number of duplicates ===
1

=== 3. Check sorting ===
PASS: wav.scp is sorted

=== 4. Find suspicious lines ===

=== 5. First 5 lines ===
AungKhantMyat_Rec1_20260907_232253 /home/thant_syn/kaldi/egs/burmese_asr/recordings/AungKhantMyat/AungKhantMyat_Rec1_20260907_232253.wav
AungKhantMyat_Rec1_20260907_232313 /home/thant_syn/kaldi/egs/burmese_asr/recordings/AungKhantMyat/AungKhantMyat_Rec1_20260907_232313.wav
AungKhantMyat_Rec1_20260907_232326 /home/thant_syn/kaldi/egs/burmese_asr/recordings/AungKhantMyat/AungKhantMyat_Rec1_20260907_232326.wav
AungKhantMyat_Rec1_20260907_232345 /home/thant_syn/kaldi/egs/burmese_asr/recordings/AungKhantMyat/AungKhantMyat_Rec1_20260907_232345.wav
AungKhantMyat_Rec1_20260907_232406 /home/thant_syn/kaldi/egs/burmese_asr/recordings/AungKhantMyat/AungKhantMyat_Rec1_20260907_232406.wav

=== 6. Last 5 lines ===
ThidaAye_20260906_123624 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThidaAy

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== Duplicate WAV entries ==="
grep -n '^ThidaAye_20260905_102030 ' data/test/wav.scp

=== Duplicate WAV entries ===
1039:ThidaAye_20260905_102030 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThidaAye/ThidaAye_20260905_102030.wav
1040:ThidaAye_20260905_102030 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThidaAye/ThidaAye_20260905_102030.wav


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== Remove duplicate WAV entry ==="
awk '!seen[$1]++' data/test/wav.scp > data/test/wav.scp.new
mv data/test/wav.scp.new data/test/wav.scp

echo
echo "=== Counts ==="
wc -l data/test/utt2spk data/test/spk2utt data/test/wav.scp data/test/text

echo
echo "=== Validate TEST ==="
utils/validate_data_dir.sh --no-feats --non-print data/test

=== Remove duplicate WAV entry ===

=== Counts ===
  1521 data/test/utt2spk
     6 data/test/spk2utt
  1520 data/test/wav.scp
  1521 data/test/text
  4568 total

=== Validate TEST ===
utils/validate_data_dir.sh: Error: in data/test, utterance lists extracted from utt2spk and wav.scp
utils/validate_data_dir.sh: differ, partial diff is:
--- /tmp/kaldi.blop/utts	2026-09-10 07:20:40.621583894 +0000
+++ /tmp/kaldi.blop/utts.wav	2026-09-10 07:20:40.678258469 +0000
@@ -1520,2 +1520 @@
 ThidaAye_20260906_123652
-ThidaAye_20260906_123705
...
[Lengths are /tmp/kaldi.blop/utts=1521 versus /tmp/kaldi.blop/utts.wav=1520]


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\necho "=== Remove duplicate WAV entry ==="\nawk \'!seen[$1]++\' data/test/wav.scp > data/test/wav.scp.new\nmv data/test/wav.scp.new data/test/wav.scp\n\necho\necho "=== Counts ==="\nwc -l data/test/utt2spk data/test/spk2utt data/test/wav.scp data/test/text\n\necho\necho "=== Validate TEST ==="\nutils/validate_data_dir.sh --no-feats --non-print data/test\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Check WAV entry ==="
grep '^ThidaAye_20260906_123705 ' data/test/wav.scp || echo "No wav.scp entry"

echo
echo "=== 2. Check actual WAV file ==="
ls -lh recordings/ThidaAye/ThidaAye_20260906_123705.wav 2>&1 || true

echo
echo "=== 3. Check utt2spk ==="
grep '^ThidaAye_20260906_123705 ' data/test/utt2spk

echo
echo "=== 4. Check text ==="
grep '^ThidaAye_20260906_123705 ' data/test/text

=== 1. Check WAV entry ===
No wav.scp entry

=== 2. Check actual WAV file ===
-rw-r--r-- 1 thant_syn thant_syn 100K Sep  8 17:48 recordings/ThidaAye/ThidaAye_20260906_123705.wav

=== 3. Check utt2spk ===
ThidaAye_20260906_123705 ThidaAye

=== 4. Check text ===
ThidaAye_20260906_123705 နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Add missing WAV entry ==="

echo "ThidaAye_20260906_123705 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThidaAye/ThidaAye_20260906_123705.wav" >> data/test/wav.scp

echo
echo "=== 2. Sort wav.scp ==="
sort -k1,1 data/test/wav.scp -o data/test/wav.scp

echo
echo "=== 3. Check counts ==="
wc -l data/test/utt2spk data/test/spk2utt data/test/wav.scp data/test/text

echo
echo "=== 4. Validate TEST ==="
utils/validate_data_dir.sh --no-feats --non-print data/test

=== 1. Add missing WAV entry ===

=== 2. Sort wav.scp ===

=== 3. Check counts ===
  1521 data/test/utt2spk
     6 data/test/spk2utt
  1521 data/test/wav.scp
  1521 data/test/text
  4569 total

=== 4. Validate TEST ===
utils/validate_data_dir.sh: Successfully validated data-directory data/test


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== Current directory ==="
pwd

echo
echo "=== Kaldi environment ==="
echo "KALDI_ROOT=$KALDI_ROOT"

echo
echo "=== Required directories ==="
for d in utils steps local data/train data/test; do
    if [ -d "$d" ]; then
        echo "OK: $d"
    else
        echo "MISSING: $d"
    fi
done

echo
echo "=== Required Kaldi scripts ==="
for f in utils/prepare_lang.sh utils/validate_lang.pl utils/mkgraph.sh steps/make_mfcc.sh; do
    if [ -f "$f" ]; then
        echo "OK: $f"
    else
        echo "MISSING: $f"
    fi
done

echo
echo "=== Data validation ==="
utils/validate_data_dir.sh --no-feats --non-print data/train
utils/validate_data_dir.sh --no-feats --non-print data/test

=== Current directory ===
/home/thant_syn/kaldi/egs/burmese_asr

=== Kaldi environment ===
KALDI_ROOT=/home/thant_syn/kaldi

=== Required directories ===
OK: utils
OK: steps
OK: local
OK: data/train
OK: data/test

=== Required Kaldi scripts ===
OK: utils/prepare_lang.sh
OK: utils/validate_lang.pl
OK: utils/mkgraph.sh
OK: steps/make_mfcc.sh

=== Data validation ===
utils/validate_data_dir.sh: Successfully validated data-directory data/train
utils/validate_data_dir.sh: Successfully validated data-directory data/test


### Data Validation (Completed)

```
Recordings
   ↓
data/train + data/test          ✅ DONE
   ↓
validate_data_dir               ✅ DONE
   ↓
Dictionary / Lexicon
   ↓
Feature extraction
   ↓
Language model
   ↓
Monophone / GMM training
   ↓
Tripone / GMM training
   ↓
DNN training (if required)
   ↓
Decoding
   ↓
WER / CER evaluation
```

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "=== 1. Project files ==="
find . -maxdepth 2 -type f | sort

echo
echo "=== 2. Existing dictionary/language files ==="
find data -maxdepth 3 -type f | sort | grep -E 'dict|lang|lexicon|phone|lm' || true

echo
echo "=== 3. Local scripts ==="
ls -lh local/

echo
echo "=== 4. Existing local scripts content ==="
for f in local/*.sh local/*.py; do
    [ -f "$f" ] || continue
    echo
    echo "========== $f =========="
    sed -n '1,220p' "$f"
done


=== 1. Project files ===
./.ipynb_checkpoints/ASR_Mini_Project-checkpoint.ipynb
./.ipynb_checkpoints/burmese_asr-checkpoint.ipynb
./ASR_Mini_Project.ipynb
./cmd.sh
./data/train_test.zip
./echo
./local/update_wav_scp.sh
./recordings/recording.zip

=== 2. Existing dictionary/language files ===

=== 3. Local scripts ===
total 4.0K
-rwxr-xr-x 1 thant_syn thant_syn 844 Sep  8 18:25 update_wav_scp.sh

=== 4. Existing local scripts content ===

========== local/update_wav_scp.sh ==========
#!/bin/bash

set -e

RECORDINGS="/home/thant_syn/kaldi/egs/burmese_asr/recordings"

for DATA in train test; do

    INPUT="data/$DATA/wav.scp"
    OUTPUT="data/$DATA/wav.scp.new"

    if [ ! -f "$INPUT" ]; then
        echo "ERROR: $INPUT does not exist"
        continue
    fi

    > "$OUTPUT"

    while read -r UTT_ID OLD_PATH; do

        # Speaker name = everything before the first "_"
        SPEAKER="${UTT_ID%%_*}"

        # Remove Windows directory path.
        # Replace backslashes with forward sl

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "========================================"
echo "1. Notebook names"
echo "========================================"
ls -lh *.ipynb

echo
echo "========================================"
echo "2. Search notebook for Kaldi pipeline"
echo "========================================"
grep -RniE \
"prepare_lang|lexicon|nonsilence|silence_phones|train_mono|train_deltas|make_mfcc|compute_cmvn|mkgraph|decode|arpa|ngram|lm" \
*.ipynb | head -100

echo
echo "========================================"
echo "3. Search notebook for Burmese/text processing"
echo "========================================"
grep -RniE \
"burmese|Myanmar|unicode|token|character|syllable|phoneme|pronunciation|transcript" \
*.ipynb | head -100

1. Notebook names
-rw-r--r-- 1 thant_syn thant_syn 296K Sep 10 07:25 ASR_Mini_Project.ipynb

2. Search notebook for Kaldi pipeline
2184:    "Dictionary / Lexicon\n",
2226:      "OK: utils/prepare_lang.sh\n",
2228:      "OK: utils/mkgraph.sh\n",
2229:      "OK: steps/make_mfcc.sh\n",
2260:    "for f in utils/prepare_lang.sh utils/validate_lang.pl utils/mkgraph.sh steps/make_mfcc.sh; do\n",
2357:    "find data -maxdepth 3 -type f | sort | grep -E 'dict|lang|lexicon|phone|lm' || true\n",

3. Search notebook for Burmese/text processing
39:       "'/home/thant_syn/kaldi/egs/burmese_asr'"
61:      "/home/thant_syn/kaldi/egs/burmese_asr\n"
257:      "ThantSinTun_20260901_091117 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091117.wav\n",
258:      "ThantSinTun_20260901_091140 /home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_091140.wav\n",
259:      "ThantSinTun_20260901_091159 /home/thant_syn/kaldi/egs/burmese_asr/recordings

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "========================================"
echo "1. Sample TRAIN transcripts"
echo "========================================"
head -20 data/train/text

echo
echo "========================================"
echo "2. Sample TEST transcripts"
echo "========================================"
head -20 data/test/text

echo
echo "========================================"
echo "3. Number of unique transcript words"
echo "========================================"
awk '{$1=""; sub(/^ /,""); print}' data/train/text | \
tr ' ' '\n' | \
sort -u | wc -l

echo
echo "========================================"
echo "4. Unique transcript words"
echo "========================================"
awk '{$1=""; sub(/^ /,""); print}' data/train/text | \
tr ' ' '\n' | \
sort -u | \
head -100

echo
echo "========================================"
echo "5. Character inventory"
echo "========================================"
python3 - <<'PY'
from pathlib import Path

chars = set()

for line in Path("data/train/text").read_text(encoding="utf-8").splitlines():
    parts = line.split(maxsplit=1)
    if len(parts) == 2:
        chars.update(parts[1].replace(" ", ""))

print("Number of unique characters:", len(chars))
print("Characters:")
print(" ".join(sorted(chars)))
PY


1. Sample TRAIN transcripts
MyintThuSoe_Rec1_20260907_191704 ၀
MyintThuSoe_Rec1_20260907_191712 ၁
MyintThuSoe_Rec1_20260907_191719 ၂
MyintThuSoe_Rec1_20260907_191726 ၃
MyintThuSoe_Rec1_20260907_191733 ၄
MyintThuSoe_Rec1_20260907_191741 ၅
MyintThuSoe_Rec1_20260907_191835 ၆
MyintThuSoe_Rec1_20260907_191844 ၇
MyintThuSoe_Rec1_20260907_191851 ၈
MyintThuSoe_Rec1_20260907_191858 ၉
MyintThuSoe_Rec1_20260907_191908 နံပါတ် ၀ ပါ
MyintThuSoe_Rec1_20260907_191917 နံပါတ် ၁ ပါ
MyintThuSoe_Rec1_20260907_191925 နံပါတ် ၂ ပါ
MyintThuSoe_Rec1_20260907_191934 နံပါတ် ၃ ပါ
MyintThuSoe_Rec1_20260907_191942 နံပါတ် ၄ ပါ
MyintThuSoe_Rec1_20260907_191950 နံပါတ် ၅ ပါ
MyintThuSoe_Rec1_20260907_191959 နံပါတ် ၆ ပါ
MyintThuSoe_Rec1_20260907_192007 နံပါတ် ၇ ပါ
MyintThuSoe_Rec1_20260907_192015 နံပါတ် ၈ ပါ
MyintThuSoe_Rec1_20260907_192023 နံပါတ် ၉ ပါ

2. Sample TEST transcripts
AungKhantMyat_Rec1_20260907_232253 ၀
AungKhantMyat_Rec1_20260907_232313 ၁
AungKhantMyat_Rec1_20260907_232326 ၂
AungKhantMyat_Rec1_20260907_23234

---

## Lexicon Dictionary (Lexicon)

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

echo "========================================"
echo "Creating Kaldi dictionary"
echo "========================================"

# Create dictionary directories
mkdir -p data/local/dict

echo
echo "=== 1. Creating nonsilence_phones.txt ==="

python3 - <<'PY'
from pathlib import Path

chars = set()

for line in Path("data/train/text").read_text(encoding="utf-8").splitlines():
    parts = line.split(maxsplit=1)
    if len(parts) == 2:
        chars.update(parts[1].replace(" ", ""))

# Sort by Unicode code point
chars = sorted(chars)

with open("data/local/dict/nonsilence_phones.txt", "w", encoding="utf-8") as f:
    for ch in chars:
        f.write(ch + "\n")

print(f"Created {len(chars)} non-silence units.")
PY

echo
echo "=== 2. Creating silence_phones.txt ==="

cat > data/local/dict/silence_phones.txt <<'EOF'
SIL
EOF

echo
echo "=== 3. Creating optional_silence.txt ==="

cat > data/local/dict/optional_silence.txt <<'EOF'
SIL
EOF

echo
echo "=== 4. Creating lexicon.txt ==="

python3 - <<'PY'
from pathlib import Path

chars = set()

for line in Path("data/train/text").read_text(encoding="utf-8").splitlines():
    parts = line.split(maxsplit=1)
    if len(parts) == 2:
        chars.update(parts[1].replace(" ", ""))

chars = sorted(chars)

with open("data/local/dict/lexicon.txt", "w", encoding="utf-8") as f:

    # Silence
    f.write("SIL SIL\n")

    # Unknown word
    f.write("<UNK> SIL\n")

    # Character-based entries
    for ch in chars:
        f.write(f"{ch} {ch}\n")

print(f"Created lexicon entries for {len(chars)} characters.")
PY

echo
echo "========================================"
echo "Dictionary files"
echo "========================================"

ls -lh data/local/dict/

echo
echo "=== nonsilence_phones.txt ==="
cat data/local/dict/nonsilence_phones.txt

echo
echo "=== silence_phones.txt ==="
cat data/local/dict/silence_phones.txt

echo
echo "=== optional_silence.txt ==="
cat data/local/dict/optional_silence.txt

echo
echo "=== First 30 lexicon entries ==="
head -30 data/local/dict/lexicon.txt

echo
echo "========================================"
echo "Dictionary counts"
echo "========================================"

echo -n "Non-silence units: "
wc -l < data/local/dict/nonsilence_phones.txt

echo -n "Lexicon entries: "
wc -l < data/local/dict/lexicon.txt

Creating Kaldi dictionary

=== 1. Creating nonsilence_phones.txt ===
Created 48 non-silence units.

=== 2. Creating silence_phones.txt ===

=== 3. Creating optional_silence.txt ===

=== 4. Creating lexicon.txt ===
Created lexicon entries for 48 characters.

Dictionary files
total 16K
-rw-r--r-- 1 thant_syn thant_syn 398 Sep 10 07:27 lexicon.txt
-rw-r--r-- 1 thant_syn thant_syn 190 Sep 10 07:27 nonsilence_phones.txt
-rw-r--r-- 1 thant_syn thant_syn   4 Sep 10 07:27 optional_silence.txt
-rw-r--r-- 1 thant_syn thant_syn   4 Sep 10 07:27 silence_phones.txt

=== nonsilence_phones.txt ===
,
က
ခ
ဂ
င
စ
ဇ
ည
တ
ထ
ဒ
န
ပ
ဖ
ဘ
မ
ယ
ရ
လ
ဝ
အ
ဧ
ဩ
ါ
ာ
ိ
ီ
ု
ူ
ေ
ဲ
ံ
း
်
ျ
ြ
ွ
ှ
၀
၁
၂
၃
၄
၅
၆
၇
၈
၉

=== silence_phones.txt ===
SIL

=== optional_silence.txt ===
SIL

=== First 30 lexicon entries ===
SIL SIL
<UNK> SIL
, ,
က က
ခ ခ
ဂ ဂ
င င
စ စ
ဇ ဇ
ည ည
တ တ
ထ ထ
ဒ ဒ
န န
ပ ပ
ဖ ဖ
ဘ ဘ
မ မ
ယ ယ
ရ ရ
လ လ
ဝ ဝ
အ အ
ဧ ဧ
ဩ ဩ
ါ ါ
ာ ာ
ိ ိ
ီ ီ
ု ု

Dictionary counts
Non-silence units: 48
Lexicon entries: 50


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

echo "========================================"
echo "Preparing Kaldi language directory"
echo "========================================"

# Remove any previous language directories
rm -rf data/local/lang data/lang

echo
echo "=== Running prepare_lang.sh ==="

utils/prepare_lang.sh \
    data/local/dict \
    "<UNK>" \
    data/local/lang \
    data/lang

echo
echo "========================================"
echo "Language preparation completed"
echo "========================================"

echo
echo "=== data/lang contents ==="
find data/lang -maxdepth 1 -type f -printf '%f\n' | sort

echo
echo "=== Important files ==="

for f in \
    data/lang/words.txt \
    data/lang/phones.txt \
    data/lang/L.fst \
    data/lang/G.fst
do
    if [ -f "$f" ]; then
        echo "OK: $f"
    else
        echo "MISSING: $f"
    fi
done

echo
echo "=== Number of words ==="
wc -l data/lang/words.txt

echo
echo "=== Number of phones ==="
wc -l data/lang/phones.txt

echo
echo "=== First 30 words ==="
head -30 data/lang/words.txt

Preparing Kaldi language directory

=== Running prepare_lang.sh ===
utils/prepare_lang.sh data/local/dict <UNK> data/local/lang data/lang
Checking data/local/dict/silence_phones.txt ...
--> reading data/local/dict/silence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/silence_phones.txt is OK

Checking data/local/dict/optional_silence.txt ...
--> reading data/local/dict/optional_silence.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/optional_silence.txt is OK

Checking data/local/dict/nonsilence_phones.txt ...
--> reading data/local/dict/nonsilence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/nonsilence_phones.txt is OK

Checking disjoint: silence_phones.txt, nonsilence_phones.txt
--> disjoint property is OK.

Checking data/local/dict/lexic

utils/prepare_lang.sh: line 466: fstcompile: command not found
utils/prepare_lang.sh: line 468: fstarcsort: command not found


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\n\necho "========================================"\necho "Preparing Kaldi language directory"\necho "========================================"\n\n# Remove any previous language directories\nrm -rf data/local/lang data/lang\n\necho\necho "=== Running prepare_lang.sh ==="\n\nutils/prepare_lang.sh \\\n    data/local/dict \\\n    "<UNK>" \\\n    data/local/lang \\\n    data/lang\n\necho\necho "========================================"\necho "Language preparation completed"\necho "========================================"\n\necho\necho "=== data/lang contents ==="\nfind data/lang -maxdepth 1 -type f -printf \'%f\\n\' | sort\n\necho\necho "=== Important files ==="\n\nfor f in \\\n    data/lang/words.txt \\\n    data/lang/phones.txt \\\n    data/lang/L.fst \\\n    data/lang/G.fst\ndo\n    if [ -f "$f" ]; then\n        echo "OK: $f"\n    else\n        echo "MISSING: $f"\n    fi\ndone\n\necho\necho "=== Number of words ==="\nwc

#### OpenFST

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

echo "========================================"
echo "Checking OpenFST"
echo "========================================"

echo
echo "Kaldi root:"
echo "$KALDI_ROOT"

echo
echo "OpenFST directory:"
ls -ld ~/kaldi/tools/openfst/bin

echo
echo "Checking fstcompile:"
ls -l ~/kaldi/tools/openfst/bin/fstcompile 2>/dev/null || true

echo
echo "Checking fstarcsort:"
ls -l ~/kaldi/tools/openfst/bin/fstarcsort 2>/dev/null || true

echo
echo "Current PATH:"
echo "$PATH" | tr ':' '\n' | grep -E 'kaldi|openfst' || true

Checking OpenFST

Kaldi root:
/home/thant_syn/kaldi

OpenFST directory:
drwxr-xr-x 2 thant_syn thant_syn 4096 Aug 29 07:15 /home/thant_syn/kaldi/tools/openfst/bin

Checking fstcompile:
-rwxr-xr-x 1 thant_syn thant_syn 680744 Aug 29 07:13 /home/thant_syn/kaldi/tools/openfst/bin/fstcompile

Checking fstarcsort:
-rwxr-xr-x 1 thant_syn thant_syn 443248 Aug 29 07:13 /home/thant_syn/kaldi/tools/openfst/bin/fstarcsort

Current PATH:
/home/thant_syn/kaldi/egs/burmese_asr/utils
/home/thant_syn/kaldi/egs/burmese_asr/steps
/home/thant_syn/kaldi/src/bin
/home/thant_syn/kaldi/src/featbin
/home/thant_syn/kaldi/src/gmmbin
/home/thant_syn/kaldi/src/fstbin
/home/thant_syn/kaldi/src/nnet3bin


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

export KALDI_ROOT=/home/thant_syn/kaldi

# Add OpenFST to PATH
export PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$PATH"

echo "========================================"
echo "Checking OpenFST PATH"
echo "========================================"

echo
echo "fstcompile:"
which fstcompile

echo
echo "fstarcsort:"
which fstarcsort

echo
echo "========================================"
echo "Preparing Kaldi language directory"
echo "========================================"

rm -rf data/local/lang data/lang

utils/prepare_lang.sh \
    data/local/dict \
    "<UNK>" \
    data/local/lang \
    data/lang

echo
echo "========================================"
echo "Language preparation completed"
echo "========================================"

echo
echo "=== data/lang files ==="
find data/lang -maxdepth 1 -type f -printf '%f\n' | sort

echo
echo "=== Important files ==="

for f in \
    data/lang/words.txt \
    data/lang/phones.txt \
    data/lang/L.fst
do
    if [ -f "$f" ]; then
        echo "OK: $f"
    else
        echo "MISSING: $f"
    fi
done

echo
echo "=== Word count ==="
wc -l data/lang/words.txt

echo
echo "=== Phone count ==="
wc -l data/lang/phones.txt

Checking OpenFST PATH

fstcompile:
/home/thant_syn/kaldi/tools/openfst/bin/fstcompile

fstarcsort:
/home/thant_syn/kaldi/tools/openfst/bin/fstarcsort

Preparing Kaldi language directory
utils/prepare_lang.sh data/local/dict <UNK> data/local/lang data/lang
Checking data/local/dict/silence_phones.txt ...
--> reading data/local/dict/silence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/silence_phones.txt is OK

Checking data/local/dict/optional_silence.txt ...
--> reading data/local/dict/optional_silence.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/optional_silence.txt is OK

Checking data/local/dict/nonsilence_phones.txt ...
--> reading data/local/dict/nonsilence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/nonsilence_phones.txt is OK

Che

fstaddselfloops data/lang/phones/wdisambig_phones.int data/lang/phones/wdisambig_words.int 


prepare_lang.sh: validating output directory
utils/validate_lang.pl data/lang
Checking existence of separator file
separator file data/lang/subword_separator.txt is empty or does not exist, deal in word case.
Checking data/lang/phones.txt ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/phones.txt is OK

Checking words.txt: #0 ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/words.txt is OK

Checking disjoint: silence.txt, nonsilence.txt, disambig.txt ...
--> silence.txt and nonsilence.txt are disjoint
--> silence.txt and disambig.txt are disjoint
--> disambig.txt and nonsilence.txt are disjoint
--> disjoint property is OK

Checking sumation: silence.txt, nonsilence.txt, disambig.txt ...
--> found no unexplainable phones in phones.txt

Checking data/lang/phones/context_indep.{txt, int, csl} ...
--> text seems to be UTF-8 or ASCII, checking whitespa

sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file


--> generating a 16 word/subword sequence
--> ERROR: number of reconstructed words 0 does not match real number of words 16; indicates problem in L.fst or word_boundary.int.  phoneseq = , wordseq = <UNK> တ င ၁ ာ ၉ ျ ည ည ည ာ ါ ဩ ဂ ၀ ဇ 
--> generating a 78 word/subword sequence

Checking data/lang/oov.{txt, int} ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> 1 entry/entries in data/lang/oov.txt
--> data/lang/oov.int corresponds to data/lang/oov.txt
--> data/lang/oov.{txt, int} are OK

--> ERROR: data/lang/L.fst is not olabel sorted
--> ERROR: data/lang/L_disambig.fst is not olabel sorted
--> ERROR (see error messages above)
prepare_lang.sh: error validating output


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\n\nexport KALDI_ROOT=/home/thant_syn/kaldi\n\n# Add OpenFST to PATH\nexport PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$PATH"\n\necho "========================================"\necho "Checking OpenFST PATH"\necho "========================================"\n\necho\necho "fstcompile:"\nwhich fstcompile\n\necho\necho "fstarcsort:"\nwhich fstarcsort\n\necho\necho "========================================"\necho "Preparing Kaldi language directory"\necho "========================================"\n\nrm -rf data/local/lang data/lang\n\nutils/prepare_lang.sh \\\n    data/local/dict \\\n    "<UNK>" \\\n    data/local/lang \\\n    data/lang\n\necho\necho "========================================"\necho "Language preparation completed"\necho "========================================"\n\necho\necho "=== data/lang files ==="\nfind data/lang -maxdepth 1 -type f -printf \'%f\\n\' | sort\n\necho\ne

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

export KALDI_ROOT=/home/thant_syn/kaldi
export PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$PATH"

echo "========================================"
echo "Rebuilding Burmese Character Lexicon"
echo "========================================"

rm -rf data/local/dict
mkdir -p data/local/dict

echo
echo "=== Creating character inventory ==="

python3 <<'PY'
from pathlib import Path

text_file = Path("data/train/text")

# Collect every unique whitespace-delimited transcript word
words = set()

with text_file.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split(maxsplit=1)

        if len(parts) < 2:
            continue

        transcript = parts[1]

        for word in transcript.split():
            words.add(word)

# Collect Unicode characters from all transcript words
chars = set()

for word in words:
    chars.update(word)

# Sort using Unicode code points
chars = sorted(chars)

print(f"Unique transcript words: {len(words)}")
print(f"Unique characters: {len(chars)}")

# Write character inventory
Path("data/local/dict/nonsilence_phones.txt").write_text(
    "\n".join(chars) + "\n",
    encoding="utf-8"
)

# Silence files
Path("data/local/dict/silence_phones.txt").write_text(
    "SIL\n",
    encoding="utf-8"
)

Path("data/local/dict/optional_silence.txt").write_text(
    "SIL\n",
    encoding="utf-8"
)

# Empty extra questions file
Path("data/local/dict/extra_questions.txt").write_text(
    "",
    encoding="utf-8"
)

# Build lexicon:
# transcript word -> Unicode character sequence
lexicon = [
    "SIL SIL",
    "<UNK> SPN"
]

# SPN is needed for the unknown-word pronunciation
phones = chars + ["SPN"]

# Add every actual transcript word
for word in sorted(words):
    pronunciation = " ".join(list(word))
    lexicon.append(f"{word} {pronunciation}")

Path("data/local/dict/lexicon.txt").write_text(
    "\n".join(lexicon) + "\n",
    encoding="utf-8"
)

# Add SPN to non-silence phones
Path("data/local/dict/nonsilence_phones.txt").write_text(
    "\n".join(phones) + "\n",
    encoding="utf-8"
)

print()
print("Dictionary created.")
print(f"  Words     : {len(words)}")
print(f"  Characters: {len(chars)}")
print(f"  Phones    : {len(phones)}")
PY

echo
echo "========================================"
echo "Dictionary files"
echo "========================================"

ls -lh data/local/dict/

echo
echo "=== Character/phone inventory ==="
wc -l data/local/dict/nonsilence_phones.txt

echo
echo "=== Lexicon entries ==="
wc -l data/local/dict/lexicon.txt

echo
echo "=== Sample lexicon ==="
head -20 data/local/dict/lexicon.txt

echo
echo "=== Burmese word examples ==="
grep -E '^(နံပါတ်|ဖြည်းဖြည်း|ဖုန်းနံပါတ်) ' \
    data/local/dict/lexicon.txt || true

Rebuilding Burmese Character Lexicon

=== Creating character inventory ===
Unique transcript words: 136
Unique characters: 48

Dictionary created.
  Words     : 136
  Characters: 48
  Phones    : 49

Dictionary files
total 20K
-rw-r--r-- 1 thant_syn thant_syn    0 Sep 10 07:30 extra_questions.txt
-rw-r--r-- 1 thant_syn thant_syn 4.2K Sep 10 07:30 lexicon.txt
-rw-r--r-- 1 thant_syn thant_syn  194 Sep 10 07:30 nonsilence_phones.txt
-rw-r--r-- 1 thant_syn thant_syn    4 Sep 10 07:30 optional_silence.txt
-rw-r--r-- 1 thant_syn thant_syn    4 Sep 10 07:30 silence_phones.txt

=== Character/phone inventory ===
49 data/local/dict/nonsilence_phones.txt

=== Lexicon entries ===
138 data/local/dict/lexicon.txt

=== Sample lexicon ===
SIL SIL
<UNK> SPN
ကို က ိ ု
ကျပ် က ျ ပ ်
ခုပါ ခ ု ပ ါ
ငွေက င ွ ေ က
စက်တင်ဘာ စ က ် တ င ် ဘ ာ
ဇန်နဝါရီ ဇ န ် န ဝ ါ ရ ီ
ဇူလိုင် ဇ ူ လ ိ ု င ်
ဇွန်လ ဇ ွ န ် လ
တစ်ကြိမ် တ စ ် က ြ ိ မ ်
ထပ်ပြောပါ ထ ပ ် ပ ြ ေ ာ ပ ါ
နံပါတ် န ံ ပ ါ တ ်
နံပါတ်ကို န ံ ပ ါ တ ် က ိ ု
နှိပ်ပါ န ှ 

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

export KALDI_ROOT=/home/thant_syn/kaldi
export PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$PATH"

echo "========================================"
echo "Preparing Kaldi Language Directory"
echo "========================================"

rm -rf data/local/lang data/lang

utils/prepare_lang.sh \
    data/local/dict \
    "<UNK>" \
    data/local/lang \
    data/lang

echo
echo "========================================"
echo "Language preparation finished"
echo "========================================"

echo
echo "=== Checking important files ==="

for f in \
    data/lang/words.txt \
    data/lang/phones.txt \
    data/lang/phones.txt \
    data/lang/L.fst \
    data/lang/L_disambig.fst
do
    if [ -f "$f" ]; then
        echo "OK: $f"
    else
        echo "MISSING: $f"
    fi
done

echo
echo "=== Word count ==="
wc -l data/lang/words.txt

echo
echo "=== Phone count ==="
wc -l data/lang/phones.txt

echo
echo "=== Sample words ==="
head -25 data/lang/words.txt

echo
echo "=== Validating language directory ==="
utils/validate_lang.pl data/lang

echo
echo "========================================"
echo "STEP 7 COMPLETE"
echo "========================================"

Preparing Kaldi Language Directory
utils/prepare_lang.sh data/local/dict <UNK> data/local/lang data/lang
Checking data/local/dict/silence_phones.txt ...
--> reading data/local/dict/silence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/silence_phones.txt is OK

Checking data/local/dict/optional_silence.txt ...
--> reading data/local/dict/optional_silence.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/optional_silence.txt is OK

Checking data/local/dict/nonsilence_phones.txt ...
--> reading data/local/dict/nonsilence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/nonsilence_phones.txt is OK

Checking disjoint: silence_phones.txt, nonsilence_phones.txt
--> disjoint property is OK.

Checking data/local/dict/lexicon.txt
--> reading data/local/dic

fstaddselfloops data/lang/phones/wdisambig_phones.int data/lang/phones/wdisambig_words.int 


prepare_lang.sh: validating output directory
utils/validate_lang.pl data/lang
Checking existence of separator file
separator file data/lang/subword_separator.txt is empty or does not exist, deal in word case.
Checking data/lang/phones.txt ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/phones.txt is OK

Checking words.txt: #0 ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/words.txt is OK

Checking disjoint: silence.txt, nonsilence.txt, disambig.txt ...
--> silence.txt and nonsilence.txt are disjoint
--> silence.txt and disambig.txt are disjoint
--> disambig.txt and nonsilence.txt are disjoint
--> disjoint property is OK

Checking sumation: silence.txt, nonsilence.txt, disambig.txt ...
--> found no unexplainable phones in phones.txt

Checking data/lang/phones/context_indep.{txt, int, csl} ...
--> text seems to be UTF-8 or ASCII, checking whitespa

sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file
sh: 1: .: cannot open ./path.sh: No such file


--> generating a 21 word/subword sequence
--> ERROR: number of reconstructed words 0 does not match real number of words 21; indicates problem in L.fst or word_boundary.int.  phoneseq = , wordseq = ၄၂ ၉၉ ၅၀,၀၀၀ ၂၇ ၁,၀၀၀ ၉ ၀၉၈၈၈၇၇၇၆၆၅ ၃၈ ၅၆ ၇၈၉ နံပါတ်ကို ဇန်နဝါရီ ၁၀၂၃ ၂၉၉ ၅၆၇၈ ၂၁ ၁၉ ၆၀ ၆၇ ၂၃၄၅ အရေအတွက် 
--> generating a 92 word/subword sequence

Checking data/lang/oov.{txt, int} ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> 1 entry/entries in data/lang/oov.txt
--> data/lang/oov.int corresponds to data/lang/oov.txt
--> data/lang/oov.{txt, int} are OK

--> ERROR: data/lang/L.fst is not olabel sorted
--> ERROR: data/lang/L_disambig.fst is not olabel sorted
--> ERROR (see error messages above)
prepare_lang.sh: error validating output


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\n\nexport KALDI_ROOT=/home/thant_syn/kaldi\nexport PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$PATH"\n\necho "========================================"\necho "Preparing Kaldi Language Directory"\necho "========================================"\n\nrm -rf data/local/lang data/lang\n\nutils/prepare_lang.sh \\\n    data/local/dict \\\n    "<UNK>" \\\n    data/local/lang \\\n    data/lang\n\necho\necho "========================================"\necho "Language preparation finished"\necho "========================================"\n\necho\necho "=== Checking important files ==="\n\nfor f in \\\n    data/lang/words.txt \\\n    data/lang/phones.txt \\\n    data/lang/phones.txt \\\n    data/lang/L.fst \\\n    data/lang/L_disambig.fst\ndo\n    if [ -f "$f" ]; then\n        echo "OK: $f"\n    else\n        echo "MISSING: $f"\n    fi\ndone\n\necho\necho "=== Word count ==="\nwc -l data/lang/word

#### L.Fst: Kaldi/OpenFST diagnostic

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

export KALDI_ROOT=/home/thant_syn/kaldi
export PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$PATH"

echo "========================================"
echo "Kaldi / OpenFST diagnostic"
echo "========================================"

echo
echo "=== Current directory ==="
pwd

echo
echo "=== path.sh ==="
if [ -f ./path.sh ]; then
    echo "FOUND: ./path.sh"
else
    echo "NOT FOUND: ./path.sh"
fi

echo
echo "=== OpenFST commands ==="
which fstcompile
which fstarcsort
which fstinfo
which fstprint

echo
echo "=== Generated language files ==="
ls -lh data/lang/L.fst data/lang/L_disambig.fst \
       data/lang/words.txt data/lang/phones.txt

echo
echo "=== FST information: L.fst ==="
fstinfo data/lang/L.fst | head -30

echo
echo "=== FST information: L_disambig.fst ==="
fstinfo data/lang/L_disambig.fst | head -30

echo
echo "=== Check sorting ==="
echo "L.fst:"
fstinfo data/lang/L.fst | grep -E 'sorted|arc type|states|arcs' || true

echo
echo "L_disambig.fst:"
fstinfo data/lang/L_disambig.fst | grep -E 'sorted|arc type|states|arcs' || true

echo
echo "=== Sample words ==="
head -30 data/lang/words.txt

echo
echo "========================================"
echo "Diagnostic complete"
echo "========================================"

Kaldi / OpenFST diagnostic

=== Current directory ===
/home/thant_syn/kaldi/egs/burmese_asr

=== path.sh ===
NOT FOUND: ./path.sh

=== OpenFST commands ===
/home/thant_syn/kaldi/tools/openfst/bin/fstcompile
/home/thant_syn/kaldi/tools/openfst/bin/fstarcsort
/home/thant_syn/kaldi/tools/openfst/bin/fstinfo
/home/thant_syn/kaldi/tools/openfst/bin/fstprint

=== Generated language files ===
-rw-r--r-- 1 thant_syn thant_syn  17K Sep 10 07:30 data/lang/L.fst
-rw-r--r-- 1 thant_syn thant_syn  17K Sep 10 07:30 data/lang/L_disambig.fst
-rw-r--r-- 1 thant_syn thant_syn 1.9K Sep 10 07:30 data/lang/phones.txt
-rw-r--r-- 1 thant_syn thant_syn 2.4K Sep 10 07:30 data/lang/words.txt

=== FST information: L.fst ===
fst type                                          vector
arc type                                          standard
input symbol table                                none
output symbol table                               none
# of states                                       455
# of arcs    

#### Creating path.sh

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

export KALDI_ROOT=/home/thant_syn/kaldi

echo "========================================"
echo "Creating project path.sh"
echo "========================================"

cat > path.sh <<'EOF'
#!/usr/bin/env bash

export KALDI_ROOT=/home/thant_syn/kaldi

export PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$KALDI_ROOT/src/featbin:$KALDI_ROOT/src/gmmbin:$KALDI_ROOT/src/nnet3bin:$PATH"

export LC_ALL=C
EOF

chmod +x path.sh

echo
echo "=== path.sh created ==="
ls -lh path.sh

echo
echo "=== Testing path.sh ==="
source ./path.sh

echo "KALDI_ROOT=$KALDI_ROOT"

echo
echo "fstcompile:"
which fstcompile

echo
echo "fstarcsort:"
which fstarcsort

echo
echo "run.pl:"
which run.pl || true

echo
echo "========================================"
echo "PATH setup successful"
echo "========================================"

Creating project path.sh

=== path.sh created ===
-rwxr-xr-x 1 thant_syn thant_syn 244 Sep 10 07:32 path.sh

=== Testing path.sh ===
KALDI_ROOT=/home/thant_syn/kaldi

fstcompile:
/home/thant_syn/kaldi/tools/openfst/bin/fstcompile

fstarcsort:
/home/thant_syn/kaldi/tools/openfst/bin/fstarcsort

run.pl:
/home/thant_syn/kaldi/egs/burmese_asr/utils/run.pl

PATH setup successful


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

source ./path.sh

echo "========================================"
echo "Preparing Kaldi Language Directory"
echo "========================================"

rm -rf data/local/lang data/lang

utils/prepare_lang.sh \
    data/local/dict \
    "<UNK>" \
    data/local/lang \
    data/lang

echo
echo "========================================"
echo "Language preparation completed"
echo "========================================"

echo
echo "=== Checking L.fst ==="
fstinfo data/lang/L.fst | grep -E \
    '# of states|# of arcs|input label sorted|output label sorted|error'

echo
echo "=== Checking L_disambig.fst ==="
fstinfo data/lang/L_disambig.fst | grep -E \
    '# of states|# of arcs|input label sorted|output label sorted|error'

echo
echo "=== Final language validation ==="
utils/validate_lang.pl data/lang

echo
echo "========================================"
echo "STEP 9 COMPLETE"
echo "========================================"

Preparing Kaldi Language Directory
utils/prepare_lang.sh data/local/dict <UNK> data/local/lang data/lang
Checking data/local/dict/silence_phones.txt ...
--> reading data/local/dict/silence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/silence_phones.txt is OK

Checking data/local/dict/optional_silence.txt ...
--> reading data/local/dict/optional_silence.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/optional_silence.txt is OK

Checking data/local/dict/nonsilence_phones.txt ...
--> reading data/local/dict/nonsilence_phones.txt
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/local/dict/nonsilence_phones.txt is OK

Checking disjoint: silence_phones.txt, nonsilence_phones.txt
--> disjoint property is OK.

Checking data/local/dict/lexicon.txt
--> reading data/local/dic

fstaddselfloops data/lang/phones/wdisambig_phones.int data/lang/phones/wdisambig_words.int 


prepare_lang.sh: validating output directory
utils/validate_lang.pl data/lang
Checking existence of separator file
separator file data/lang/subword_separator.txt is empty or does not exist, deal in word case.
Checking data/lang/phones.txt ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/phones.txt is OK

Checking words.txt: #0 ...
--> text seems to be UTF-8 or ASCII, checking whitespaces
--> text contains only allowed whitespaces
--> data/lang/words.txt is OK

Checking disjoint: silence.txt, nonsilence.txt, disambig.txt ...
--> silence.txt and nonsilence.txt are disjoint
--> silence.txt and disambig.txt are disjoint
--> disambig.txt and nonsilence.txt are disjoint
--> disjoint property is OK

Checking sumation: silence.txt, nonsilence.txt, disambig.txt ...
--> found no unexplainable phones in phones.txt

Checking data/lang/phones/context_indep.{txt, int, csl} ...
--> text seems to be UTF-8 or ASCII, checking whitespa

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "Checking Burmese ASR Audio Format"
echo "========================================"

echo
echo "=== First 10 WAV files ==="

find recordings -type f -iname "*.wav" | head -10

echo
echo "=== Audio information ==="

WAV=$(find recordings -type f -iname "*.wav" | head -1)

echo "Selected file:"
echo "$WAV"

echo
echo "Using sox:"
if command -v sox >/dev/null 2>&1; then
    sox --i "$WAV"
else
    echo "sox is not installed"
fi

echo
echo "Using file:"
file "$WAV"

echo
echo "========================================"
echo "Audio check complete"
echo "========================================"

Checking Burmese ASR Audio Format

=== First 10 WAV files ===
recordings/ThidaAye/ThidaAye_20260905_120545.wav
recordings/ThidaAye/ThidaAye_20260906_121826.wav
recordings/ThidaAye/ThidaAye_20260906_121820.wav
recordings/ThidaAye/ThidaAye_20260905_095916.wav
recordings/ThidaAye/ThidaAye_20260905_100220.wav
recordings/ThidaAye/ThidaAye_20260905_120322.wav
recordings/ThidaAye/ThidaAye_20260905_120052.wav
recordings/ThidaAye/ThidaAye_20260905_100021.wav
recordings/ThidaAye/ThidaAye_20260905_124025.wav
recordings/ThidaAye/ThidaAye_20260905_013408.wav

=== Audio information ===
Selected file:
recordings/ThidaAye/ThidaAye_20260905_120545.wav

Using sox:

Input File     : 'recordings/ThidaAye/ThidaAye_20260905_120545.wav'
Channels       : 1
Sample Rate    : 16000
Precision      : 16-bit
Duration       : 00:00:02.00 = 32032 samples ~ 150.15 CDDA sectors
File Size      : 64.1k
Bit Rate       : 256k
Sample Encoding: 16-bit Signed Integer PCM


Using file:
recordings/ThidaAye/ThidaAye_20260905_120

## Extract MFCC features

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 11: Extracting MFCC Features"
echo "========================================"

# Check that required scripts exist
echo
echo "Checking Kaldi MFCC tools..."

which compute-mfcc-feats
which copy-feats
which steps/make_mfcc.sh

echo
echo "=== Creating MFCC features for TRAIN ==="

steps/make_mfcc.sh \
    --nj 4 \
    --cmd run.pl \
    data/train \
    exp/make_mfcc/train \
    mfcc/train

echo
echo "=== Computing CMVN statistics for TRAIN ==="

steps/compute_cmvn_stats.sh \
    data/train \
    exp/make_mfcc/train \
    mfcc/train

echo
echo "=== Creating MFCC features for TEST ==="

steps/make_mfcc.sh \
    --nj 4 \
    --cmd run.pl \
    data/test \
    exp/make_mfcc/test \
    mfcc/test

echo
echo "=== Computing CMVN statistics for TEST ==="

steps/compute_cmvn_stats.sh \
    data/test \
    exp/make_mfcc/test \
    mfcc/test

echo
echo "========================================"
echo "STEP 11 COMPLETE"
echo "========================================"

echo
echo "TRAIN feature files:"
ls -lh data/train/feats.scp data/train/cmvn.scp

echo
echo "TEST feature files:"
ls -lh data/test/feats.scp data/test/cmvn.scp

STEP 11: Extracting MFCC Features

Checking Kaldi MFCC tools...
/home/thant_syn/kaldi/src/featbin/compute-mfcc-feats
/home/thant_syn/kaldi/src/featbin/copy-feats
steps/make_mfcc.sh

=== Creating MFCC features for TRAIN ===
steps/make_mfcc.sh --nj 4 --cmd run.pl data/train exp/make_mfcc/train mfcc/train
steps/make_mfcc.sh: no such file conf/mfcc.conf


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\nsource ./path.sh\n\necho "========================================"\necho "STEP 11: Extracting MFCC Features"\necho "========================================"\n\n# Check that required scripts exist\necho\necho "Checking Kaldi MFCC tools..."\n\nwhich compute-mfcc-feats\nwhich copy-feats\nwhich steps/make_mfcc.sh\n\necho\necho "=== Creating MFCC features for TRAIN ==="\n\nsteps/make_mfcc.sh \\\n    --nj 4 \\\n    --cmd run.pl \\\n    data/train \\\n    exp/make_mfcc/train \\\n    mfcc/train\n\necho\necho "=== Computing CMVN statistics for TRAIN ==="\n\nsteps/compute_cmvn_stats.sh \\\n    data/train \\\n    exp/make_mfcc/train \\\n    mfcc/train\n\necho\necho "=== Creating MFCC features for TEST ==="\n\nsteps/make_mfcc.sh \\\n    --nj 4 \\\n    --cmd run.pl \\\n    data/test \\\n    exp/make_mfcc/test \\\n    mfcc/test\n\necho\necho "=== Computing CMVN statistics for TEST ==="\n\nsteps/compute_cmvn_stats.sh \\\n    data/

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 11A: Creating MFCC Configuration"
echo "========================================"

mkdir -p conf

cat > conf/mfcc.conf << 'EOF'
--use-energy=false
--sample-frequency=16000
--num-mel-bins=23
--num-ceps=13
--low-freq=20
--high-freq=-400
EOF

echo
echo "Created:"
cat conf/mfcc.conf

echo
echo "========================================"
echo "MFCC configuration ready"
echo "========================================"

STEP 11A: Creating MFCC Configuration

Created:
--use-energy=false
--sample-frequency=16000
--num-mel-bins=23
--num-ceps=13
--low-freq=20
--high-freq=-400

MFCC configuration ready


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 11B: Extracting MFCC Features"
echo "========================================"

echo
echo "=== TRAIN MFCC ==="

steps/make_mfcc.sh \
    --nj 4 \
    --cmd run.pl \
    data/train \
    exp/make_mfcc/train \
    mfcc/train

echo
echo "=== TRAIN CMVN ==="

steps/compute_cmvn_stats.sh \
    data/train \
    exp/make_mfcc/train \
    mfcc/train

echo
echo "=== TEST MFCC ==="

steps/make_mfcc.sh \
    --nj 4 \
    --cmd run.pl \
    data/test \
    exp/make_mfcc/test \
    mfcc/test

echo
echo "=== TEST CMVN ==="

steps/compute_cmvn_stats.sh \
    data/test \
    exp/make_mfcc/test \
    mfcc/test

echo
echo "========================================"
echo "STEP 11 COMPLETE"
echo "========================================"

echo
echo "=== TRAIN FEATURES ==="
ls -lh data/train/feats.scp data/train/cmvn.scp

echo
echo "=== TEST FEATURES ==="
ls -lh data/test/feats.scp data/test/cmvn.scp

STEP 11B: Extracting MFCC Features

=== TRAIN MFCC ===
steps/make_mfcc.sh --nj 4 --cmd run.pl data/train exp/make_mfcc/train mfcc/train
utils/validate_data_dir.sh: Successfully validated data-directory data/train
steps/make_mfcc.sh: [info]: no segments file exists: assuming wav.scp indexed by utterance.
steps/make_mfcc.sh: It seems not all of the feature files were successfully procesed (2999 != 3057); consider using utils/fix_data_dir.sh data/train
steps/make_mfcc.sh: Succeeded creating MFCC features for train

=== TRAIN CMVN ===
steps/compute_cmvn_stats.sh data/train exp/make_mfcc/train mfcc/train
Succeeded creating CMVN stats for train

=== TEST MFCC ===
steps/make_mfcc.sh --nj 4 --cmd run.pl data/test exp/make_mfcc/test mfcc/test
utils/validate_data_dir.sh: Successfully validated data-directory data/test
steps/make_mfcc.sh: [info]: no segments file exists: assuming wav.scp indexed by utterance.
steps/make_mfcc.sh: It seems not all of the feature files were successfully procesed (15

#### Missing MFCC Features

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 12: Finding Missing MFCC Features"
echo "========================================"

echo
echo "=== TRAIN ==="

comm -23 \
    <(cut -d' ' -f1 data/train/wav.scp | sort) \
    <(cut -d' ' -f1 data/train/feats.scp | sort) \
    > /tmp/train_missing.txt

echo "Missing TRAIN features:"
wc -l /tmp/train_missing.txt
cat /tmp/train_missing.txt

echo
echo "=== TEST ==="

comm -23 \
    <(cut -d' ' -f1 data/test/wav.scp | sort) \
    <(cut -d' ' -f1 data/test/feats.scp | sort) \
    > /tmp/test_missing.txt

echo "Missing TEST features:"
wc -l /tmp/test_missing.txt
cat /tmp/test_missing.txt

echo
echo "========================================"
echo "STEP 12 COMPLETE"
echo "========================================"

STEP 12: Finding Missing MFCC Features

=== TRAIN ===
Missing TRAIN features:
58 /tmp/train_missing.txt
ThantSinTun_20260901_100446
ThantSinTun_20260901_100803
ThantSinTun_20260901_104053
ThantSinTun_20260901_104402
ThantSinTun_20260901_104617
ThantSinTun_20260901_104649
ThantSinTun_20260901_104842
ThantSinTun_20260901_155435
ThantSinTun_20260901_160031
ThantSinTun_20260901_174418
ThantSinTun_20260901_174558
ThantSinTun_20260901_174952
ThantSinTun_20260901_175054
ThantSinTun_20260901_175854
ThantSinTun_20260901_180117
ThantSinTun_20260901_181249
ThantSinTun_20260901_213231
ThantSinTun_20260901_213708
ThantSinTun_20260901_215808
ThantSinTun_20260901_215847
ThantSinTun_20260901_220238
ThantSinTun_20260901_220605
ThantSinTun_20260901_221150
ThantSinTun_20260901_221224
ThantSinTun_20260901_223146
ThantSinTun_20260901_223205
ThantSinTun_20260901_223325
ThantSinTun_20260901_223339
ThantSinTun_20260901_223358
ThantSinTun_20260901_223420
ThantSinTun_20260901_223454
ThantSinTun_20260901_223612


#### Diagnosing Failed WAV Files

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 13: Diagnosing Failed WAV Files"
echo "========================================"

echo
echo "=== TRAIN FAILED FILES ==="

while read utt; do
    echo
    echo "----------------------------------------"
    echo "UTT: $utt"

    wav=$(awk -v id="$utt" '$1 == id {print substr($0, index($0,$2))}' data/train/wav.scp)

    echo "WAV entry:"
    echo "$wav"

    if [ -f "$wav" ]; then
        echo "File exists: YES"
        sox --i "$wav" 2>&1 | head -12
    else
        echo "File exists: NO"
    fi
done < /tmp/train_missing.txt

echo
echo "=== TEST FAILED FILES ==="

while read utt; do
    echo
    echo "----------------------------------------"
    echo "UTT: $utt"

    wav=$(awk -v id="$utt" '$1 == id {print substr($0, index($0,$2))}' data/test/wav.scp)

    echo "WAV entry:"
    echo "$wav"

    if [ -f "$wav" ]; then
        echo "File exists: YES"
        sox --i "$wav" 2>&1 | head -12
    else
        echo "File exists: NO"
    fi
done < /tmp/test_missing.txt

echo
echo "========================================"
echo "STEP 13 COMPLETE"
echo "========================================"

STEP 13: Diagnosing Failed WAV Files

=== TRAIN FAILED FILES ===

----------------------------------------
UTT: ThantSinTun_20260901_100446
WAV entry:
/home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_100446.wav
File exists: NO

----------------------------------------
UTT: ThantSinTun_20260901_100803
WAV entry:
/home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_100803.wav
File exists: NO

----------------------------------------
UTT: ThantSinTun_20260901_104053
WAV entry:
/home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_104053.wav
File exists: NO

----------------------------------------
UTT: ThantSinTun_20260901_104402
WAV entry:
/home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSinTun_20260901_104402.wav
File exists: NO

----------------------------------------
UTT: ThantSinTun_20260901_104617
WAV entry:
/home/thant_syn/kaldi/egs/burmese_asr/recordings/ThantSinTun/ThantSin

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

echo "Creating backup of current data directories..."

cp -a data/train data/train_before_missing_wav_cleanup
cp -a data/test data/test_before_missing_wav_cleanup

echo "Backups created:"
echo "  data/train_before_missing_wav_cleanup"
echo "  data/test_before_missing_wav_cleanup"

Creating backup of current data directories...
Backups created:
  data/train_before_missing_wav_cleanup
  data/test_before_missing_wav_cleanup


#### Removing Missing-WAV Utterances

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 14: Removing Missing-WAV Utterances"
echo "========================================"

# Remove train utterances whose WAV files do not exist.
while read utt; do
    sed -i "/^${utt}[[:space:]]/d" data/train/wav.scp
    sed -i "/^${utt}[[:space:]]/d" data/train/text
    sed -i "/^${utt}[[:space:]]/d" data/train/utt2spk
done < /tmp/train_missing.txt

# Remove test utterances whose WAV files do not exist.
while read utt; do
    sed -i "/^${utt}[[:space:]]/d" data/test/wav.scp
    sed -i "/^${utt}[[:space:]]/d" data/test/text
    sed -i "/^${utt}[[:space:]]/d" data/test/utt2spk
done < /tmp/test_missing.txt

# Recreate spk2utt.
utils/utt2spk_to_spk2utt.pl data/train/utt2spk > data/train/spk2utt
utils/utt2spk_to_spk2utt.pl data/test/utt2spk > data/test/spk2utt

# Sort the relevant files.
for f in wav.scp text utt2spk spk2utt; do
    sort -k1,1 "data/train/$f" -o "data/train/$f"
    sort -k1,1 "data/test/$f" -o "data/test/$f"
done

echo
echo "=== New counts ==="

echo "TRAIN:"
wc -l data/train/wav.scp data/train/text data/train/utt2spk data/train/spk2utt

echo
echo "TEST:"
wc -l data/test/wav.scp data/test/text data/test/utt2spk data/test/spk2utt

echo
echo "========================================"
echo "STEP 14 COMPLETE"
echo "========================================"

STEP 14: Removing Missing-WAV Utterances

=== New counts ===
TRAIN:
  2999 data/train/wav.scp
  2999 data/train/text
  2999 data/train/utt2spk
    12 data/train/spk2utt
  9009 total

TEST:
  1502 data/test/wav.scp
  1502 data/test/text
  1502 data/test/utt2spk
     6 data/test/spk2utt
  4512 total

STEP 14 COMPLETE


| Dataset | Before | Missing WAVs |       Now |
| ------- | -----: | -----------: | --------: |
| Train   |  3,057 |           58 | **2,999** |
| Test    |  1,521 |           19 | **1,502** |

wav.scp  = text = utt2spk = feats.scp


#### Finalized data

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 15: Validating Feature Data"
echo "========================================"

echo
echo "=== TRAIN ==="

utils/validate_data_dir.sh --non-print data/train

echo
echo "=== TEST ==="

utils/validate_data_dir.sh --non-print data/test

echo
echo "========================================"
echo "Checking feature counts"
echo "========================================"

echo
echo "TRAIN:"
echo "wav.scp : $(wc -l < data/train/wav.scp)"
echo "text    : $(wc -l < data/train/text)"
echo "utt2spk : $(wc -l < data/train/utt2spk)"
echo "feats   : $(wc -l < data/train/feats.scp)"
echo "cmvn    : $(wc -l < data/train/cmvn.scp)"

echo
echo "TEST:"
echo "wav.scp : $(wc -l < data/test/wav.scp)"
echo "text    : $(wc -l < data/test/text)"
echo "utt2spk : $(wc -l < data/test/utt2spk)"
echo "feats   : $(wc -l < data/test/feats.scp)"
echo "cmvn    : $(wc -l < data/test/cmvn.scp)"

echo
echo "========================================"
echo "STEP 15 COMPLETE"
echo "========================================"

STEP 15: Validating Feature Data

=== TRAIN ===
utils/validate_data_dir.sh: Successfully validated data-directory data/train

=== TEST ===
utils/validate_data_dir.sh: Successfully validated data-directory data/test

Checking feature counts

TRAIN:
wav.scp : 2999
text    : 2999
utt2spk : 2999
feats   : 2999
cmvn    : 12

TEST:
wav.scp : 1502
text    : 1502
utt2spk : 1502
feats   : 1502
cmvn    : 6

STEP 15 COMPLETE


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

source ./path.sh

echo "========================================"
echo "STEP 16A: Inspecting Language Resources"
echo "========================================"

echo
echo "=== Dictionary ==="
wc -l data/local/dict/*.txt

echo
echo "=== Language directory ==="
ls -lh data/lang/

echo
echo "=== L.fst ==="
fstinfo data/lang/L.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== L_disambig.fst ==="
fstinfo data/lang/L_disambig.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "========================================"
echo "STEP 16A COMPLETE"
echo "========================================"

STEP 16A: Inspecting Language Resources

=== Dictionary ===
   0 data/local/dict/extra_questions.txt
 138 data/local/dict/lexicon.txt
 138 data/local/dict/lexiconp.txt
  49 data/local/dict/nonsilence_phones.txt
   1 data/local/dict/optional_silence.txt
   1 data/local/dict/silence_phones.txt
 327 total

=== Language directory ===
total 64K
-rw-r--r-- 1 thant_syn thant_syn  17K Sep 10 07:32 L.fst
-rw-r--r-- 1 thant_syn thant_syn  17K Sep 10 07:32 L_disambig.fst
-rw-r--r-- 1 thant_syn thant_syn    2 Sep 10 07:32 oov.int
-rw-r--r-- 1 thant_syn thant_syn    6 Sep 10 07:32 oov.txt
drwxr-xr-x 2 thant_syn thant_syn 4.0K Sep 10 07:32 phones
-rw-r--r-- 1 thant_syn thant_syn 1.9K Sep 10 07:32 phones.txt
-rw-r--r-- 1 thant_syn thant_syn 1.6K Sep 10 07:32 topo
-rw-r--r-- 1 thant_syn thant_syn 2.4K Sep 10 07:32 words.txt

=== L.fst ===
# of states                                       455
# of arcs                                         731
input label sorted                                n
outpu

## Build a simple language model (G.fst)

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 16B: Building Language Model"
echo "========================================"

# Create language-model directory
mkdir -p data/local/lm

# Extract all transcript words from training data
awk '{
    for (i=2; i<=NF; i++)
        print $i
}' data/train/text | sort | uniq -c | sort -k2,2 > data/local/lm/word_counts.txt

echo
echo "=== Vocabulary statistics ==="
echo "Unique words:"
awk 'END {print NR}' data/local/lm/word_counts.txt

echo
echo "Sample word counts:"
head -20 data/local/lm/word_counts.txt

# Create unigram ARPA LM
python3 - <<'PY'
from collections import Counter

text_file = "data/train/text"
arpa_file = "data/local/lm/word.1gram.arpa"

counts = Counter()

with open(text_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        for word in parts[1:]:
            counts[word] += 1

total = sum(counts.values())

with open(arpa_file, "w", encoding="utf-8") as f:
    f.write("\\data\\\n")
    f.write(f"ngram 1={len(counts)}\n\n")
    f.write("\\1-grams:\n")

    for word, count in sorted(counts.items()):
        prob = count / total
        f.write(f"{prob:.8f}\t{word}\n")

    f.write("\n\\end\\\n")

print(f"Created {arpa_file}")
print(f"Total word tokens: {total}")
print(f"Unique words: {len(counts)}")
PY

echo
echo "=== ARPA LM ==="
head -20 data/local/lm/word.1gram.arpa

echo
echo "========================================"
echo "STEP 16B COMPLETE"
echo "========================================"

STEP 16B: Building Language Model

=== Vocabulary statistics ===
Unique words:
136

Sample word counts:
    199 ကို
    200 ကျပ်
    200 ခုပါ
    200 ငွေက
     20 စက်တင်ဘာ
     20 ဇန်နဝါရီ
     20 ဇူလိုင်
     20 ဇွန်လ
     20 တစ်ကြိမ်
     20 ထပ်ပြောပါ
    499 နံပါတ်
     40 နံပါတ်ကို
    100 နှိပ်ပါ
    959 ပါ
     20 ပြောပါ
    199 ဖုန်းနံပါတ်
     20 ဖေဖော်ဝါရီ
     20 ဖြည်းဖြည်း
     20 မတ်လ
     20 မေလ
Created data/local/lm/word.1gram.arpa
Total word tokens: 6994
Unique words: 136

=== ARPA LM ===
\data\
ngram 1=136

\1-grams:
0.02845296	ကို
0.02859594	ကျပ်
0.02859594	ခုပါ
0.02859594	ငွေက
0.00285959	စက်တင်ဘာ
0.00285959	ဇန်နဝါရီ
0.00285959	ဇူလိုင်
0.00285959	ဇွန်လ
0.00285959	တစ်ကြိမ်
0.00285959	ထပ်ပြောပါ
0.07134687	နံပါတ်
0.00571919	နံပါတ်ကို
0.01429797	နှိပ်ပါ
0.13711753	ပါ
0.00285959	ပြောပါ
0.02845296	ဖုန်းနံပါတ်

STEP 16B COMPLETE


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

source ./path.sh

echo "========================================"
echo "STEP 16C: Checking LM Conversion Tools"
echo "========================================"

echo
echo "=== arpa2fst ==="
which arpa2fst || true

echo
echo "=== fstcompile ==="
which fstcompile || true

echo
echo "=== fstarcsort ==="
which fstarcsort || true

echo
echo "=== Checking ARPA file ==="
ls -lh data/local/lm/word.1gram.arpa

echo
echo "========================================"
echo "STEP 16C COMPLETE"
echo "========================================"

STEP 16C: Checking LM Conversion Tools

=== arpa2fst ===

=== fstcompile ===
/home/thant_syn/kaldi/tools/openfst/bin/fstcompile

=== fstarcsort ===
/home/thant_syn/kaldi/tools/openfst/bin/fstarcsort

=== Checking ARPA file ===
-rw-r--r-- 1 thant_syn thant_syn 3.4K Sep 10 07:38 data/local/lm/word.1gram.arpa

STEP 16C COMPLETE


#### Locating arpa2fst

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

source ./path.sh

echo "========================================"
echo "STEP 16D: Locating arpa2fst"
echo "========================================"

echo
echo "=== Searching Kaldi ==="

find "$KALDI_ROOT" -type f -name "arpa2fst" 2>/dev/null

echo
echo "=== Checking lmbin ==="

ls -lh "$KALDI_ROOT/src/lmbin/arpa2fst" 2>/dev/null || true

echo
echo "========================================"
echo "STEP 16D COMPLETE"
echo "========================================"

STEP 16D: Locating arpa2fst

=== Searching Kaldi ===
/home/thant_syn/kaldi/src/lmbin/arpa2fst

=== Checking lmbin ===
-rwxr-xr-x 1 thant_syn thant_syn 2.3M Aug 29 07:23 /home/thant_syn/kaldi/src/lmbin/arpa2fst

STEP 16D COMPLETE


#### Build G.fst

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

echo "========================================"
echo "STEP 16E: Adding arpa2fst and Building G.fst"
echo "========================================"

# Add Kaldi language-model binaries to path.sh if not already present
if ! grep -q 'src/lmbin' path.sh; then
    sed -i 's|src/nnet3bin:$PATH|src/nnet3bin:$KALDI_ROOT/src/lmbin:$PATH|' path.sh
fi

source ./path.sh

echo
echo "=== Checking arpa2fst ==="
which arpa2fst

echo
echo "=== Converting ARPA LM to G.fst ==="

arpa2fst \
    --disambig-symbol=#0 \
    --read-symbol-table=data/lang/words.txt \
    data/local/lm/word.1gram.arpa \
    data/lang/G.fst

echo
echo "=== Sorting G.fst ==="

fstarcsort \
    --sort_type=ilabel \
    data/lang/G.fst \
    data/lang/G.fst.tmp

mv data/lang/G.fst.tmp data/lang/G.fst

echo
echo "=== G.fst information ==="

fstinfo data/lang/G.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== G.fst file ==="
ls -lh data/lang/G.fst

echo
echo "========================================"
echo "STEP 16E COMPLETE"
echo "========================================"

STEP 16E: Adding arpa2fst and Building G.fst

=== Checking arpa2fst ===
/home/thant_syn/kaldi/src/lmbin/arpa2fst

=== Converting ARPA LM to G.fst ===


arpa2fst --disambig-symbol=#0 --read-symbol-table=data/lang/words.txt data/local/lm/word.1gram.arpa data/lang/G.fst 
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:94) Reading \data\ section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \1-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:RemoveRedundantStates():arpa-lm-compiler.cc:359) Reduced num-states from 1 to 1
ERROR (arpa2fst[5.5.1182~1-e02e3]:Check():arpa-lm-compiler.cc:365) Arpa file did not contain the beginning-of-sentence symbol <s>.

[ Stack-Trace: ]
/home/thant_syn/kaldi/src/lib/libkaldi-base.so(kaldi::MessageLogger::LogMessage() const+0x7a8) [0x74d4a153353a]
arpa2fst(kaldi::MessageLogger::LogAndThrow::operator=(kaldi::MessageLogger const&)+0x25) [0x56ab85ccdbd5]
/home/thant_syn/kaldi/src/lib/libkaldi-lm.so(kaldi::ArpaLmCompiler::Check() const+0xe1) [0x74d4a159af15]
/home/thant_syn/kaldi/src/lib/libkaldi-lm.so(kaldi::ArpaLmCompiler::ReadComplete()+0xbd) [0x74d4a159cc11]
/home/thant_syn

CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\n\necho "========================================"\necho "STEP 16E: Adding arpa2fst and Building G.fst"\necho "========================================"\n\n# Add Kaldi language-model binaries to path.sh if not already present\nif ! grep -q \'src/lmbin\' path.sh; then\n    sed -i \'s|src/nnet3bin:$PATH|src/nnet3bin:$KALDI_ROOT/src/lmbin:$PATH|\' path.sh\nfi\n\nsource ./path.sh\n\necho\necho "=== Checking arpa2fst ==="\nwhich arpa2fst\n\necho\necho "=== Converting ARPA LM to G.fst ==="\n\narpa2fst \\\n    --disambig-symbol=#0 \\\n    --read-symbol-table=data/lang/words.txt \\\n    data/local/lm/word.1gram.arpa \\\n    data/lang/G.fst\n\necho\necho "=== Sorting G.fst ==="\n\nfstarcsort \\\n    --sort_type=ilabel \\\n    data/lang/G.fst \\\n    data/lang/G.fst.tmp\n\nmv data/lang/G.fst.tmp data/lang/G.fst\n\necho\necho "=== G.fst information ==="\n\nfstinfo data/lang/G.fst | grep -E "# of states|# of arcs|input label sorte

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 16E-1: Rebuilding Valid ARPA LM"
echo "========================================"

python3 - <<'PY'
from collections import Counter

text_file = "data/train/text"
arpa_file = "data/local/lm/word.1gram.arpa"

counts = Counter()
num_sentences = 0

with open(text_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()

        if len(parts) < 2:
            continue

        num_sentences += 1

        for word in parts[1:]:
            counts[word] += 1

total_words = sum(counts.values())

# Give sentence-boundary symbols probabilities.
# The word probabilities are based on training word frequency.
bos_prob = 1.0
eos_prob = 1.0

with open(arpa_file, "w", encoding="utf-8") as f:

    f.write("\\data\\\n")
    f.write(f"ngram 1={len(counts) + 2}\n")
    f.write("\n")

    f.write("\\1-grams:\n")

    # Beginning/end of sentence
    f.write(f"{bos_prob:.8f}\t<s>\n")
    f.write(f"{eos_prob:.8f}\t</s>\n")

    # Vocabulary
    for word, count in sorted(counts.items()):
        prob = count / total_words
        f.write(f"{prob:.8f}\t{word}\n")

    f.write("\n\\end\\\n")

print(f"Created: {arpa_file}")
print(f"Sentences: {num_sentences}")
print(f"Total word tokens: {total_words}")
print(f"Unique vocabulary words: {len(counts)}")
print(f"Total 1-grams: {len(counts) + 2}")
PY

echo
echo "=== ARPA header ==="
head -8 data/local/lm/word.1gram.arpa

echo
echo "=== Sentence boundary symbols ==="
grep -E $'\t(<s>|</s>)$' data/local/lm/word.1gram.arpa

echo
echo "========================================"
echo "STEP 16E-1 COMPLETE"
echo "========================================"

STEP 16E-1: Rebuilding Valid ARPA LM
Created: data/local/lm/word.1gram.arpa
Sentences: 2999
Total word tokens: 6994
Unique vocabulary words: 136
Total 1-grams: 138

=== ARPA header ===
\data\
ngram 1=138

\1-grams:
1.00000000	<s>
1.00000000	</s>
0.02845296	ကို
0.02859594	ကျပ်

=== Sentence boundary symbols ===
1.00000000	<s>
1.00000000	</s>

STEP 16E-1 COMPLETE


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 16E-2: Converting ARPA to G.fst"
echo "========================================"

echo
echo "=== Checking required symbols ==="

grep -E $'\t(<s>|</s>)$' data/lang/words.txt

echo
echo "=== Converting ARPA LM ==="

rm -f data/lang/G.fst data/lang/G.fst.tmp

arpa2fst \
    --disambig-symbol=#0 \
    --read-symbol-table=data/lang/words.txt \
    data/local/lm/word.1gram.arpa \
    data/lang/G.fst

echo
echo "=== Sorting G.fst ==="

fstarcsort \
    --sort_type=ilabel \
    data/lang/G.fst \
    data/lang/G.fst.tmp

mv data/lang/G.fst.tmp data/lang/G.fst

echo
echo "=== G.fst information ==="

fstinfo data/lang/G.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== G.fst file ==="

ls -lh data/lang/G.fst

echo
echo "========================================"
echo "STEP 16E-2 COMPLETE"
echo "========================================"

STEP 16E-2: Converting ARPA to G.fst

=== Checking required symbols ===


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\nsource ./path.sh\n\necho "========================================"\necho "STEP 16E-2: Converting ARPA to G.fst"\necho "========================================"\n\necho\necho "=== Checking required symbols ==="\n\ngrep -E $\'\\t(<s>|</s>)$\' data/lang/words.txt\n\necho\necho "=== Converting ARPA LM ==="\n\nrm -f data/lang/G.fst data/lang/G.fst.tmp\n\narpa2fst \\\n    --disambig-symbol=#0 \\\n    --read-symbol-table=data/lang/words.txt \\\n    data/local/lm/word.1gram.arpa \\\n    data/lang/G.fst\n\necho\necho "=== Sorting G.fst ==="\n\nfstarcsort \\\n    --sort_type=ilabel \\\n    data/lang/G.fst \\\n    data/lang/G.fst.tmp\n\nmv data/lang/G.fst.tmp data/lang/G.fst\n\necho\necho "=== G.fst information ==="\n\nfstinfo data/lang/G.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"\n\necho\necho "=== G.fst file ==="\n\nls -lh data/lang/G.fst\n\necho\necho "======================================

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

source ./path.sh

echo "========================================"
echo "STEP 16E-2a: Checking Word Symbols"
echo "========================================"

echo
echo "=== First 20 entries ==="
head -20 data/lang/words.txt

echo
echo "=== Sentence symbols ==="
grep -nE '(<s>|</s>)' data/lang/words.txt || true

echo
echo "=== Special symbols ==="
grep -nE '(<eps>|#0|#1|<UNK>)' data/lang/words.txt || true

echo
echo "========================================"
echo "STEP 16E-2a COMPLETE"
echo "========================================"

STEP 16E-2a: Checking Word Symbols

=== First 20 entries ===
<eps> 0
<UNK> 1
SIL 2
ကို 3
ကျပ် 4
ခုပါ 5
ငွေက 6
စက်တင်ဘာ 7
ဇန်နဝါရီ 8
ဇူလိုင် 9
ဇွန်လ 10
တစ်ကြိမ် 11
ထပ်ပြောပါ 12
နံပါတ် 13
နံပါတ်ကို 14
နှိပ်ပါ 15
ပါ 16
ပြောပါ 17
ဖုန်းနံပါတ် 18
ဖေဖော်ဝါရီ 19

=== Sentence symbols ===
141:<s> 140
142:</s> 141

=== Special symbols ===
1:<eps> 0
2:<UNK> 1
140:#0 139

STEP 16E-2a COMPLETE


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 16E-2b: Converting ARPA to G.fst"
echo "========================================"

echo
echo "=== Checking sentence symbols ==="

grep -nE '(<s>|</s>)' data/lang/words.txt

echo
echo "=== Converting ARPA LM ==="

rm -f data/lang/G.fst data/lang/G.fst.tmp

arpa2fst \
    --disambig-symbol=#0 \
    --read-symbol-table=data/lang/words.txt \
    data/local/lm/word.1gram.arpa \
    data/lang/G.fst

echo
echo "=== Sorting G.fst ==="

fstarcsort \
    --sort_type=ilabel \
    data/lang/G.fst \
    data/lang/G.fst.tmp

mv data/lang/G.fst.tmp data/lang/G.fst

echo
echo "=== G.fst information ==="

fstinfo data/lang/G.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== G.fst file ==="

ls -lh data/lang/G.fst

echo
echo "========================================"
echo "STEP 16E-2b COMPLETE"
echo "========================================"

STEP 16E-2b: Converting ARPA to G.fst

=== Checking sentence symbols ===
141:<s> 140
142:</s> 141

=== Converting ARPA LM ===


arpa2fst --disambig-symbol=#0 --read-symbol-table=data/lang/words.txt data/local/lm/word.1gram.arpa data/lang/G.fst 
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:94) Reading \data\ section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \1-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:RemoveRedundantStates():arpa-lm-compiler.cc:359) Reduced num-states from 1 to 1



=== Sorting G.fst ===

=== G.fst information ===
# of states                                       1
# of arcs                                         136
input label sorted                                y
output label sorted                               y

=== G.fst file ===
-rw-r--r-- 1 thant_syn thant_syn 2.3K Sep 10 07:42 data/lang/G.fst

STEP 16E-2b COMPLETE


## Compose L.fst + G.fst

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 17: Composing L.fst + G.fst"
echo "========================================"

echo
echo "=== Checking L.fst ==="
fstinfo data/lang/L.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== Checking G.fst ==="
fstinfo data/lang/G.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== Composing L.fst and G.fst ==="

fsttablecompose \
    data/lang/L.fst \
    data/lang/G.fst \
    data/lang/LG.fst

echo
echo "=== Sorting LG.fst ==="

fstarcsort \
    --sort_type=ilabel \
    data/lang/LG.fst \
    data/lang/LG.fst.tmp

mv data/lang/LG.fst.tmp data/lang/LG.fst

echo
echo "=== LG.fst information ==="

fstinfo data/lang/LG.fst | grep -E "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== LG.fst file ==="

ls -lh data/lang/LG.fst

echo
echo "========================================"
echo "STEP 17 COMPLETE"
echo "========================================"

STEP 17: Composing L.fst + G.fst

=== Checking L.fst ===
# of states                                       455
# of arcs                                         731
input label sorted                                n
output label sorted                               y

=== Checking G.fst ===
# of states                                       1
# of arcs                                         136
input label sorted                                y
output label sorted                               y

=== Composing L.fst and G.fst ===


fsttablecompose data/lang/L.fst data/lang/G.fst data/lang/LG.fst 



=== Sorting LG.fst ===

=== LG.fst information ===
# of states                                       455
# of arcs                                         727
input label sorted                                y
output label sorted                               n

=== LG.fst file ===
-rw-r--r-- 1 thant_syn thant_syn 17K Sep 10 07:42 data/lang/LG.fst

STEP 17 COMPLETE


Current project status

You now have:
```
                    BURMESE ASR
                         │
          ┌──────────────┴──────────────┐
          │                             │
       AUDIO                         TEXT
          │                             │
       2999 train                   2999 train
       1502 test                    1502 test
          │                             │
         MFCC                          LM
          │                             │
         CMVN                      136 words
          │                             │
          │                           G.fst ✅
          │                             │
          └──────────────┬──────────────┘
                         │
                    L.fst + G.fst
                         │
                         ▼
                       LG.fst ✅
```

### Verifying Acoustic Model Setup

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 18: Verifying Acoustic Model Setup"
echo "========================================"

echo
echo "=== Feature dimension ==="

feat-to-dim \
    scp:data/train/feats.scp \
    -

echo
echo "=== Number of phones ==="

echo "Nonsilence phones:"
wc -l data/local/dict/nonsilence_phones.txt

echo "Silence phones:"
wc -l data/local/dict/silence_phones.txt

echo
echo "=== Phone symbols ==="

head -10 data/lang/phones.txt

echo
echo "=== Topology ==="

cat data/lang/topo

echo
echo "=== Training data ==="

echo "Utterances: $(wc -l < data/train/text)"
echo "Speakers:   $(wc -l < data/train/spk2utt)"
echo "Features:   $(wc -l < data/train/feats.scp)"

echo
echo "========================================"
echo "STEP 18 COMPLETE"
echo "========================================"

STEP 18: Verifying Acoustic Model Setup

=== Feature dimension ===


feat-to-dim scp:data/train/feats.scp - 


13

=== Number of phones ===
Nonsilence phones:
49 data/local/dict/nonsilence_phones.txt
Silence phones:
1 data/local/dict/silence_phones.txt

=== Phone symbols ===
<eps> 0
SIL 1
SIL_B 2
SIL_E 3
SIL_I 4
SIL_S 5
,_B 6
,_E 7
,_I 8
,_S 9

=== Topology ===
<Topology>
<TopologyEntry>
<ForPhones>
6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201
</ForPhones>
<State> 0

## Monophone Acoustic Model

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 19: Monophone Acoustic Model"
echo "========================================"

echo
echo "=== Checking training data ==="

utils/validate_data_dir.sh --non-print data/train

echo
echo "=== Checking feature dimension ==="

feat-to-dim scp:data/train/feats.scp -

echo
echo "=== Preparing monophone training directory ==="

rm -rf exp/mono

mkdir -p exp/mono

echo
echo "=== Starting monophone training ==="

steps/train_mono.sh \
    --nj 4 \
    --cmd run.pl \
    data/train \
    data/lang \
    exp/mono

echo
echo "========================================"
echo "STEP 19 COMPLETE"
echo "========================================"

echo
echo "=== Monophone model files ==="

ls -lh exp/mono/final.mdl
ls -lh exp/mono/tree

STEP 19: Monophone Acoustic Model

=== Checking training data ===
utils/validate_data_dir.sh: Successfully validated data-directory data/train

=== Checking feature dimension ===


feat-to-dim scp:data/train/feats.scp - 


13

=== Preparing monophone training directory ===

=== Starting monophone training ===
steps/train_mono.sh --nj 4 --cmd run.pl data/train data/lang exp/mono
steps/train_mono.sh: Initializing monophone system.
steps/train_mono.sh: Compiling training graphs
steps/train_mono.sh: Aligning data equally (pass 0)
steps/train_mono.sh: Pass 1
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 2
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 3
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 4
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 5
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 6
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 7
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 8
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 9
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 10
steps/train_mono.sh: Aligning data
steps/train_mono.sh: Pass 11
steps/train_mono.sh: Pass 12
st

                    BURMESE ASR
                         │
                         ▼
                  Data preparation
                         │
                    2,999 train
                    1,502 test
                         │
                         ▼
                       MFCC
                         │
                       13-D
                         │
                         ▼
                        CMVN
                         │
                         ▼
                  Dictionary + L.fst
                         │
                         ▼
                       G.fst
                         │
                         ▼
                       LG.fst
                         │
                         ▼
                 MONOPHONE GMM-HMM
                         │
                         ▼
                    exp/mono/final.mdl
                         ✅

### Monophone Alignment

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 20: Monophone Alignment"
echo "========================================"

echo
echo "=== Checking monophone model ==="

ls -lh exp/mono/final.mdl
ls -lh exp/mono/tree

echo
echo "=== Starting alignment ==="

rm -rf exp/mono_ali

steps/align_si.sh \
    --nj 4 \
    --cmd run.pl \
    data/train \
    data/lang \
    exp/mono \
    exp/mono_ali

echo
echo "========================================"
echo "STEP 20 COMPLETE"
echo "========================================"

echo
echo "=== Alignment files ==="

ls -lh exp/mono_ali/final.mdl
ls -lh exp/mono_ali/ali.*.gz

STEP 20: Monophone Alignment

=== Checking monophone model ===
lrwxrwxrwx 1 thant_syn thant_syn 6 Sep 10 07:48 exp/mono/final.mdl -> 40.mdl
-rw-r--r-- 1 thant_syn thant_syn 5.0K Sep 10 07:44 exp/mono/tree

=== Starting alignment ===
steps/align_si.sh --nj 4 --cmd run.pl data/train data/lang exp/mono exp/mono_ali
steps/align_si.sh: feature type is delta
steps/align_si.sh: aligning data in data/train using model from exp/mono, putting alignments in exp/mono_ali
steps/diagnostic/analyze_alignments.sh --cmd run.pl data/lang exp/mono_ali
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 75.31730126920507% of the time at utterance end.  This may not be optimal.
steps/diagnostic/analyze_alignments.sh: see stats in exp/mono_ali/log/analyze_alignments.log
steps/align_si.sh: done aligning data.

STEP 20 COMPLETE

=== Alignment files ===
-rw-r--r-- 1 thant_syn thant_syn 345K Sep 10 07:49 exp/mono_ali/final.mdl
-rw-r--r-- 1 thant_syn thant_syn 48K Sep 10 07:49 exp/mono_ali/

## Triphone GMM-HMM Training

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 21: Triphone GMM-HMM Training"
echo "========================================"

echo
echo "=== Checking monophone alignments ==="

ls -lh exp/mono_ali/final.mdl
ls -lh exp/mono_ali/ali.*.gz

echo
echo "=== Starting triphone training ==="

rm -rf exp/tri1

steps/train_deltas.sh \
    --cmd run.pl \
    2000 \
    10000 \
    data/train \
    data/lang \
    exp/mono_ali \
    exp/tri1

echo
echo "========================================"
echo "STEP 21 COMPLETE"
echo "========================================"

echo
echo "=== Triphone model ==="

ls -lh exp/tri1/final.mdl
ls -lh exp/tri1/tree

echo
echo "=== Model summary ==="

gmm-info exp/tri1/final.mdl | head -30

STEP 21: Triphone GMM-HMM Training

=== Checking monophone alignments ===
-rw-r--r-- 1 thant_syn thant_syn 345K Sep 10 07:49 exp/mono_ali/final.mdl
-rw-r--r-- 1 thant_syn thant_syn 48K Sep 10 07:49 exp/mono_ali/ali.1.gz
-rw-r--r-- 1 thant_syn thant_syn 50K Sep 10 07:49 exp/mono_ali/ali.2.gz
-rw-r--r-- 1 thant_syn thant_syn 63K Sep 10 07:50 exp/mono_ali/ali.3.gz
-rw-r--r-- 1 thant_syn thant_syn 47K Sep 10 07:49 exp/mono_ali/ali.4.gz

=== Starting triphone training ===
steps/train_deltas.sh --cmd run.pl 2000 10000 data/train data/lang exp/mono_ali exp/tri1
steps/train_deltas.sh: accumulating tree stats
steps/train_deltas.sh: getting questions for tree-building, via clustering
steps/train_deltas.sh: building the tree
WARNING (gmm-init-model[5.5.1182~1-e02e3]:InitAmGmm():gmm-init-model.cc:55) Tree has pdf-id 49 with no stats; corresponding phone list: 198 199 200 201 
** The warnings above about 'no stats' generally mean you have phones **
** (or groups of phones) in your phone set that ha

gmm-info exp/tri1/final.mdl 


number of phones 201
number of pdfs 833
number of transition-ids 6096
number of transition-states 3028
feature dimension 39
number of gaussians 10050


### Align Training Data with the Triphone Model

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 22: Triphone Alignment"
echo "========================================"

echo
echo "=== Checking triphone model ==="

ls -lh exp/tri1/final.mdl
ls -lh exp/tri1/tree

echo
echo "=== Starting triphone alignment ==="

rm -rf exp/tri1_ali

steps/align_si.sh \
    --nj 4 \
    --cmd run.pl \
    data/train \
    data/lang \
    exp/tri1 \
    exp/tri1_ali

echo
echo "========================================"
echo "STEP 22 COMPLETE"
echo "========================================"

echo
echo "=== Alignment files ==="

ls -lh exp/tri1_ali/final.mdl
ls -lh exp/tri1_ali/ali.*.gz

STEP 22: Triphone Alignment

=== Checking triphone model ===
lrwxrwxrwx 1 thant_syn thant_syn 6 Sep 10 07:53 exp/tri1/final.mdl -> 35.mdl
-rw-r--r-- 1 thant_syn thant_syn 158K Sep 10 07:50 exp/tri1/tree

=== Starting triphone alignment ===
steps/align_si.sh --nj 4 --cmd run.pl data/train data/lang exp/tri1 exp/tri1_ali
steps/align_si.sh: feature type is delta
steps/align_si.sh: aligning data in data/train using model from exp/tri1, putting alignments in exp/tri1_ali
steps/diagnostic/analyze_alignments.sh --cmd run.pl data/lang exp/tri1_ali
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 74.54120787454121% of the time at utterance end.  This may not be optimal.
steps/diagnostic/analyze_alignments.sh: see stats in exp/tri1_ali/log/analyze_alignments.log
steps/align_si.sh: done aligning data.

STEP 22 COMPLETE

=== Alignment files ===
-rw-r--r-- 1 thant_syn thant_syn 3.3M Sep 10 07:53 exp/tri1_ali/final.mdl
-rw-r--r-- 1 thant_syn thant_syn 62K Sep 10 07:54 exp/tr

                    BURMESE ASR
                        │
                        ▼
                 WAV + transcripts
                        │
                        ▼
                  MFCC + CMVN
                        │
                        ▼
              Monophone GMM-HMM
                    ✅ Step 19
                        │
                        ▼
              Monophone alignment
                    ✅ Step 20
                        │
                        ▼
              Triphone GMM-HMM
                    ✅ Step 21
                        │
                        ▼
              Triphone alignment
                    ✅ Step 22
                        │
                        ▼
                 DECODING TEST
                        │
                        ▼
                   WER / CER

### Triphone Decoding Graph

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 23: Build Triphone Decoding Graph"
echo "========================================"

echo
echo "=== Checking required files ==="

ls -lh exp/tri1/final.mdl
ls -lh exp/tri1/tree
ls -lh data/lang/L.fst
ls -lh data/lang/G.fst

echo
echo "=== Building decoding graph ==="

rm -rf exp/tri1/graph
utils/mkgraph.sh \
    data/lang \
    exp/tri1 \
    exp/tri1/graph

echo
echo "========================================"
echo "STEP 23 COMPLETE"
echo "========================================"

echo
echo "=== Graph files ==="

ls -lh exp/tri1/graph/HCLG.fst
ls -lh exp/tri1/graph/words.txt
ls -lh exp/tri1/graph/phones.txt

STEP 23: Build Triphone Decoding Graph

=== Checking required files ===
lrwxrwxrwx 1 thant_syn thant_syn 6 Sep 10 07:53 exp/tri1/final.mdl -> 35.mdl
-rw-r--r-- 1 thant_syn thant_syn 158K Sep 10 07:50 exp/tri1/tree
-rw-r--r-- 1 thant_syn thant_syn 17K Sep 10 07:32 data/lang/L.fst
-rw-r--r-- 1 thant_syn thant_syn 2.3K Sep 10 07:42 data/lang/G.fst

=== Building decoding graph ===


tree-info exp/tri1/tree 
tree-info exp/tri1/tree 
fsttablecompose data/lang/L_disambig.fst data/lang/G.fst 
fstpushspecial 
fstminimizeencoded 
fstdeterminizestar --use-log=true 
fstisstochastic data/lang/tmp/LG.fst 


-2.48043 -2.48044
[info]: LG not stochastic.


fstcomposecontext --context-size=3 --central-position=1 --read-disambig-syms=data/lang/phones/disambig.int --write-disambig-syms=data/lang/tmp/disambig_ilabels_3_1.int data/lang/tmp/ilabels_3_1.20726 data/lang/tmp/LG.fst 
fstisstochastic data/lang/tmp/CLG_3_1.fst 


0 -2.48044
[info]: CLG not stochastic.


make-h-transducer --disambig-syms-out=exp/tri1/graph/disambig_tid.int --transition-scale=1.0 data/lang/tmp/ilabels_3_1 exp/tri1/tree exp/tri1/final.mdl 
fsttablecompose exp/tri1/graph/Ha.fst data/lang/tmp/CLG_3_1.fst 
fstdeterminizestar --use-log=true 
fstminimizeencoded 
fstrmsymbols exp/tri1/graph/disambig_tid.int 
fstrmepslocal 
fstisstochastic exp/tri1/graph/HCLGa.fst 


0.000472116 -4.59378
HCLGa is not stochastic


add-self-loops --self-loop-scale=0.1 --reorder=true exp/tri1/final.mdl exp/tri1/graph/HCLGa.fst 



STEP 23 COMPLETE

=== Graph files ===
-rw-r--r-- 1 thant_syn thant_syn 187K Sep 10 07:54 exp/tri1/graph/HCLG.fst
-rw-r--r-- 1 thant_syn thant_syn 2.4K Sep 10 07:54 exp/tri1/graph/words.txt
-rw-r--r-- 1 thant_syn thant_syn 1.9K Sep 10 07:54 exp/tri1/graph/phones.txt


### Decode the Burmese Test Set

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 24: Decode Test Set"
echo "========================================"

echo
echo "=== Checking decoding graph ==="

ls -lh exp/tri1/graph/HCLG.fst

echo
echo "=== Checking test data ==="

wc -l data/test/wav.scp
wc -l data/test/text
wc -l data/test/utt2spk
wc -l data/test/feats.scp

echo
echo "=== Starting decoding ==="

rm -rf exp/tri1/decode_test

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri1/graph \
    data/test \
    exp/tri1/decode_test

echo
echo "========================================"
echo "STEP 24 COMPLETE"
echo "========================================"

echo
echo "=== Decode results ==="

ls -lh exp/tri1/decode_test/

echo
echo "=== Number of decoded utterances ==="

cat exp/tri1/decode_test/num_jobs

STEP 24: Decode Test Set

=== Checking decoding graph ===
-rw-r--r-- 1 thant_syn thant_syn 187K Sep 10 07:54 exp/tri1/graph/HCLG.fst

=== Checking test data ===
1502 data/test/wav.scp
1502 data/test/text
1502 data/test/utt2spk
1502 data/test/feats.scp

=== Starting decoding ===
steps/decode.sh --nj 4 --cmd run.pl exp/tri1/graph data/test exp/tri1/decode_test
decode.sh: feature type is delta
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri1/graph exp/tri1/decode_test


run.pl: 4 / 4 failed, log is in exp/tri1/decode_test/log/lattice_best_path.*.log


steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\nsource ./path.sh\n\necho "========================================"\necho "STEP 24: Decode Test Set"\necho "========================================"\n\necho\necho "=== Checking decoding graph ==="\n\nls -lh exp/tri1/graph/HCLG.fst\n\necho\necho "=== Checking test data ==="\n\nwc -l data/test/wav.scp\nwc -l data/test/text\nwc -l data/test/utt2spk\nwc -l data/test/feats.scp\n\necho\necho "=== Starting decoding ==="\n\nrm -rf exp/tri1/decode_test\n\nsteps/decode.sh \\\n    --nj 4 \\\n    --cmd run.pl \\\n    exp/tri1/graph \\\n    data/test \\\n    exp/tri1/decode_test\n\necho\necho "========================================"\necho "STEP 24 COMPLETE"\necho "========================================"\n\necho\necho "=== Decode results ==="\n\nls -lh exp/tri1/decode_test/\n\necho\necho "=== Number of decoded utterances ==="\n\ncat exp/tri1/decode_test/num_jobs\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

source ./path.sh

echo "========================================"
echo "STEP 24A: Inspect Decoding Error"
echo "========================================"

echo
echo "=== Job 1 log ==="
cat exp/tri1/decode_test/log/lattice_best_path.1.log

echo
echo "========================================"
echo "=== Job 2 log ==="
echo "========================================"
cat exp/tri1/decode_test/log/lattice_best_path.2.log

echo
echo "========================================"
echo "=== Job 3 log ==="
echo "========================================"
cat exp/tri1/decode_test/log/lattice_best_path.3.log

echo
echo "========================================"
echo "=== Job 4 log ==="
echo "========================================"
cat exp/tri1/decode_test/log/lattice_best_path.4.log

STEP 24A: Inspect Decoding Error

=== Job 1 log ===
# lattice-depth-per-frame "ark:gunzip -c exp/tri1/decode_test/lat.1.gz|" "ark,t:|gzip -c > exp/tri1/decode_test/depth_tmp.1.gz" ark:- | lattice-best-path --acoustic-scale=0.1 ark:- ark:/dev/null "ark,t:|gzip -c >exp/tri1/decode_test/ali_tmp.1.gz" 
# Started at Thu Sep 10 08:06:14 UTC 2026
#
bash: line 1: lattice-best-path: command not found
bash: line 1: lattice-depth-per-frame: command not found
# Accounting: time=0 threads=1
# Ended (code 127) at Thu Sep 10 08:06:14 UTC 2026, elapsed time 0 seconds

=== Job 2 log ===
# lattice-depth-per-frame "ark:gunzip -c exp/tri1/decode_test/lat.2.gz|" "ark,t:|gzip -c > exp/tri1/decode_test/depth_tmp.2.gz" ark:- | lattice-best-path --acoustic-scale=0.1 ark:- ark:/dev/null "ark,t:|gzip -c >exp/tri1/decode_test/ali_tmp.2.gz" 
# Started at Thu Sep 10 08:06:14 UTC 2026
#
bash: line 1: lattice-depth-per-frame: command not found
bash: line 1: lattice-best-path: command not found
# Accounting: time=0 th

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

echo "========================================"
echo "STEP 24B: Fix Kaldi Lattice PATH"
echo "========================================"

source ./path.sh

echo
echo "=== Checking lattice binaries ==="

ls -lh "$KALDI_ROOT/src/latbin/lattice-best-path"
ls -lh "$KALDI_ROOT/src/latbin/lattice-depth-per-frame"

echo
echo "=== Adding latbin to PATH ==="

export PATH="$KALDI_ROOT/src/latbin:$PATH"

echo
echo "=== Verifying commands ==="

command -v lattice-best-path
command -v lattice-depth-per-frame

echo
echo "=== Versions / help ==="

lattice-best-path --help 2>&1 | head -5
lattice-depth-per-frame --help 2>&1 | head -5

echo
echo "========================================"
echo "STEP 24B COMPLETE"
echo "========================================"

STEP 24B: Fix Kaldi Lattice PATH

=== Checking lattice binaries ===
-rwxr-xr-x 1 thant_syn thant_syn 3.7M Aug 29 08:01 /home/thant_syn/kaldi/src/latbin/lattice-best-path
-rwxr-xr-x 1 thant_syn thant_syn 2.8M Aug 29 08:01 /home/thant_syn/kaldi/src/latbin/lattice-depth-per-frame

=== Adding latbin to PATH ===

=== Verifying commands ===
/home/thant_syn/kaldi/src/latbin/lattice-best-path
/home/thant_syn/kaldi/src/latbin/lattice-depth-per-frame

=== Versions / help ===

Generate 1-best path through lattices; output as transcriptions and alignments
Note: if you want output as FSTs, use lattice-1best; if you want output
with acoustic and LM scores, use lattice-1best | nbest-to-linear
Usage: lattice-best-path [options]  <lattice-rspecifier> [ <transcriptions-wspecifier> [ <alignments-wspecifier>] ]

For each lattice, compute a vector of length (num-frames) saying how
may arcs cross each frame.  See also lattice-depth
Usage: lattice-depth-per-frame <lattice-rspecifier> <depth-wspecifier> [<lat

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e

if ! grep -q 'src/latbin' ./path.sh; then
    sed -i '/src\/lmbin/a export PATH="$KALDI_ROOT/src/latbin:$PATH"' ./path.sh
fi

echo "=== Current path.sh ==="
cat ./path.sh

echo
echo "=== Verify ==="

source ./path.sh
command -v lattice-best-path
command -v lattice-depth-per-frame

=== Current path.sh ===
#!/usr/bin/env bash

export KALDI_ROOT=/home/thant_syn/kaldi

export PATH="$KALDI_ROOT/tools/openfst/bin:$KALDI_ROOT/src/bin:$KALDI_ROOT/src/fstbin:$KALDI_ROOT/src/featbin:$KALDI_ROOT/src/gmmbin:$KALDI_ROOT/src/nnet3bin:$KALDI_ROOT/src/lmbin:$PATH"
export PATH="$KALDI_ROOT/src/latbin:$PATH"

export LC_ALL=C

=== Verify ===
/home/thant_syn/kaldi/src/latbin/lattice-best-path
/home/thant_syn/kaldi/src/latbin/lattice-depth-per-frame


In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 24D: Retry Test Decoding"
echo "========================================"

echo
echo "=== Decoder tools ==="
command -v lattice-best-path
command -v lattice-depth-per-frame

echo
echo "=== Cleaning previous failed decoding ==="
rm -rf exp/tri1/decode_test

echo
echo "=== Starting decoding ==="

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri1/graph \
    data/test \
    exp/tri1/decode_test

echo
echo "========================================"
echo "STEP 24D COMPLETE"
echo "========================================"

echo
echo "=== Decode output ==="
ls -lh exp/tri1/decode_test/

STEP 24D: Retry Test Decoding

=== Decoder tools ===
/home/thant_syn/kaldi/src/latbin/lattice-best-path
/home/thant_syn/kaldi/src/latbin/lattice-depth-per-frame

=== Cleaning previous failed decoding ===

=== Starting decoding ===
steps/decode.sh --nj 4 --cmd run.pl exp/tri1/graph data/test exp/tri1/decode_test
decode.sh: feature type is delta
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri1/graph exp/tri1/decode_test
steps/diagnostic/analyze_lats.sh: see stats in exp/tri1/decode_test/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(14,105,748) and mean=300.1
steps/diagnostic/analyze_lats.sh: see stats in exp/tri1/decode_test/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.


CalledProcessError: Command 'b'cd ~/kaldi/egs/burmese_asr\n\nset -e\nsource ./path.sh\n\necho "========================================"\necho "STEP 24D: Retry Test Decoding"\necho "========================================"\n\necho\necho "=== Decoder tools ==="\ncommand -v lattice-best-path\ncommand -v lattice-depth-per-frame\n\necho\necho "=== Cleaning previous failed decoding ==="\nrm -rf exp/tri1/decode_test\n\necho\necho "=== Starting decoding ==="\n\nsteps/decode.sh \\\n    --nj 4 \\\n    --cmd run.pl \\\n    exp/tri1/graph \\\n    data/test \\\n    exp/tri1/decode_test\n\necho\necho "========================================"\necho "STEP 24D COMPLETE"\necho "========================================"\n\necho\necho "=== Decode output ==="\nls -lh exp/tri1/decode_test/\n'' returned non-zero exit status 1.

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 25: Inspect Decoded Hypotheses"
echo "========================================"

echo
echo "=== Decode directory ==="
ls -lh exp/tri1/decode_test/

echo
echo "=== Number of jobs ==="
cat exp/tri1/decode_test/num_jobs

echo
echo "=== Extracting 1-best hypotheses ==="

for j in 1 2 3 4; do
    echo
    echo "--- Job $j ---"
    lattice-best-path \
        "ark:gunzip -c exp/tri1/decode_test/lat.$j.gz|" \
        "ark,t:-" 2>/dev/null | head -5
done

echo
echo "========================================"
echo "STEP 25 COMPLETE"
echo "========================================"

STEP 25: Inspect Decoded Hypotheses

=== Decode directory ===
total 57M
-rw-r--r-- 1 thant_syn thant_syn  13M Sep 10 08:15 lat.1.gz
-rw-r--r-- 1 thant_syn thant_syn  11M Sep 10 08:14 lat.2.gz
-rw-r--r-- 1 thant_syn thant_syn 4.6M Sep 10 08:11 lat.3.gz
-rw-r--r-- 1 thant_syn thant_syn  30M Sep 10 08:19 lat.4.gz
drwxr-xr-x 2 thant_syn thant_syn 4.0K Sep 10 08:19 log
-rw-r--r-- 1 thant_syn thant_syn    2 Sep 10 08:08 num_jobs

=== Number of jobs ===
4

=== Extracting 1-best hypotheses ===

--- Job 1 ---
AungKhantMyat_Rec1_20260907_232253 123 32 
AungKhantMyat_Rec1_20260907_232313 106 43 106 129 
AungKhantMyat_Rec1_20260907_232326 125 
AungKhantMyat_Rec1_20260907_232345 88 
AungKhantMyat_Rec1_20260907_232406 129 77 91 

--- Job 2 ---
AungKhantMyat_Rec3_20260908_142437 77 32 106 
AungKhantMyat_Rec3_20260908_142448 77 43 106 43 
AungKhantMyat_Rec3_20260908_142505 77 6 63 77 125 129 43 6 
AungKhantMyat_Rec3_20260908_142529 129 6 97 121 
AungKhantMyat_Rec3_20260908_142539 97 106 43 106 43 

--

### Convert word IDs to Burmese text

In [ ]:
%%bash
cd ~/kaldi/egs/burmese_asr

set -e
source ./path.sh

echo "========================================"
echo "STEP 26: Convert Word IDs to Text"
echo "========================================"

echo
echo "=== Creating hypothesis directory ==="

rm -rf exp/tri1/decode_test/hyp
mkdir -p exp/tri1/decode_test/hyp

echo
echo "=== Converting each decoding job ==="

for j in 1 2 3 4; do
    lattice-best-path \
        "ark:gunzip -c exp/tri1/decode_test/lat.$j.gz|" \
        ark,t:- 2>/dev/null \
        | utils/int2sym.pl -f 2- exp/tri1/graph/words.txt \
        > exp/tri1/decode_test/hyp/hyp.$j.txt
done

echo
echo "=== Combining hypotheses ==="

cat exp/tri1/decode_test/hyp/hyp.*.txt \
    | sort -k1,1 \
    > exp/tri1/decode_test/hyp.txt

echo
echo "=== First 20 decoded sentences ==="

head -20 exp/tri1/decode_test/hyp.txt

echo
echo "========================================"
echo "STEP 26 COMPLETE"
echo "========================================"

STEP 26: Convert Word IDs to Text

=== Creating hypothesis directory ===

=== Converting each decoding job ===

=== Combining hypotheses ===

=== First 20 decoded sentences ===
AungKhantMyat_Rec1_20260907_232253 ၆၇ ၀ 
AungKhantMyat_Rec1_20260907_232313 ၄ ၁ ၄ ၈ 
AungKhantMyat_Rec1_20260907_232326 ၇ 
AungKhantMyat_Rec1_20260907_232345 ၂၃ 
AungKhantMyat_Rec1_20260907_232406 ၈ ၂ ၂၄ 
AungKhantMyat_Rec1_20260907_232432 ၇ ၄ ၀ ၅ 
AungKhantMyat_Rec1_20260907_232450 ၆ ငွေက 
AungKhantMyat_Rec1_20260907_232502 ၆ ၇ ၂ ငွေက 
AungKhantMyat_Rec1_20260907_232539 ၆ ငွေက ၈၈ ငွေက 
AungKhantMyat_Rec1_20260907_232610 ၁ ၉ ငွေက 
AungKhantMyat_Rec1_20260907_232627 ငွေက ၂ ၇ ၈၂ ၂၄ ၇ ၀ ၀ ၀ ပါ ငွေက 
AungKhantMyat_Rec1_20260907_232641 ၂ ၁ ၁ ၇ ၉ ၁ ၂ ၂ ပါ 
AungKhantMyat_Rec1_20260907_232654 ၂ ၇ ၂၁ ၇ ၂ ပါ 
AungKhantMyat_Rec1_20260907_232744 ၂ ၇ ၃၄ ၂ ပါ 
AungKhantMyat_Rec1_20260907_232804 ငွေက ၇ ၁ ၄ ရက်ပါ 
AungKhantMyat_Rec1_20260907_232824 ၆ ၂ ၇ ၅ ၀ ၀ ၀ 
AungKhantMyat_Rec1_20260907_232850 ၂ ၇ ၃ ၇ ၆ ပါ 
AungKhantMyat_Re

#### Compare Reference vs Hypothesis

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 27: Compare Reference vs Hypothesis"
echo "========================================"

echo
echo "=== Reference (data/test/text) ==="
head -10 data/test/text

echo
echo "=== Hypothesis (decoded) ==="
head -10 exp/tri1/decode_test/hyp.txt

echo
echo "=== Side-by-side comparison ==="

paste \
    <(head -20 data/test/text) \
    <(head -20 exp/tri1/decode_test/hyp.txt) \
    | awk -F '\t' '{print "\nUTT: "$1; print "REF: "$2; print "HYP: "$4}'

echo
echo "========================================"
echo "STEP 27 COMPLETE"
echo "========================================"

STEP 27: Compare Reference vs Hypothesis

=== Reference (data/test/text) ===
AungKhantMyat_Rec1_20260907_232253 ၀
AungKhantMyat_Rec1_20260907_232313 ၁
AungKhantMyat_Rec1_20260907_232326 ၂
AungKhantMyat_Rec1_20260907_232345 ၃
AungKhantMyat_Rec1_20260907_232406 ၄
AungKhantMyat_Rec1_20260907_232432 ၅
AungKhantMyat_Rec1_20260907_232450 ၆
AungKhantMyat_Rec1_20260907_232502 ၇
AungKhantMyat_Rec1_20260907_232539 ၈
AungKhantMyat_Rec1_20260907_232610 ၉

=== Hypothesis (decoded) ===
AungKhantMyat_Rec1_20260907_232253 ၆၇ ၀ 
AungKhantMyat_Rec1_20260907_232313 ၄ ၁ ၄ ၈ 
AungKhantMyat_Rec1_20260907_232326 ၇ 
AungKhantMyat_Rec1_20260907_232345 ၂၃ 
AungKhantMyat_Rec1_20260907_232406 ၈ ၂ ၂၄ 
AungKhantMyat_Rec1_20260907_232432 ၇ ၄ ၀ ၅ 
AungKhantMyat_Rec1_20260907_232450 ၆ ငွေက 
AungKhantMyat_Rec1_20260907_232502 ၆ ၇ ၂ ငွေက 
AungKhantMyat_Rec1_20260907_232539 ၆ ငွေက ၈၈ ငွေက 
AungKhantMyat_Rec1_20260907_232610 ၁ ၉ ငွေက 

=== Side-by-side comparison ===

UTT: AungKhantMyat_Rec1_20260907_232253 ၀
REF: AungKha

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 28: Correct Reference vs Hypothesis"
echo "========================================"

mkdir -p exp/tri1/decode_test/eval

# Remove any accidental trailing spaces
sed 's/[[:space:]]*$//' data/test/text > exp/tri1/decode_test/eval/reference.txt
sed 's/[[:space:]]*$//' exp/tri1/decode_test/hyp.txt > exp/tri1/decode_test/eval/hypothesis.txt

echo
echo "=== Number of reference utterances ==="
wc -l exp/tri1/decode_test/eval/reference.txt

echo
echo "=== Number of hypothesis utterances ==="
wc -l exp/tri1/decode_test/eval/hypothesis.txt

echo
echo "=== Correct side-by-side comparison ==="

join -1 1 -2 1 \
    <(sort -k1,1 exp/tri1/decode_test/eval/reference.txt) \
    <(sort -k1,1 exp/tri1/decode_test/eval/hypothesis.txt) \
    | awk 'BEGIN {
        print "----------------------------------------"
    }
    NR <= 20 {
        ref=$0
        split(ref,a," ")

        # Print using the tab produced by join
        split($0,b,"\t")

        print "UTT: " a[1]
        print "REF: " b[1]
        print "HYP: " b[2]
        print "----------------------------------------"
    }'

echo
echo "========================================"
echo "STEP 28 COMPLETE"
echo "========================================"

STEP 28: Correct Reference vs Hypothesis

=== Number of reference utterances ===
1502 exp/tri1/decode_test/eval/reference.txt

=== Number of hypothesis utterances ===
1502 exp/tri1/decode_test/eval/hypothesis.txt

=== Correct side-by-side comparison ===
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232253
REF: AungKhantMyat_Rec1_20260907_232253 ၀ ၆၇ ၀
HYP: 
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232313
REF: AungKhantMyat_Rec1_20260907_232313 ၁ ၄ ၁ ၄ ၈
HYP: 
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232326
REF: AungKhantMyat_Rec1_20260907_232326 ၂ ၇
HYP: 
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232345
REF: AungKhantMyat_Rec1_20260907_232345 ၃ ၂၃
HYP: 
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232406
REF: AungKhantMyat_Rec1_20260907_232406 ၄ ၈ ၂ ၂၄
HYP: 
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_23

Comparison code is wrong

#### Correct comparison by utterance ID

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 28B: Reliable Reference vs Hypothesis"
echo "========================================"

mkdir -p exp/tri1/decode_test/eval

awk '
FNR == NR {
    id = $1
    ref[id] = substr($0, index($0, $2))
    next
}
{
    id = $1
    hyp[id] = (NF >= 2 ? substr($0, index($0, $2)) : "")
}
END {
    count = 0

    while ((getline line < "data/test/text") > 0) {
        id = $1
        if (id in ref && id in hyp) {
            print "----------------------------------------"
            print "UTT: " id
            print "REF: " ref[id]
            print "HYP: " hyp[id]
            count++

            if (count >= 20)
                break
        }
    }

    close("data/test/text")
}
' data/test/text exp/tri1/decode_test/hyp.txt

echo
echo "========================================"
echo "STEP 28B COMPLETE"
echo "========================================"

STEP 28B: Reliable Reference vs Hypothesis
----------------------------------------
UTT: ThidaAye_20260906_123705
REF: နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ
HYP: ၂၂ ၅ ၉ ၈၀ ၆ ၁ ၂ ၇ ၄ ၀ ပါ 
----------------------------------------
UTT: ThidaAye_20260906_123705
REF: နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ
HYP: ၂၂ ၅ ၉ ၈၀ ၆ ၁ ၂ ၇ ၄ ၀ ပါ 
----------------------------------------
UTT: ThidaAye_20260906_123705
REF: နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ
HYP: ၂၂ ၅ ၉ ၈၀ ၆ ၁ ၂ ၇ ၄ ၀ ပါ 
----------------------------------------
UTT: ThidaAye_20260906_123705
REF: နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ
HYP: ၂၂ ၅ ၉ ၈၀ ၆ ၁ ၂ ၇ ၄ ၀ ပါ 
----------------------------------------
UTT: ThidaAye_20260906_123705
REF: နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ
HYP: ၂၂ ၅ ၉ ၈၀ ၆ ၁ ၂ ၇ ၄ ၀ ပါ 
----------------------------------------
UTT: ThidaAye_20260906_123705
REF: နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ
HYP: ၂၂ ၅ ၉ ၈၀ ၆ ၁ ၂ ၇ ၄ ၀ ပါ 
----------------------------------------
UTT: ThidaAye_20260906_123705
REF: နံပါတ်ကို ဖြည်းဖြည်း ပြောပါ
HYP: ၂၂ ၅ ၉ ၈၀ ၆ ၁ ၂ ၇ ၄ ၀ ပါ 
------------

In [ ]:
# Final reliable comparison

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 28C: Final Reference vs Hypothesis"
echo "========================================"

mkdir -p exp/tri1/decode_test/eval

# Build a lookup table:
# utterance-id -> hypothesis
awk '{
    id=$1
    hyp[id]=""
    for (i=2; i<=NF; i++) {
        if (i>2) hyp[id]=hyp[id]" "
        hyp[id]=hyp[id]$i
    }
}
END {
    for (id in hyp)
        print id "\t" hyp[id]
}' exp/tri1/decode_test/hyp.txt \
> exp/tri1/decode_test/eval/hyp_lookup.txt

echo
echo "=== First 20 comparisons ==="

count=0

while IFS=$'\t' read -r id ref_text; do

    hyp_text=$(awk -F '\t' -v id="$id" '$1 == id {print $2; exit}' \
        exp/tri1/decode_test/eval/hyp_lookup.txt)

    echo "----------------------------------------"
    echo "UTT: $id"
    echo "REF: $ref_text"
    echo "HYP: $hyp_text"

    count=$((count + 1))

    if [ "$count" -ge 20 ]; then
        break
    fi

done < <(
    awk '{
        id=$1
        text=""
        for (i=2; i<=NF; i++) {
            if (i>2) text=text" "
            text=text$i
        }
        print id "\t" text
    }' data/test/text
)

echo
echo "========================================"
echo "STEP 28C COMPLETE"
echo "========================================"

STEP 28C: Final Reference vs Hypothesis

=== First 20 comparisons ===
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232253
REF: ၀
HYP: ၆၇ ၀
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232313
REF: ၁
HYP: ၄ ၁ ၄ ၈
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232326
REF: ၂
HYP: ၇
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232345
REF: ၃
HYP: ၂၃
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232406
REF: ၄
HYP: ၈ ၂ ၂၄
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232432
REF: ၅
HYP: ၇ ၄ ၀ ၅
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232450
REF: ၆
HYP: ၆ ငွေက
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232502
REF: ၇
HYP: ၆ ၇ ၂ ငွေက
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232539
REF: ၈
HYP: ၆ ငွေက ၈၈ ငွေက
---------------------------

#### Tri1 + Unigram WER

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 29: Calculate Word Error Rate (WER)"
echo "========================================"

mkdir -p exp/tri1/decode_test/eval

# Prepare reference
awk '{
    printf "%s", $1
    for (i=2; i<=NF; i++)
        printf " %s", $i
    printf "\n"
}' data/test/text \
> exp/tri1/decode_test/eval/ref.txt

# Prepare hypothesis
awk '{
    printf "%s", $1
    for (i=2; i<=NF; i++)
        printf " %s", $i
    printf "\n"
}' exp/tri1/decode_test/hyp.txt \
> exp/tri1/decode_test/eval/hyp.txt

echo
echo "=== Reference utterances ==="
wc -l exp/tri1/decode_test/eval/ref.txt

echo
echo "=== Hypothesis utterances ==="
wc -l exp/tri1/decode_test/eval/hyp.txt

echo
echo "=== WER RESULT ==="

compute-wer \
    ark:"sort -k1,1 exp/tri1/decode_test/eval/ref.txt|" \
    ark:"sort -k1,1 exp/tri1/decode_test/eval/hyp.txt|" \
    2>&1 | tee exp/tri1/decode_test/eval/wer.txt

echo
echo "========================================"
echo "STEP 29 COMPLETE"
echo "========================================"

STEP 29: Calculate Word Error Rate (WER)

=== Reference utterances ===
1502 exp/tri1/decode_test/eval/ref.txt

=== Hypothesis utterances ===
1502 exp/tri1/decode_test/eval/hyp.txt

=== WER RESULT ===
compute-wer 'ark:sort -k1,1 exp/tri1/decode_test/eval/ref.txt|' 'ark:sort -k1,1 exp/tri1/decode_test/eval/hyp.txt|' 
%WER 373.06 [ 13072 / 3504, 10490 ins, 52 del, 2530 sub ]
%SER 98.93 [ 1486 / 1502 ]
Scored 1502 sentences, 0 not present in hyp.

STEP 29 COMPLETE


#### Burmese Character Error Rate

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 30: Burmese Character Error Rate"
echo "========================================"

mkdir -p exp/tri1/decode_test/eval/cer

python3 <<'PY'
from pathlib import Path

base = Path("exp/tri1/decode_test/eval")

ref_file = base / "ref.txt"
hyp_file = base / "hyp.txt"

refs = {}
hyps = {}

# Read reference
for line in ref_file.read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if parts:
        refs[parts[0]] = "".join(parts[1:])

# Read hypothesis
for line in hyp_file.read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if parts:
        hyps[parts[0]] = "".join(parts[1:])

# Levenshtein distance
def edit_distance(ref, hyp):
    prev = list(range(len(hyp) + 1))

    for i, r in enumerate(ref, start=1):
        curr = [i]

        for j, h in enumerate(hyp, start=1):
            if r == h:
                cost = 0
            else:
                cost = 1

            curr.append(min(
                curr[-1] + 1,       # insertion
                prev[j] + 1,        # deletion
                prev[j-1] + cost    # substitution
            ))

        prev = curr

    return prev[-1]

total_errors = 0
total_chars = 0
sent_errors = 0

for utt in refs:
    ref = refs[utt]
    hyp = hyps.get(utt, "")

    errors = edit_distance(ref, hyp)

    total_errors += errors
    total_chars += len(ref)

    if errors > 0:
        sent_errors += 1

cer = 100.0 * total_errors / total_chars if total_chars else 0.0
ser = 100.0 * sent_errors / len(refs) if refs else 0.0

print()
print("=== CER RESULT ===")
print(f"Reference characters : {total_chars}")
print(f"Character errors     : {total_errors}")
print(f"Sentence errors      : {sent_errors}")
print(f"Total sentences      : {len(refs)}")
print(f"CER                  : {cer:.2f}%")
print(f"Sentence Error Rate   : {ser:.2f}%")

print()
print("=== Sample character-level comparison ===")

count = 0
for utt in refs:
    if count >= 10:
        break

    print("-" * 40)
    print("UTT:", utt)
    print("REF:", refs[utt])
    print("HYP:", hyps.get(utt, ""))

    count += 1

# Save CER summary
summary = (
    f"Reference characters : {total_chars}\n"
    f"Character errors     : {total_errors}\n"
    f"Sentence errors      : {sent_errors}\n"
    f"Total sentences      : {len(refs)}\n"
    f"CER                  : {cer:.2f}%\n"
    f"Sentence Error Rate   : {ser:.2f}%\n"
)

(base / "cer" / "cer_summary.txt").write_text(
    summary,
    encoding="utf-8"
)

PY

echo
echo "========================================"
echo "STEP 30 COMPLETE"
echo "========================================"

STEP 30: Burmese Character Error Rate

=== CER RESULT ===
Reference characters : 15499
Character errors     : 17832
Sentence errors      : 1486
Total sentences      : 1502
CER                  : 115.05%
Sentence Error Rate   : 98.93%

=== Sample character-level comparison ===
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232253
REF: ၀
HYP: ၆၇၀
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232313
REF: ၁
HYP: ၄၁၄၈
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232326
REF: ၂
HYP: ၇
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232345
REF: ၃
HYP: ၂၃
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232406
REF: ၄
HYP: ၈၂၂၄
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232432
REF: ၅
HYP: ၇၄၀၅
----------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232450
REF: ၆
HYP: ၆ငွေက
----------------------------------------
U

#### Analyze base-line error

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 31: Baseline Error Analysis"
echo "========================================"

echo
echo "=== 1. Most frequent reference tokens ==="

awk '{
    for (i=2; i<=NF; i++)
        count[$i]++
}
END {
    for (w in count)
        print count[w], w
}' data/train/text \
| sort -nr \
| head -20

echo
echo "=== 2. Most frequent hypothesis tokens ==="

awk '{
    for (i=2; i<=NF; i++)
        count[$i]++
}
END {
    for (w in count)
        print count[w], w
}' exp/tri1/decode_test/hyp.txt \
| sort -nr \
| head -20

echo
echo "=== 3. Average reference/hypothesis length ==="

python3 <<'PY'
from pathlib import Path

ref = Path("data/test/text")
hyp = Path("exp/tri1/decode_test/hyp.txt")

ref_total = 0
hyp_total = 0
ref_sent = 0
hyp_sent = 0

for line in ref.read_text(encoding="utf-8").splitlines():
    p = line.split()
    if p:
        ref_total += len(p) - 1
        ref_sent += 1

for line in hyp.read_text(encoding="utf-8").splitlines():
    p = line.split()
    if p:
        hyp_total += len(p) - 1
        hyp_sent += 1

print(f"Reference tokens : {ref_total}")
print(f"Hypothesis tokens: {hyp_total}")
print(f"Reference average: {ref_total/ref_sent:.2f}")
print(f"Hypothesis average: {hyp_total/hyp_sent:.2f}")
print(f"Ratio HYP/REF    : {hyp_total/ref_total:.2f}")
PY

echo
echo "=== 4. Number of very short references ==="

awk 'NF == 2 {count++} END {print "One-token utterances:", count}' data/test/text

echo
echo "=== 5. Number of longer references ==="

awk 'NF > 2 {count++} END {print "Multi-token utterances:", count}' data/test/text

echo
echo "========================================"
echo "STEP 31 COMPLETE"
echo "========================================"

STEP 31: Baseline Error Analysis

=== 1. Most frequent reference tokens ===
959 ပါ
499 နံပါတ်
200 ၂၀၂၆
200 အရေအတွက်
200 ရက်ပါ
200 ရက်စွဲက
200 ငွေက
200 ခုပါ
200 ကျပ်
199 ဖုန်းနံပါတ်
199 ကို
160 အော်ဒါနံပါတ်
100 ၅
100 ရွေးချယ်မှု
100 နှိပ်ပါ
99 ရွေးပါ
80 ၇
80 ၂
60 ၉
60 ၆

=== 2. Most frequent hypothesis tokens ===
1915 ၂
1672 ၇
1239 ၀
1171 ၁
969 ၅
812 ၄
662 ပါ
601 ၆
473 ၃
395 ၉
383 ငွေက
298 ၈
285 ၁၂
232 ၂၂
210 ၃၁
208 ရက်ပါ
180 ၁၅
163 ကို
133 ၈၂
106 ၇၄

=== 3. Average reference/hypothesis length ===
Reference tokens : 3504
Hypothesis tokens: 13942
Reference average: 2.33
Hypothesis average: 9.28
Ratio HYP/REF    : 3.98

=== 4. Number of very short references ===
One-token utterances: 701

=== 5. Number of longer references ===
Multi-token utterances: 801

STEP 31 COMPLETE


#### Test Acoustic model On Training data

We'll decode a small portion of the training data using the existing tri1 model.

If training recognition is much better than test recognition, the main issue is likely generalization / acoustic modeling.

If training recognition is also terrible, we need to investigate the lexicon, language model, topology, or acoustic model configuration.

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 32: Training-Set Diagnostic"
echo "========================================"

rm -rf exp/tri1/decode_train_diag

echo
echo "=== Decoding training data ==="

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri1/graph \
    data/train \
    exp/tri1/decode_train_diag

echo
echo "=== Checking decoded lattices ==="

ls -lh exp/tri1/decode_train_diag/lat.*.gz

echo
echo "=== Extracting training hypotheses ==="

mkdir -p exp/tri1/decode_train_diag/hyp

for j in 1 2 3 4; do
    lattice-best-path \
        "ark:gunzip -c exp/tri1/decode_train_diag/lat.$j.gz|" \
        ark,t:- 2>/dev/null \
        | utils/int2sym.pl \
            -f 2- \
            exp/tri1/graph/words.txt \
        > exp/tri1/decode_train_diag/hyp/hyp.$j.txt
done

cat exp/tri1/decode_train_diag/hyp/hyp.*.txt \
    | sort -k1,1 \
    > exp/tri1/decode_train_diag/hyp.txt

echo
echo "=== Training-set WER ==="

compute-wer \
    ark:"sort -k1,1 data/train/text|" \
    ark:"sort -k1,1 exp/tri1/decode_train_diag/hyp.txt|"

echo
echo "=== First 10 training comparisons ==="

paste \
    <(head -10 data/train/text) \
    <(head -10 exp/tri1/decode_train_diag/hyp.txt)

echo
echo "========================================"
echo "STEP 32 COMPLETE"
echo "========================================"

STEP 32: Training-Set Diagnostic

=== Decoding training data ===
steps/decode.sh --nj 4 --cmd run.pl exp/tri1/graph data/train exp/tri1/decode_train_diag
decode.sh: feature type is delta
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri1/graph exp/tri1/decode_train_diag
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 75.79193064354784% of the time at utterance begin.  This may not be optimal.
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 53.202134756504336% of the time at utterance end.  This may not be optimal.
steps/diagnostic/analyze_lats.sh: see stats in exp/tri1/decode_train_diag/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,3,30) and mean=18.2
steps/diagnostic/analyze_lats.sh: see stats in exp/tri1/decode_train_diag/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.

=== Checking decoded lattices ===
-rw-r--r-- 1 thant_syn t

compute-wer 'ark:sort -k1,1 data/train/text|' 'ark:sort -k1,1 exp/tri1/decode_train_diag/hyp.txt|' 


%WER 165.93 [ 11605 / 6994, 9867 ins, 23 del, 1715 sub ]
%SER 80.49 [ 2414 / 2999 ]
Scored 2999 sentences, 0 not present in hyp.

=== First 10 training comparisons ===
MyintThuSoe_Rec1_20260907_191704 ၀	MyintThuSoe_Rec1_20260907_191704 ၀ 
MyintThuSoe_Rec1_20260907_191712 ၁	MyintThuSoe_Rec1_20260907_191712 ၁ 
MyintThuSoe_Rec1_20260907_191719 ၂	MyintThuSoe_Rec1_20260907_191719 ငွေက ၂ 
MyintThuSoe_Rec1_20260907_191726 ၃	MyintThuSoe_Rec1_20260907_191726 ငွေက ၃ ပါ 
MyintThuSoe_Rec1_20260907_191733 ၄	MyintThuSoe_Rec1_20260907_191733 ၄ ပါ 
MyintThuSoe_Rec1_20260907_191741 ၅	MyintThuSoe_Rec1_20260907_191741 ငွေက ၅ 
MyintThuSoe_Rec1_20260907_191835 ၆	MyintThuSoe_Rec1_20260907_191835 ၆ 
MyintThuSoe_Rec1_20260907_191844 ၇	MyintThuSoe_Rec1_20260907_191844 ၇ 
MyintThuSoe_Rec1_20260907_191851 ၈	MyintThuSoe_Rec1_20260907_191851 ၈ 
MyintThuSoe_Rec1_20260907_191858 ၉	MyintThuSoe_Rec1_20260907_191858 ငွေက ၉ ပါ 

STEP 32 COMPLETE


| Dataset      |         WER |        SER | Main issue                |
| ------------ | ----------: | ---------: | ------------------------- |
| **Training** | **165.93%** | **80.49%** | Very high insertion       |
| **Test**     | **373.06%** | **98.93%** | Even worse generalization |


The important comparison is:

 - Training WER: 165.93%
 - Test WER: 373.06%
 - Test WER is about 2.25× higher.
 - Training hypotheses are already wrong on many utterances, even though the model was trained on that data.

```
For example:

REF: ၂
HYP: ငွေက ၂

REF: ၃
HYP: ငွေက ၃ ပါ

REF: ၉
HYP: ငွေက ၉ ပါ
```

This is a strong clue that the problem is not simply overfitting/generalization.

**One important observation**

Your training WER of 165.93% means the current system is not yet a good ASR model, but it does not mean the entire project failed.

Actually, this gives you a good experimental story for your report:

Baseline GMM-HMM + character-based lexicon + unigram language model produced high WER, with insertion errors being the dominant error type. A stronger n-gram language model was therefore investigated as the next improvement.

---

### Build a Bigram LM

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 33: Build Bigram Language Model"
echo "========================================"

mkdir -p data/local/lm

echo
echo "=== 1. Building bigram counts from training text ==="

python3 - <<'PY'
from collections import Counter

text_file = "data/train/text"
out_file = "data/local/lm/word.2gram.arpa"

unigrams = Counter()
bigrams = Counter()
sentences = 0

with open(text_file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()

        if not parts:
            continue

        # Remove utterance ID
        words = parts[1:]

        if not words:
            continue

        sentences += 1

        # Sentence boundaries
        seq = ["<s>"] + words + ["</s>"]

        for w in seq:
            unigrams[w] += 1

        for i in range(len(seq) - 1):
            bigrams[(seq[i], seq[i+1])] += 1

print(f"Sentences: {sentences}")
print(f"Unique unigrams: {len(unigrams)}")
print(f"Unique bigrams: {len(bigrams)}")

# Create ARPA
with open(out_file, "w", encoding="utf-8") as f:

    f.write("\\data\\\n")
    f.write(f"ngram 1={len(unigrams)}\n")
    f.write(f"ngram 2={len(bigrams)}\n")
    f.write("\n")

    # -------------------------------------------------
    # Unigrams
    # -------------------------------------------------
    f.write("\\1-grams:\n")

    total = sum(unigrams.values())

    for word, count in sorted(unigrams.items()):
        prob = count / total

        # log10 probability
        import math
        logprob = math.log10(prob)

        f.write(f"{logprob:.8f}\t{word}\n")

    f.write("\n")

    # -------------------------------------------------
    # Bigrams
    # -------------------------------------------------
    f.write("\\2-grams:\n")

    for (w1, w2), count in sorted(bigrams.items()):
        prob = count / unigrams[w1]

        import math
        logprob = math.log10(prob)

        f.write(f"{logprob:.8f}\t{w1} {w2}\n")

    f.write("\n")
    f.write("\\end\\\n")

print(f"\nCreated: {out_file}")
PY

echo
echo "=== 2. Inspecting ARPA header ==="

head -20 data/local/lm/word.2gram.arpa

echo
echo "=== 3. Converting ARPA to G.fst ==="

rm -f data/lang/G.fst

arpa2fst \
    --read-symbol-table=data/lang/words.txt \
    data/local/lm/word.2gram.arpa \
    data/lang/G.fst

echo
echo "=== 4. Checking G.fst ==="

fstinfo data/lang/G.fst | grep -E \
    "states|arcs|input label|output label"

echo
echo "========================================"
echo "STEP 33 COMPLETE"
echo "========================================"

STEP 33: Build Bigram Language Model

=== 1. Building bigram counts from training text ===
Sentences: 2999
Unique unigrams: 138
Unique bigrams: 324

Created: data/local/lm/word.2gram.arpa

=== 2. Inspecting ARPA header ===
\data\
ngram 1=138
ngram 2=324

\1-grams:
-0.63669955	</s>
-0.63669955	<s>
-1.81482294	ကို
-1.81264602	ကျပ်
-1.81264602	ခုပါ
-1.81264602	ငွေက
-2.81264602	စက်တင်ဘာ
-2.81264602	ဇန်နဝါရီ
-2.81264602	ဇူလိုင်
-2.81264602	ဇွန်လ
-2.81264602	တစ်ကြိမ်
-2.81264602	ထပ်ပြောပါ
-1.41557547	နံပါတ်
-2.51161602	နံပါတ်ကို
-2.11367601	နှိပ်ပါ

=== 3. Converting ARPA to G.fst ===


arpa2fst --read-symbol-table=data/lang/words.txt data/local/lm/word.2gram.arpa data/lang/G.fst 
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:94) Reading \data\ section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \1-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \2-grams: section.



=== 4. Checking G.fst ===
# of states                                       140
# of arcs                                         599
# of final states                                 1
input label multiplicity                          1
output label multiplicity                         1
# of accessible states                            140
# of coaccessible states                          140
# of connected states                             140
input label sorted                                y
output label sorted                               y

STEP 33 COMPLETE


#### Compose L.fst + Bigram G.fst

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 34: Compose L.fst + Bigram G.fst"
echo "========================================"

echo
echo "=== 1. Checking L.fst ==="

fstinfo data/lang/L.fst | grep -E \
    "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== 2. Checking Bigram G.fst ==="

fstinfo data/lang/G.fst | grep -E \
    "# of states|# of arcs|input label sorted|output label sorted"

echo
echo "=== 3. Composing L.fst + G.fst ==="

rm -f data/lang/LG.fst

fsttablecompose \
    data/lang/L.fst \
    data/lang/G.fst \
    data/lang/LG.fst

echo
echo "=== 4. Sorting LG.fst ==="

fstarcsort \
    --sort_type=olabel \
    data/lang/LG.fst \
    data/lang/LG_sorted.fst

mv data/lang/LG_sorted.fst data/lang/LG.fst

echo
echo "=== 5. Checking new LG.fst ==="

fstinfo data/lang/LG.fst | grep -E \
    "# of states|# of arcs|# of final states|input label sorted|output label sorted"

echo
echo "=== 6. Checking output file ==="

ls -lh data/lang/LG.fst

echo
echo "========================================"
echo "STEP 34 COMPLETE"
echo "========================================"

STEP 34: Compose L.fst + Bigram G.fst

=== 1. Checking L.fst ===
# of states                                       455
# of arcs                                         731
input label sorted                                n
output label sorted                               y

=== 2. Checking Bigram G.fst ===
# of states                                       140
# of arcs                                         599
input label sorted                                y
output label sorted                               y

=== 3. Composing L.fst + G.fst ===


fsttablecompose data/lang/L.fst data/lang/G.fst data/lang/LG.fst 



=== 4. Sorting LG.fst ===

=== 5. Checking new LG.fst ===
# of states                                       0
# of arcs                                         0
# of final states                                 0
input label sorted                                y
output label sorted                               y

=== 6. Checking output file ===
-rw-r--r-- 1 thant_syn thant_syn 66 Sep 10 08:40 data/lang/LG.fst

STEP 34 COMPLETE


In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 34B: Diagnose Empty LG.fst"
echo "========================================"

echo
echo "=== 1. Check word symbols ==="

echo "--- words.txt entries ---"
head -20 data/lang/words.txt

echo
echo "--- Important symbols ---"
grep -E '(<eps>|<UNK>|SIL|<s>|</s>|#0)' data/lang/words.txt

echo
echo "=== 2. Check L.fst text representation ==="

fstdraw --isymbols=data/lang/phones.txt \
        --osymbols=data/lang/words.txt \
        data/lang/L.fst \
        /tmp/L.dot 2>/dev/null || true

fstprint data/lang/L.fst | head -30

echo
echo "=== 3. Check G.fst text representation ==="

fstprint data/lang/G.fst | head -50

echo
echo "=== 4. Check G.fst symbol labels ==="

fstprint data/lang/G.fst | \
    awk '{print $3}' | \
    sort -n | \
    uniq | \
    head -30

echo
echo "=== 5. Check whether G.fst contains word labels ==="

fstprint data/lang/G.fst | \
    awk '{print $3}' | \
    sort -n | \
    uniq | \
    while read id; do
        awk -v id="$id" '$2 == id {print; exit}' data/lang/words.txt
    done | head -50

echo
echo "=== 6. Compose WITHOUT overwriting LG.fst ==="

rm -f /tmp/LG_test.fst

fsttablecompose \
    data/lang/L.fst \
    data/lang/G.fst \
    /tmp/LG_test.fst

echo
echo "=== 7. Composition result ==="

ls -lh /tmp/LG_test.fst

fstinfo /tmp/LG_test.fst | grep -E \
    "# of states|# of arcs|# of final states"

echo
echo "========================================"
echo "STEP 34B COMPLETE"
echo "========================================"

STEP 34B: Diagnose Empty LG.fst

=== 1. Check word symbols ===
--- words.txt entries ---
<eps> 0
<UNK> 1
SIL 2
ကို 3
ကျပ် 4
ခုပါ 5
ငွေက 6
စက်တင်ဘာ 7
ဇန်နဝါရီ 8
ဇူလိုင် 9
ဇွန်လ 10
တစ်ကြိမ် 11
ထပ်ပြောပါ 12
နံပါတ် 13
နံပါတ်ကို 14
နှိပ်ပါ 15
ပါ 16
ပြောပါ 17
ဖုန်းနံပါတ် 18
ဖေဖော်ဝါရီ 19

--- Important symbols ---
<eps> 0
<UNK> 1
SIL 2
#0 139
<s> 140
</s> 141

=== 2. Check L.fst text representation ===
0	1	0	0	0.693147182
0	2	0	0	0.693147182
1	1	201	1	0.693147182
1	2	201	1	0.693147182
1	1	5	2	0.693147182
1	2	5	2	0.693147182
1	3	10	3
1	5	10	4
1	8	14	5
1	11	22	6
1	14	26	7
1	21	30	8
1	28	30	9
1	34	30	10
1	38	38	11
1	45	42	12
1	53	50	13
1	58	50	14
1	66	50	15
1	72	54	16
1	73	54	17
1	78	58	18
1	88	58	19
1	97	58	20
1	106	66	21
1	109	66	22
1	111	74	23
1	117	74	24
1	121	74	25
1	131	74	26

=== 3. Check G.fst text representation ===
3	2	140	140
0	4	3	3	4.17878437
0	5	4	4	4.17377186
0	6	5	5	4.17377186
0	7	6	6	4.17377186
0	8	7	7	6.47635651
0	9	8	8	6.47635651
0	10	9	9	6.47635651
0	11	10	10	6.47635651
0	12

fsttablecompose data/lang/L.fst data/lang/G.fst /tmp/LG_test.fst 



=== 7. Composition result ===
-rw-r--r-- 1 thant_syn thant_syn 66 Sep 10 08:40 /tmp/LG_test.fst
# of states                                       0
# of arcs                                         0
# of final states                                 0

STEP 34B COMPLETE


In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 34C: Correct Bigram LG Construction"
echo "========================================"

echo
echo "=== 1. Check #0 symbol ==="

grep '^#0 ' data/lang/words.txt

echo
echo "=== 2. Check L_disambig.fst ==="

fstinfo data/lang/L_disambig.fst | grep -E \
    "# of states|# of arcs|# of final states|input label sorted|output label sorted"

echo
echo "=== 3. Rebuild G.fst with #0 disambiguation ==="

rm -f data/lang/G.fst

arpa2fst \
    --disambig-symbol=#0 \
    --read-symbol-table=data/lang/words.txt \
    data/local/lm/word.2gram.arpa \
    data/lang/G.fst

echo
echo "=== 4. Check new G.fst ==="

fstinfo data/lang/G.fst | grep -E \
    "# of states|# of arcs|# of final states|input label sorted|output label sorted"

echo
echo "=== 5. Check G.fst stochasticity ==="

fstisstochastic data/lang/G.fst || true

echo
echo "=== 6. Compose L_disambig.fst + G.fst ==="

rm -f data/lang/LG.fst

fsttablecompose \
    data/lang/L_disambig.fst \
    data/lang/G.fst \
    | fstdeterminizestar --use-log=true \
    | fstminimizeencoded \
    | fstpushspecial \
    | fstarcsort --sort-type=ilabel \
    > data/lang/LG.fst

echo
echo "=== 7. Check LG.fst ==="

fstinfo data/lang/LG.fst | grep -E \
    "# of states|# of arcs|# of final states|input label sorted|output label sorted"

echo
echo "=== 8. Check LG.fst file ==="

ls -lh data/lang/LG.fst

echo
echo "========================================"
echo "STEP 34C COMPLETE"
echo "========================================"

STEP 34C: Correct Bigram LG Construction

=== 1. Check #0 symbol ===
#0 139

=== 2. Check L_disambig.fst ===
# of states                                       456
# of arcs                                         733
# of final states                                 1
input label sorted                                n
output label sorted                               y

=== 3. Rebuild G.fst with #0 disambiguation ===


arpa2fst --disambig-symbol=#0 --read-symbol-table=data/lang/words.txt data/local/lm/word.2gram.arpa data/lang/G.fst 
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:94) Reading \data\ section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \1-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \2-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:RemoveRedundantStates():arpa-lm-compiler.cc:359) Reduced num-states from 138 to 138



=== 4. Check new G.fst ===
# of states                                       138
# of arcs                                         520
# of final states                                 78
input label sorted                                y
output label sorted                               n

=== 5. Check G.fst stochasticity ===


fstisstochastic data/lang/G.fst 


0.262449 -0.693147

=== 6. Compose L_disambig.fst + G.fst ===


fsttablecompose data/lang/L_disambig.fst data/lang/G.fst 
fstpushspecial 
FATAL: SetFlags: Bad option: --sort-type=ilabel
fstminimizeencoded 
fstdeterminizestar --use-log=true 



=== 7. Check LG.fst ===


ERROR: FstHeader::Read: Bad FST header: data/lang/LG.fst. Magic number not matched. Got: 0



=== 8. Check LG.fst file ===
-rw-r--r-- 1 thant_syn thant_syn 0 Sep 10 08:41 data/lang/LG.fst

STEP 34C COMPLETE


In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 34D: Check Kaldi Graph Utilities"
echo "========================================"

echo
echo "=== Available graph utilities ==="

ls -l utils/mkgraph.sh
ls -l utils/validate_lang.pl
ls -l utils/fst*
ls -l steps/Makefile

echo
echo "=== Check OpenFST minimize options ==="

fstminimizeencoded --help 2>&1 | head -30

echo
echo "=== Check fstpushspecial options ==="

fstpushspecial --help 2>&1 | head -30

echo
echo "=== Check fstdeterminizestar options ==="

fstdeterminizestar --help 2>&1 | head -30

echo
echo "========================================"
echo "STEP 34D CHECK COMPLETE"
echo "========================================"

STEP 34D: Check Kaldi Graph Utilities

=== Available graph utilities ===
-rwxr-xr-x 1 thant_syn thant_syn 7549 Aug 29 05:52 utils/mkgraph.sh
-rwxr-xr-x 1 thant_syn thant_syn 35147 Aug 29 05:52 utils/validate_lang.pl


ls: cannot access 'utils/fst*': No such file or directory
ls: cannot access 'steps/Makefile': No such file or directory



=== Check OpenFST minimize options ===

Minimizes FST after encoding [similar to fstminimize, but no weight-pushing]

Usage:  fstminimizeencoded [in.fst [out.fst] ]

Options:
  --delta                     : Delta likelihood used for quantization of weights (float, default = 0.000976562)

Standard options:
  --config                    : Configuration file to read (this option may be repeated) (string, default = "")
  --help                      : Print out usage message (bool, default = false)
  --print-args                : Print the command line arguments (to stderr) (bool, default = true)
  --verbose                   : Verbose level (higher->more logging) (int, default = 0)


=== Check fstpushspecial options ===

Pushes weights in an FST such that all the states
in the FST have arcs and final-probs with weights that
sum to the same amount (viewed as being in the log semiring).
Thus, the "extra weight" is distributed throughout the FST.
Tolerance parameter --delta controls how exac

#### Build Bigram LG.fst

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 34E: Build Bigram LG.fst"
echo "========================================"

# Remove the invalid empty LG.fst
rm -f data/lang/LG.fst

echo
echo "=== 1. Compose L_disambig.fst + G.fst ==="

fsttablecompose \
    data/lang/L_disambig.fst \
    data/lang/G.fst \
    /tmp/LG_compose.fst

echo
echo "=== 2. Check composition ==="

fstinfo /tmp/LG_compose.fst | grep -E \
    "# of states|# of arcs|# of final states"

echo
echo "=== 3. Determinize ==="

fstdeterminizestar \
    --use-log=true \
    /tmp/LG_compose.fst \
    /tmp/LG_determinized.fst

echo
echo "=== 4. Minimize ==="

fstminimizeencoded \
    /tmp/LG_determinized.fst \
    /tmp/LG_minimized.fst

echo
echo "=== 5. Push weights ==="

fstpushspecial \
    /tmp/LG_minimized.fst \
    /tmp/LG_pushed.fst

echo
echo "=== 6. Sort arcs by input label ==="

fstarcsort \
    --sort_type=ilabel \
    /tmp/LG_pushed.fst \
    data/lang/LG.fst

echo
echo "=== 7. Check final LG.fst ==="

fstinfo data/lang/LG.fst | grep -E \
    "# of states|# of arcs|# of final states|input label sorted|output label sorted"

echo
echo "=== 8. Check file size ==="

ls -lh data/lang/LG.fst

echo
echo "========================================"
echo "STEP 34E COMPLETE"
echo "========================================"

STEP 34E: Build Bigram LG.fst

=== 1. Compose L_disambig.fst + G.fst ===


fsttablecompose data/lang/L_disambig.fst data/lang/G.fst /tmp/LG_compose.fst 



=== 2. Check composition ===
# of states                                       865
# of arcs                                         1413
# of final states                                 78

=== 3. Determinize ===


fstdeterminizestar --use-log=true /tmp/LG_compose.fst /tmp/LG_determinized.fst 



=== 4. Minimize ===


fstminimizeencoded /tmp/LG_determinized.fst /tmp/LG_minimized.fst 



=== 5. Push weights ===


fstpushspecial /tmp/LG_minimized.fst /tmp/LG_pushed.fst 



=== 6. Sort arcs by input label ===

=== 7. Check final LG.fst ===
# of states                                       440
# of arcs                                         892
# of final states                                 19
input label sorted                                y
output label sorted                               n

=== 8. Check file size ===
-rw-r--r-- 1 thant_syn thant_syn 20K Sep 10 08:42 data/lang/LG.fst

STEP 34E COMPLETE


#### Build Bigram HCLG Graph

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 35: Build Bigram HCLG Graph"
echo "========================================"

echo
echo "=== 1. Check acoustic model ==="

gmm-info exp/tri1/final.mdl | grep -E \
    "number of phones|number of pdfs|number of transition-ids|feature dimension|number of gaussians"

echo
echo "=== 2. Check Bigram LG.fst ==="

fstinfo data/lang/LG.fst | grep -E \
    "# of states|# of arcs|# of final states|input label sorted|output label sorted"

echo
echo "=== 3. Remove old Bigram graph if present ==="

rm -rf exp/tri1/graph_bigram

echo
echo "=== 4. Building Bigram decoding graph ==="

utils/mkgraph.sh \
    data/lang \
    exp/tri1 \
    exp/tri1/graph_bigram

echo
echo "=== 5. Check generated graph ==="

ls -lh exp/tri1/graph_bigram/HCLG.fst
ls -lh exp/tri1/graph_bigram/words.txt
ls -lh exp/tri1/graph_bigram/phones.txt

echo
echo "=== 6. HCLG FST information ==="

fstinfo exp/tri1/graph_bigram/HCLG.fst | grep -E \
    "# of states|# of arcs|# of final states|input label sorted|output label sorted"

echo
echo "========================================"
echo "STEP 35 COMPLETE"
echo "========================================"

STEP 35: Build Bigram HCLG Graph

=== 1. Check acoustic model ===


gmm-info exp/tri1/final.mdl 


number of phones 201
number of pdfs 833
number of transition-ids 6096
feature dimension 39
number of gaussians 10050

=== 2. Check Bigram LG.fst ===
# of states                                       440
# of arcs                                         892
# of final states                                 19
input label sorted                                y
output label sorted                               n

=== 3. Remove old Bigram graph if present ===

=== 4. Building Bigram decoding graph ===


tree-info exp/tri1/tree 
tree-info exp/tri1/tree 
fsttablecompose data/lang/L_disambig.fst data/lang/G.fst 
fstdeterminizestar --use-log=true 
fstminimizeencoded 
fstpushspecial 
fstisstochastic data/lang/tmp/LG.fst 


-0.131121 -0.132035
[info]: LG not stochastic.


...

0 -0.132035
[info]: CLG not stochastic.


make-h-transducer --disambig-syms-out=exp/tri1/graph_bigram/disambig_tid.int --transition-scale=1.0 data/lang/tmp/ilabels_3_1 exp/tri1/tree exp/tri1/final.mdl 
fsttablecompose exp/tri1/graph_bigram/Ha.fst data/lang/tmp/CLG_3_1.fst 
fstdeterminizestar --use-log=true 
fstminimizeencoded 
fstrmsymbols exp/tri1/graph_bigram/disambig_tid.int 
fstrmepslocal 
fstisstochastic exp/tri1/graph_bigram/HCLGa.fst 


0.000474388 -0.37516
HCLGa is not stochastic


add-self-loops --self-loop-scale=0.1 --reorder=true exp/tri1/final.mdl exp/tri1/graph_bigram/HCLGa.fst 



=== 5. Check generated graph ===
-rw-r--r-- 1 thant_syn thant_syn 321K Sep 10 08:42 exp/tri1/graph_bigram/HCLG.fst
-rw-r--r-- 1 thant_syn thant_syn 2.4K Sep 10 08:42 exp/tri1/graph_bigram/words.txt
-rw-r--r-- 1 thant_syn thant_syn 1.9K Sep 10 08:42 exp/tri1/graph_bigram/phones.txt

=== 6. HCLG FST information ===
# of states                                       4738
# of arcs                                         14604
# of final states                                 61
input label sorted                                n
output label sorted                               n

STEP 35 COMPLETE


#### Decode test set with Bigram LM

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 36: Bigram Test Decoding"
echo "========================================"

echo
echo "=== 1. Check test data ==="

echo "Test utterances:"
wc -l data/test/text

echo
echo "Test features:"
wc -l data/test/feats.scp

echo
echo "=== 2. Remove previous Bigram decode ==="

rm -rf exp/tri1/decode_test_bigram

echo
echo "=== 3. Decode test set ==="

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri1/graph_bigram \
    data/test \
    exp/tri1/decode_test_bigram

echo
echo "=== 4. Check lattices ==="

ls -lh exp/tri1/decode_test_bigram/lat.*.gz

echo
echo "=== 5. Check decode logs ==="

grep -h "Overall, lattice depth" \
    exp/tri1/decode_test_bigram/log/*.log \
    2>/dev/null || true

echo
echo "========================================"
echo "STEP 36 COMPLETE"
echo "========================================"

STEP 36: Bigram Test Decoding

=== 1. Check test data ===
Test utterances:
1502 data/test/text

Test features:
1502 data/test/feats.scp

=== 2. Remove previous Bigram decode ===

=== 3. Decode test set ===
steps/decode.sh --nj 4 --cmd run.pl exp/tri1/graph_bigram data/test exp/tri1/decode_test_bigram
decode.sh: feature type is delta
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri1/graph_bigram exp/tri1/decode_test_bigram
steps/diagnostic/analyze_lats.sh: see stats in exp/tri1/decode_test_bigram/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,5,18) and mean=8.4
steps/diagnostic/analyze_lats.sh: see stats in exp/tri1/decode_test_bigram/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.

=== 4. Check lattices ===
-rw-r--r-- 1 thant_syn thant_syn 177K Sep 10 08:44 exp/tri1/decode_test_bigram/lat.1.gz
-rw-r--r-- 1 thant_syn thant_syn 127K Sep 10 08:44 exp/tri1/decode_test_bigram/lat.2.gz


#### Tri1 + Bigram WER

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 37: Bigram WER Evaluation"
echo "========================================"

echo
echo "=== 1. Extract 1-best hypotheses ==="

lattice-best-path \
  "ark:gunzip -c exp/tri1/decode_test_bigram/lat.*.gz|" \
  ark,t:- > exp/tri1/decode_test_bigram/1best.int.txt

echo "1-best hypotheses:"
wc -l exp/tri1/decode_test_bigram/1best.int.txt

echo
echo "=== 2. Convert word IDs to words ==="

utils/int2sym.pl \
  -f 2- \
  exp/tri1/graph_bigram/words.txt \
  exp/tri1/decode_test_bigram/1best.int.txt \
  > exp/tri1/decode_test_bigram/1best.txt

echo "Converted hypotheses:"
wc -l exp/tri1/decode_test_bigram/1best.txt

echo
echo "=== 3. Show first 20 REF vs HYP ==="

paste \
  data/test/text \
  exp/tri1/decode_test_bigram/1best.txt \
  | head -20

echo
echo "=== 4. Calculate WER ==="

python3 - <<'PY'
from pathlib import Path

ref_file = Path("data/test/text")
hyp_file = Path("exp/tri1/decode_test_bigram/1best.txt")

refs = {}
hyps = {}

with ref_file.open(encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if parts:
            refs[parts[0]] = parts[1:]

with hyp_file.open(encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if parts:
            hyps[parts[0]] = parts[1:]

# Levenshtein alignment
def edit_distance(ref, hyp):
    n = len(ref)
    m = len(hyp)

    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            sub = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + sub
            )

    # Recover S/D/I counts
    i, j = n, m
    S = D = I = 0

    while i > 0 or j > 0:
        if i > 0 and j > 0:
            sub = 0 if ref[i-1] == hyp[j-1] else 1
            if dp[i][j] == dp[i-1][j-1] + sub:
                if sub:
                    S += 1
                i -= 1
                j -= 1
                continue

        if i > 0 and dp[i][j] == dp[i-1][j] + 1:
            D += 1
            i -= 1
            continue

        I += 1
        j -= 1

    return S, D, I

total_S = total_D = total_I = 0
total_ref = 0
sentence_errors = 0
missing = 0

for utt, ref in refs.items():
    hyp = hyps.get(utt, [])

    if utt not in hyps:
        missing += 1

    S, D, I = edit_distance(ref, hyp)

    total_S += S
    total_D += D
    total_I += I
    total_ref += len(ref)

    if S + D + I > 0:
        sentence_errors += 1

errors = total_S + total_D + total_I
wer = 100 * errors / total_ref if total_ref else 0
ser = 100 * sentence_errors / len(refs) if refs else 0

print()
print("========================================")
print("BIGRAM TEST RESULTS")
print("========================================")
print(f"Reference words : {total_ref}")
print(f"Substitutions   : {total_S}")
print(f"Deletions       : {total_D}")
print(f"Insertions      : {total_I}")
print(f"Total errors    : {errors}")
print(f"WER             : {wer:.2f}%")
print(f"SER             : {ser:.2f}%")
print(f"Sentences       : {len(refs)}")
print(f"Missing hyps    : {missing}")
print("========================================")
PY

echo
echo "========================================"
echo "STEP 37 COMPLETE"
echo "========================================"

STEP 37: Bigram WER Evaluation

=== 1. Extract 1-best hypotheses ===


lattice-best-path 'ark:gunzip -c exp/tri1/decode_test_bigram/lat.*.gz|' ark,t:- 
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232253, best cost 19.9854 + 22147.9 = 22167.9 over 246 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232313, best cost 21.5825 + 21001.3 = 21022.9 over 235 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232326, best cost 16.7009 + 21078.7 = 21095.4 over 234 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232345, best cost 20.6973 + 18535.4 = 18556.1 over 204 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232406, best cost 20.3749 + 20232.5 = 20252.9 over 223 frames.
LOG (lattice-best-path[5.5.11

1-best hypotheses:
1502 exp/tri1/decode_test_bigram/1best.int.txt

=== 2. Convert word IDs to words ===
Converted hypotheses:
1502 exp/tri1/decode_test_bigram/1best.txt

=== 3. Show first 20 REF vs HYP ===
AungKhantMyat_Rec1_20260907_232253 ၀	AungKhantMyat_Rec1_20260907_232253 ၀ 
AungKhantMyat_Rec1_20260907_232313 ၁	AungKhantMyat_Rec1_20260907_232313 ၁ 
AungKhantMyat_Rec1_20260907_232326 ၂	AungKhantMyat_Rec1_20260907_232326 ၇ 
AungKhantMyat_Rec1_20260907_232345 ၃	AungKhantMyat_Rec1_20260907_232345 ၂၃ 
AungKhantMyat_Rec1_20260907_232406 ၄	AungKhantMyat_Rec1_20260907_232406 ၄ 
AungKhantMyat_Rec1_20260907_232432 ၅	AungKhantMyat_Rec1_20260907_232432 အောက်တိုဘာ ပါ 
AungKhantMyat_Rec1_20260907_232450 ၆	AungKhantMyat_Rec1_20260907_232450 ၆ ငွေက 
AungKhantMyat_Rec1_20260907_232502 ၇	AungKhantMyat_Rec1_20260907_232502 ၇ ၂ 
AungKhantMyat_Rec1_20260907_232539 ၈	AungKhantMyat_Rec1_20260907_232539 ငွေက ၈၈ ငွေက 
AungKhantMyat_Rec1_20260907_232610 ၉	AungKhantMyat_Rec1_20260907_232610 ၉ ငွေက 
AungKhan

the Bigram LM is a major improvement over the Unigram baseline. ✅

| Model              |                 WER |               SER | Substitutions | Deletions | Insertions |
| ------------------ | ------------------: | ----------------: | ------------: | --------: | ---------: |
| Triphone + Unigram |         **373.06%** |            98.93% |         2,530 |        52 | **10,490** |
| Triphone + Bigram  |         **124.46%** |            92.01% |         1,978 |       531 |  **1,852** |
| **Change**         | **↓ 248.60 points** | **↓ 6.92 points** |          ↓552 |      ↑479 | **↓8,638** |


This is a very significant improvement.

The most important result is the insertion error:
```
10,490 → 1,852
```
That's an 82.3% reduction in insertions.

So our earlier hypothesis was correct: the weak Unigram LM was allowing the decoder to generate far too many words.

This tells us the problem is no longer primarily "the LM generates unlimited extra words."

It is now more of a combination of:

1. Acoustic-model confusion
2. Limited training data
3. Character-based pronunciation/lexicon
4. Sparse Bigram LM
5. Possibly speaker/acoustic variation

> The initial Triphone GMM-HMM system with a unigram language model produced a WER of 373.06%, with 10,490 insertion errors. After replacing the unigram language model with a bigram language model, WER decreased substantially to 124.46%, while insertion errors decreased from 10,490 to 1,852. This demonstrates that language-model constraints had a major impact on decoding performance. However, the remaining substitution and deletion errors indicate that further improvement of the acoustic and language modeling components is required.

#### Bigram error analysis

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 38: BIGRAM ERROR ANALYSIS"
echo "========================================"

python3 - <<'PY'
from pathlib import Path
from collections import Counter

ref_file = Path("data/test/text")
hyp_file = Path("exp/tri1/decode_test_bigram/1best.txt")

refs = {}
hyps = {}

with ref_file.open(encoding="utf-8") as f:
    for line in f:
        p = line.strip().split()
        if p:
            refs[p[0]] = p[1:]

with hyp_file.open(encoding="utf-8") as f:
    for line in f:
        p = line.strip().split()
        if p:
            hyps[p[0]] = p[1:]


# --------------------------------------------------
# 1. Word frequencies
# --------------------------------------------------

ref_words = Counter()
hyp_words = Counter()

for utt in refs:
    ref_words.update(refs[utt])
    hyp_words.update(hyps.get(utt, []))


print()
print("=== 1. MOST FREQUENT REFERENCE WORDS ===")

for word, count in ref_words.most_common(20):
    print(f"{count:5d}  {word}")


print()
print("=== 2. MOST FREQUENT HYPOTHESIS WORDS ===")

for word, count in hyp_words.most_common(20):
    print(f"{count:5d}  {word}")


# --------------------------------------------------
# 2. Levenshtein alignment
# --------------------------------------------------

def align(ref, hyp):
    n = len(ref)
    m = len(hyp)

    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i

    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):

            sub = 0 if ref[i-1] == hyp[j-1] else 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + sub
            )

    i, j = n, m

    operations = []

    while i > 0 or j > 0:

        if i > 0 and j > 0:
            sub = 0 if ref[i-1] == hyp[j-1] else 1

            if dp[i][j] == dp[i-1][j-1] + sub:

                if sub == 0:
                    operations.append(
                        ("C", ref[i-1], hyp[j-1])
                    )
                else:
                    operations.append(
                        ("S", ref[i-1], hyp[j-1])
                    )

                i -= 1
                j -= 1
                continue

        if i > 0 and dp[i][j] == dp[i-1][j] + 1:
            operations.append(
                ("D", ref[i-1], "")
            )
            i -= 1
            continue

        operations.append(
            ("I", "", hyp[j-1])
        )
        j -= 1

    operations.reverse()
    return operations


# --------------------------------------------------
# 3. Error counts
# --------------------------------------------------

substitutions = Counter()
insertions = Counter()
deletions = Counter()

correct = 0

for utt in refs:

    ref = refs[utt]
    hyp = hyps.get(utt, [])

    ops = align(ref, hyp)

    for op, r, h in ops:

        if op == "C":
            correct += 1

        elif op == "S":
            substitutions[(r, h)] += 1

        elif op == "I":
            insertions[h] += 1

        elif op == "D":
            deletions[r] += 1


print()
print("=== 3. MOST COMMON SUBSTITUTIONS ===")

for (r, h), count in substitutions.most_common(20):
    print(f"{count:5d}  REF={r}  HYP={h}")


print()
print("=== 4. MOST COMMON INSERTIONS ===")

for word, count in insertions.most_common(20):
    print(f"{count:5d}  {word}")


print()
print("=== 5. MOST COMMON DELETIONS ===")

for word, count in deletions.most_common(20):
    print(f"{count:5d}  {word}")


# --------------------------------------------------
# 4. Length statistics
# --------------------------------------------------

total_ref_words = sum(len(x) for x in refs.values())
total_hyp_words = sum(len(hyps.get(u, [])) for u in refs)

one_word = 0
multi_word = 0

for u in refs:
    if len(refs[u]) == 1:
        one_word += 1
    else:
        multi_word += 1


print()
print("=== 6. LENGTH STATISTICS ===")

print(f"Test sentences       : {len(refs)}")
print(f"Reference words      : {total_ref_words}")
print(f"Hypothesis words     : {total_hyp_words}")
print(f"Average REF length   : {total_ref_words/len(refs):.2f}")
print(f"Average HYP length   : {total_hyp_words/len(refs):.2f}")
print(f"HYP/REF ratio        : {total_hyp_words/total_ref_words:.2f}x")
print(f"1-word utterances    : {one_word}")
print(f"Multi-word utterances: {multi_word}")


# --------------------------------------------------
# 5. Show examples
# --------------------------------------------------

print()
print("=== 7. FIRST 30 ERROR EXAMPLES ===")

shown = 0

for utt in refs:

    ref = refs[utt]
    hyp = hyps.get(utt, [])

    if ref != hyp:

        print()
        print("UTT :", utt)
        print("REF :", " ".join(ref))
        print("HYP :", " ".join(hyp))

        shown += 1

        if shown >= 30:
            break


print()
print("========================================")
print("STEP 38 COMPLETE")
print("========================================")
PY

STEP 38: BIGRAM ERROR ANALYSIS

=== 1. MOST FREQUENT REFERENCE WORDS ===
  480  ပါ
  250  နံပါတ်
  100  ငွေက
  100  ကျပ်
  100  ရက်စွဲက
  100  ၂၀၂၆
  100  ရက်ပါ
  100  အရေအတွက်
  100  ခုပါ
  100  ကို
   99  ဖုန်းနံပါတ်
   81  အော်ဒါနံပါတ်
   50  ရွေးချယ်မှု
   50  နှိပ်ပါ
   50  ရွေးပါ
   49  ၅
   40  ၂
   40  ၇
   30  ၁
   30  ၃

=== 2. MOST FREQUENT HYPOTHESIS WORDS ===
  490  ပါ
  413  ၂
  368  ရက်ပါ
  207  ၇
  176  ၁
  170  ငွေက
  141  ၅
  124  ၆
  118  ၀
  118  ၃
  101  ၂၂
   97  ၉
   85  ကို
   82  ၃၁
   78  ၁၂
   70  ခုပါ
   68  နံပါတ်
   64  ၇၉
   61  ၁၀၀၀၁
   58  ၄

=== 3. MOST COMMON SUBSTITUTIONS ===
   80  REF=ပါ  HYP=ရက်ပါ
   22  REF=ရွေးပါ  HYP=ပါ
   21  REF=ခုပါ  HYP=ပါ
   18  REF=ပါ  HYP=ငွေက
   16  REF=နံပါတ်  HYP=၇
   16  REF=နံပါတ်  HYP=၂၂
   15  REF=ပါ  HYP=ခုပါ
   14  REF=နှိပ်ပါ  HYP=ပါ
   12  REF=ကျပ်  HYP=၇
   12  REF=ကျပ်  HYP=ရက်ပါ
   11  REF=နှိပ်ပါ  HYP=ရက်ပါ
   11  REF=ကျပ်  HYP=၁၂
   11  REF=နံပါတ်  HYP=၂၂၂
   10  REF=နံပါတ်  HYP=အောက်တိုဘာ
   10  REF=နံပါ

Error analysis အရ ပါ ဆိုတဲ့ စကားလုံးတွေက common word တွေဖြစ်နေတာရယ် ပြီးတော့ ရှေ့ကအသံထက်နောက်အသံကို model က ပိုသင်ယူနေသလိုဖြစ်နေသလားလို့ ဥပမာ

၂၃ ဆိုရင် နောက်သံ ၃ ကို တော့ သိနေတယ်။ ၂၉ ကိုလည်း ၇၉ လို့ ခန့်မှန်းတာကလည်း အဲ့လိုကြောင့်ထဲကတစ်ခုလို့ ထင်မိတယ်။ နောက် ၂၇ ကို နံပါတ် ၁၂၇ ဆိုပြီး ခန့်မှန်းတာတို့၊ ‌နောက်ထပ် ၂၂ ကို ၂၁၂တို့ ဘာတို့ပေါ့။

```
REF: ၂
HYP: ၇

REF: ၅
HYP: အောက်တိုဘာ ပါ

REF: ၂၉
HYP: ၇၉
```

---

#### Before moving to a more complicated neural acoustic model, we should perform one controlled experiment:

Build a stronger language model appropriate for this dataset.

Your current Bigram is a raw unsmoothed bigram model. It only knows bigrams that actually occurred in training.

Because your dataset is small, this creates a sparse LM.

A better next step is:

- Bigram with smoothing/backoff

- rather than immediately rebuilding the acoustic model.

Inspect the training vocabulary and bigram structure

Before building the next LM, let's verify exactly what the training data contains.

##  Dataset / LM structure analysis

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 39: DATASET / LM STRUCTURE ANALYSIS"
echo "========================================"

python3 - <<'PY'
from pathlib import Path
from collections import Counter

train_file = Path("data/train/text")
test_file = Path("data/test/text")

def read_data(path):
    data = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            p = line.strip().split()
            if p:
                data.append((p[0], p[1:]))
    return data

train = read_data(train_file)
test = read_data(test_file)

train_words = Counter()
test_words = Counter()

train_bigrams = Counter()

for _, words in train:
    train_words.update(words)

    prev = "<s>"
    for word in words:
        train_bigrams[(prev, word)] += 1
        prev = word

    train_bigrams[(prev, "</s>")] += 1

for _, words in test:
    test_words.update(words)

train_vocab = set(train_words)
test_vocab = set(test_words)

print()
print("=== DATASET ===")

print(f"Training utterances : {len(train)}")
print(f"Test utterances     : {len(test)}")

print()
print("=== VOCABULARY ===")

print(f"Training vocabulary : {len(train_vocab)}")
print(f"Test vocabulary     : {len(test_vocab)}")

unseen = test_vocab - train_vocab

print(f"Unseen test words   : {len(unseen)}")

if unseen:
    print("Unseen words:")
    for word in sorted(unseen):
        print(" ", word)

print()
print("=== TRAINING WORD FREQUENCY ===")

for word, count in train_words.most_common(30):
    print(f"{count:5d}  {word}")

print()
print("=== MOST COMMON BIGRAMS ===")

for (w1, w2), count in train_bigrams.most_common(30):
    print(f"{count:5d}  {w1} -> {w2}")

print()
print("=== ONE-WORD vs MULTI-WORD ===")

one = sum(1 for _, w in train if len(w) == 1)
multi = sum(1 for _, w in train if len(w) > 1)

print(f"One-word training utterances : {one}")
print(f"Multi-word training utterances: {multi}")

print()
print("=== TEST LENGTH DISTRIBUTION ===")

lengths = Counter(len(w) for _, w in test)

for length in sorted(lengths):
    print(f"{length:2d} words : {lengths[length]} utterances")

print()
print("========================================")
print("STEP 39 COMPLETE")
print("========================================")
PY

STEP 39: DATASET / LM STRUCTURE ANALYSIS

=== DATASET ===
Training utterances : 2999
Test utterances     : 1502

=== VOCABULARY ===
Training vocabulary : 136
Test vocabulary     : 136
Unseen test words   : 0

=== TRAINING WORD FREQUENCY ===
  959  ပါ
  499  နံပါတ်
  200  ငွေက
  200  ကျပ်
  200  ရက်စွဲက
  200  ၂၀၂၆
  200  ရက်ပါ
  200  အရေအတွက်
  200  ခုပါ
  199  ဖုန်းနံပါတ်
  199  ကို
  160  အော်ဒါနံပါတ်
  100  ၅
  100  ရွေးချယ်မှု
  100  နှိပ်ပါ
   99  ရွေးပါ
   80  ၂
   80  ၇
   60  ၁
   60  ၃
   60  ၄
   60  ၆
   60  ၉
   60  ၁၂
   59  ၈
   40  ၀
   40  ၁၀
   40  ၁၅
   40  ၁၉
   40  ၂၁

=== MOST COMMON BIGRAMS ===
  959  ပါ -> </s>
  499  <s> -> နံပါတ်
  200  <s> -> ငွေက
  200  ကျပ် -> ပါ
  200  <s> -> ရက်စွဲက
  200  ရက်စွဲက -> ၂၀၂၆
  200  ရက်ပါ -> </s>
  200  <s> -> အရေအတွက်
  200  ခုပါ -> </s>
  199  <s> -> ဖုန်းနံပါတ်
  160  <s> -> အော်ဒါနံပါတ်
  100  <s> -> ရွေးချယ်မှု
  100  ကို -> နှိပ်ပါ
  100  နှိပ်ပါ -> </s>
   99  ကို -> ရွေးပါ
   99  ရွေးပါ -> </s>
   40  နံပါတ် -> ၆
   40

The Bigram LM is helping, but it has a limitation

Look at your most frequent bigrams:
```
<s> → နံပါတ်       499
<s> → ငွေက         200
<s> → ရက်စွဲက      200
<s> → အရေအတွက်     200
<s> → ဖုန်းနံပါတ်   199
<s> → အော်ဒါနံပါတ် 160
```

### Train LDA + MLLT acoustic model

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 40A: CHECK LDA/MLLT REQUIREMENTS"
echo "========================================"

echo
echo "=== Alignment ==="
ls -lh exp/tri1_ali/final.mdl
ls -lh exp/tri1_ali/final.mat 2>/dev/null || true

echo
echo "=== Tri1 model ==="
ls -lh exp/tri1/final.mdl

echo
echo "=== Feature configuration ==="
cat conf/mfcc.conf

echo
echo "=== Required Kaldi scripts ==="

for f in \
    steps/train_lda_mllt.sh \
    steps/align_si.sh \
    steps/make_mfcc.sh \
    steps/compute_cmvn_stats.sh
do
    if [ -x "$f" ] || [ -f "$f" ]; then
        echo "FOUND: $f"
    else
        echo "MISSING: $f"
    fi
done

echo
echo "========================================"
echo "STEP 40A COMPLETE"
echo "========================================"

STEP 40A: CHECK LDA/MLLT REQUIREMENTS

=== Alignment ===
-rw-r--r-- 1 thant_syn thant_syn 3.3M Sep 10 07:53 exp/tri1_ali/final.mdl

=== Tri1 model ===
lrwxrwxrwx 1 thant_syn thant_syn 6 Sep 10 07:53 exp/tri1/final.mdl -> 35.mdl

=== Feature configuration ===
--use-energy=false
--sample-frequency=16000
--num-mel-bins=23
--num-ceps=13
--low-freq=20
--high-freq=-400

=== Required Kaldi scripts ===
FOUND: steps/train_lda_mllt.sh
FOUND: steps/align_si.sh
FOUND: steps/make_mfcc.sh
FOUND: steps/compute_cmvn_stats.sh

STEP 40A COMPLETE


In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 40B: TRAIN LDA + MLLT TRIPHONE"
echo "========================================"

echo
echo "=== 1. Clean previous attempt ==="

rm -rf exp/tri2

echo
echo "=== 2. Train LDA + MLLT model ==="

steps/train_lda_mllt.sh \
    --cmd run.pl \
    --splice-opts "--left-context=3 --right-context=3" \
    2500 15000 \
    data/train \
    data/lang \
    exp/tri1_ali \
    exp/tri2

echo
echo "=== 3. Check final model ==="

ls -lh exp/tri2/final.mdl

echo
echo "=== 4. Model information ==="

gmm-info exp/tri2/final.mdl | head -30

echo
echo "=== 5. Check transformation matrix ==="

ls -lh exp/tri2/final.mat 2>/dev/null || true

echo
echo "========================================"
echo "STEP 40B COMPLETE"
echo "========================================"

STEP 40B: TRAIN LDA + MLLT TRIPHONE

=== 1. Clean previous attempt ===

=== 2. Train LDA + MLLT model ===
steps/train_lda_mllt.sh --cmd run.pl --splice-opts --left-context=3 --right-context=3 2500 15000 data/train data/lang exp/tri1_ali exp/tri2
steps/train_lda_mllt.sh: Accumulating LDA statistics.
steps/train_lda_mllt.sh: Accumulating tree stats
steps/train_lda_mllt.sh: Getting questions for tree clustering.
steps/train_lda_mllt.sh: Building the tree
steps/train_lda_mllt.sh: Initializing the model
WARNING (gmm-init-model[5.5.1182~1-e02e3]:InitAmGmm():gmm-init-model.cc:55) Tree has pdf-id 49 with no stats; corresponding phone list: 198 199 200 201 
This is a bad warning.
steps/train_lda_mllt.sh: Converting alignments from exp/tri1_ali to use current tree
steps/train_lda_mllt.sh: Compiling graphs of transcripts
Training pass 1
Training pass 2
steps/train_lda_mllt.sh: Estimating MLLT
Training pass 3
Training pass 4
steps/train_lda_mllt.sh: Estimating MLLT
Training pass 5
Training pass 6


gmm-info exp/tri2/final.mdl 


number of phones 201
number of pdfs 985
number of transition-ids 7474
number of transition-states 3717
feature dimension 40
number of gaussians 15042

=== 5. Check transformation matrix ===
lrwxrwxrwx 1 thant_syn thant_syn 6 Sep 10 09:03 exp/tri2/final.mat -> 12.mat

STEP 40B COMPLETE


#### Align data with tri2

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 40C: ALIGN TRAINING DATA WITH TRI2"
echo "========================================"

echo
echo "=== 1. Remove previous alignment ==="
rm -rf exp/tri2_ali

echo
echo "=== 2. Align training data ==="

steps/align_si.sh \
    --nj 4 \
    --cmd run.pl \
    data/train \
    data/lang \
    exp/tri2 \
    exp/tri2_ali

echo
echo "=== 3. Check alignment ==="

ls -lh exp/tri2_ali/final.mdl

echo
echo "Number of alignment files:"
ls exp/tri2_ali/ali.*.gz | wc -l

echo
echo "=== 4. Alignment logs ==="

grep -h "Overall, aligned" \
    exp/tri2_ali/log/*.log \
    2>/dev/null || true

echo
echo "========================================"
echo "STEP 40C COMPLETE"
echo "========================================"

STEP 40C: ALIGN TRAINING DATA WITH TRI2

=== 1. Remove previous alignment ===

=== 2. Align training data ===
steps/align_si.sh --nj 4 --cmd run.pl data/train data/lang exp/tri2 exp/tri2_ali
steps/align_si.sh: feature type is lda
steps/align_si.sh: aligning data in data/train using model from exp/tri2, putting alignments in exp/tri2_ali
steps/diagnostic/analyze_alignments.sh --cmd run.pl data/lang exp/tri2_ali
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 74.14080747414081% of the time at utterance end.  This may not be optimal.
steps/diagnostic/analyze_alignments.sh: see stats in exp/tri2_ali/log/analyze_alignments.log
steps/align_si.sh: done aligning data.

=== 3. Check alignment ===
-rw-r--r-- 1 thant_syn thant_syn 4.9M Sep 10 09:04 exp/tri2_ali/final.mdl

Number of alignment files:
4

=== 4. Alignment logs ===

STEP 40C COMPLETE


```
tri1
 ↓
tri1_ali
 ↓
LDA + MLLT training
 ↓
tri2
 ↓
tri2_ali        ← Step 40C
 ↓
Bigram graph
 ↓
test decoding
 ↓
WER
```

#### Build tri2 Bigram Graph

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 40D: BUILD TRI2 BIGRAM GRAPH"
echo "========================================"

echo
echo "=== 1. Remove old tri2 graph ==="
rm -rf exp/tri2/graph_bigram

echo
echo "=== 2. Build HCLG graph ==="

utils/mkgraph.sh \
    data/lang \
    exp/tri2 \
    exp/tri2/graph_bigram

echo
echo "=== 3. Check graph files ==="

ls -lh exp/tri2/graph_bigram/HCLG.fst
ls -lh exp/tri2/graph_bigram/words.txt
ls -lh exp/tri2/graph_bigram/phones.txt

echo
echo "=== 4. Graph information ==="

fstinfo exp/tri2/graph_bigram/HCLG.fst | \
    grep -E "states|arcs|initial state|# of final states"

echo
echo "========================================"
echo "STEP 40D COMPLETE"
echo "========================================"

STEP 40D: BUILD TRI2 BIGRAM GRAPH

=== 1. Remove old tri2 graph ===

=== 2. Build HCLG graph ===


tree-info exp/tri2/tree 
tree-info exp/tri2/tree 
make-h-transducer --disambig-syms-out=exp/tri2/graph_bigram/disambig_tid.int --transition-scale=1.0 data/lang/tmp/ilabels_3_1 exp/tri2/tree exp/tri2/final.mdl 
fstrmsymbols exp/tri2/graph_bigram/disambig_tid.int 
fsttablecompose exp/tri2/graph_bigram/Ha.fst data/lang/tmp/CLG_3_1.fst 
fstminimizeencoded 
fstdeterminizestar --use-log=true 
fstrmepslocal 
fstisstochastic exp/tri2/graph_bigram/HCLGa.fst 


0.000461894 -0.375218
HCLGa is not stochastic


add-self-loops --self-loop-scale=0.1 --reorder=true exp/tri2/final.mdl exp/tri2/graph_bigram/HCLGa.fst 



=== 3. Check graph files ===
-rw-r--r-- 1 thant_syn thant_syn 356K Sep 10 09:05 exp/tri2/graph_bigram/HCLG.fst
-rw-r--r-- 1 thant_syn thant_syn 2.4K Sep 10 09:05 exp/tri2/graph_bigram/words.txt
-rw-r--r-- 1 thant_syn thant_syn 1.9K Sep 10 09:05 exp/tri2/graph_bigram/phones.txt

=== 4. Graph information ===
# of states                                       5180
# of arcs                                         16281
initial state                                     0
# of final states                                 49
# of accessible states                            5180
# of coaccessible states                          5180
# of connected states                             5180
cyclic at initial state                           n

STEP 40D COMPLETE


**HCLG = HMM + Context dependency + Lexicon + Bigram Language Model**

The message:
```
HCLGa is not stochastic
```
is a diagnostic warning, not an error. ```mkgraph.sh``` continued and successfully produced HCLG.fst, so we should proceed.

Also, your tri2 graph is slightly larger than the tri1 Bigram graph:

| Model             | HCLG states |  HCLG arcs |
| ----------------- | ----------: | ---------: |
| tri1 + Bigram     |       4,738 |     14,604 |
| **tri2 + Bigram** |   **5,180** | **16,281** |


In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 40E: TRI2 BIGRAM TEST DECODING"
echo "========================================"

echo
echo "=== 1. Remove previous decoding ==="
rm -rf exp/tri2/decode_test_bigram

echo
echo "=== 2. Decode test data ==="

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri2/graph_bigram \
    data/test \
    exp/tri2/decode_test_bigram

echo
echo "=== 3. Check decoding output ==="

echo "Lattice files:"
ls exp/tri2/decode_test_bigram/lat.*.gz | wc -l

echo
echo "Log files:"
ls exp/tri2/decode_test_bigram/log/decode.*.log | wc -l

echo
echo "=== 4. Lattice depth ==="

grep -h "lattice depth" \
    exp/tri2/decode_test_bigram/log/decode.*.log \
    2>/dev/null || true

echo
echo "========================================"
echo "STEP 40E COMPLETE"
echo "========================================"

STEP 40E: TRI2 BIGRAM TEST DECODING

=== 1. Remove previous decoding ===

=== 2. Decode test data ===
steps/decode.sh --nj 4 --cmd run.pl exp/tri2/graph_bigram data/test exp/tri2/decode_test_bigram
decode.sh: feature type is lda
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri2/graph_bigram exp/tri2/decode_test_bigram
steps/diagnostic/analyze_lats.sh: see stats in exp/tri2/decode_test_bigram/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,4,11) and mean=5.3
steps/diagnostic/analyze_lats.sh: see stats in exp/tri2/decode_test_bigram/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.

=== 3. Check decoding output ===
Lattice files:
4

Log files:
4

=== 4. Lattice depth ===

STEP 40E COMPLETE


#### Tri2 + Bigram WER

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 40F: TRI2 BIGRAM WER EVALUATION"
echo "========================================"

echo
echo "=== 1. Extract 1-best hypotheses ==="

rm -f exp/tri2/decode_test_bigram/tra.1best.txt

for job in 1 2 3 4; do
    lattice-best-path \
        "ark:gunzip -c exp/tri2/decode_test_bigram/lat.${job}.gz|" \
        "ark,t:exp/tri2/decode_test_bigram/tra.${job}.txt"
done

cat exp/tri2/decode_test_bigram/tra.*.txt | \
    sort -k1,1 > exp/tri2/decode_test_bigram/tra.1best.txt

echo
echo "=== 2. Convert word IDs to words ==="

utils/int2sym.pl \
    -f 2- \
    exp/tri2/graph_bigram/words.txt \
    exp/tri2/decode_test_bigram/tra.1best.txt \
    > exp/tri2/decode_test_bigram/hyp.txt

echo
echo "=== 3. Number of hypotheses ==="

wc -l exp/tri2/decode_test_bigram/hyp.txt

echo
echo "=== 4. Show first 20 REF / HYP pairs ==="

python3 - <<'PY'
from pathlib import Path

ref = {}
for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        ref[p[0]] = p[1]

hyp = {}
for line in Path("exp/tri2/decode_test_bigram/hyp.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        hyp[p[0]] = p[1]
    elif len(p) == 1:
        hyp[p[0]] = ""

for i, utt in enumerate(sorted(ref)[:20]):
    print(f"REF: {ref[utt]}")
    print(f"HYP: {hyp.get(utt, '')}")
    print()
PY

echo
echo "=== 5. Calculate WER ==="

python3 - <<'PY'
from pathlib import Path

def edit_distance(ref, hyp):
    n = len(ref)
    m = len(hyp)

    d = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        d[i][0] = i

    for j in range(m+1):
        d[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1

            d[i][j] = min(
                d[i-1][j] + 1,
                d[i][j-1] + 1,
                d[i-1][j-1] + cost
            )

    return d[n][m]

refs = {}
for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        refs[p[0]] = p[1].split()

hyps = {}
for line in Path("exp/tri2/decode_test_bigram/hyp.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)

    if len(p) == 2:
        hyps[p[0]] = p[1].split()
    elif len(p) == 1:
        hyps[p[0]] = []

S = 0
D = 0
I = 0
N = 0
SER = 0
missing = 0

for utt, r in refs.items():
    h = hyps.get(utt)

    if h is None:
        missing += 1
        h = []

    n = len(r)
    m = len(h)

    dp = [[(0,0,0,0)]*(m+1) for _ in range(n+1)]

    for i in range(1, n+1):
        dp[i][0] = (i,0,i,0)

    for j in range(1, m+1):
        dp[0][j] = (j,j,0,0)

    for i in range(1, n+1):
        for j in range(1, m+1):
            candidates = []

            # substitution / match
            if r[i-1] == h[j-1]:
                prev = dp[i-1][j-1]
                candidates.append(prev)
            else:
                prev = dp[i-1][j-1]
                candidates.append(
                    (prev[0]+1, prev[1], prev[2], prev[3]+1)
                )

            # deletion
            prev = dp[i-1][j]
            candidates.append(
                (prev[0]+1, prev[1], prev[2]+1, prev[3])
            )

            # insertion
            prev = dp[i][j-1]
            candidates.append(
                (prev[0]+1, prev[1]+1, prev[2], prev[3])
            )

            dp[i][j] = min(candidates, key=lambda x: x[0])

    errors, ins, dele, sub = dp[n][m]

    S += sub
    D += dele
    I += ins
    N += n

    if errors > 0:
        SER += 1

print(f"Reference words : {N}")
print(f"Substitutions   : {S}")
print(f"Deletions       : {D}")
print(f"Insertions      : {I}")
print(f"Total errors    : {S+D+I}")
print(f"WER             : {(S+D+I)/N*100:.2f}%")
print(f"SER             : {SER/len(refs)*100:.2f}%")
print(f"Sentences       : {len(refs)}")
print(f"Missing hyps    : {missing}")
PY

echo
echo "========================================"
echo "STEP 40F COMPLETE"
echo "========================================"

STEP 40F: TRI2 BIGRAM WER EVALUATION

=== 1. Extract 1-best hypotheses ===


lattice-best-path 'ark:gunzip -c exp/tri2/decode_test_bigram/lat.1.gz|' ark,t:exp/tri2/decode_test_bigram/tra.1.txt 
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232253, best cost 21.3852 + 12674.6 = 12696 over 246 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232313, best cost 22.7228 + 11888.6 = 11911.3 over 235 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232326, best cost 19.2383 + 11964.5 = 11983.7 over 234 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232345, best cost 16.1283 + 10652.3 = 10668.4 over 204 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232406, best cost 12.9948 + 11560.5 = 11573.5 over 223 fra


=== 2. Convert word IDs to words ===

=== 3. Number of hypotheses ===
1502 exp/tri2/decode_test_bigram/hyp.txt

=== 4. Show first 20 REF / HYP pairs ===
REF: ၀
HYP: ၁၀၀၀၁ 

REF: ၁
HYP: 

REF: ၂
HYP: ပါ 

REF: ၃
HYP: ၁၀၀၁ 

REF: ၄
HYP: 

REF: ၅
HYP: ရက်ပါ 

REF: ၆
HYP: ၆ 

REF: ၇
HYP: ၇ 

REF: ၈
HYP: ၆ ၈၈ 

REF: ၉
HYP: ၃ 

REF: နံပါတ် ၀ ပါ
HYP: 

REF: နံပါတ် ၁ ပါ
HYP: ၂၁ ရက်ပါ 

REF: နံပါတ် ၂ ပါ
HYP: ၁၀၀၁ 

REF: နံပါတ် ၃ ပါ
HYP: ရက်ပါ ၁၀၀၁ 

REF: နံပါတ် ၄ ပါ
HYP: ၇ 

REF: နံပါတ် ၅ ပါ
HYP: 

REF: နံပါတ် ၆ ပါ
HYP: ၇ 

REF: နံပါတ် ၇ ပါ
HYP: ၇ ၃ 

REF: နံပါတ် ၈ ပါ
HYP: နံပါတ် ၈ ၁၁ ပါ 

REF: နံပါတ် ၉ ပါ
HYP: နံပါတ် ၇၉ ရက်ပါ 


=== 5. Calculate WER ===
Reference words : 3504
Substitutions   : 1738
Deletions       : 1225
Insertions      : 666
Total errors    : 3629
WER             : 103.57%
SER             : 95.14%
Sentences       : 1502
Missing hyps    : 0

STEP 40F COMPLETE


**This is a meaningful improvement, but it also tells us something important about the acoustic model. ✅**

1. Tri2 + Bigram result

| System                     |         WER |    SER | Insertions |
| -------------------------- | ----------: | -----: | ---------: |
| Tri1 + Unigram             | **373.06%** | 98.93% |     10,490 |
| Tri1 + Bigram              | **124.46%** | 92.01% |      1,852 |
| **Tri2 LDA+MLLT + Bigram** | **103.57%** | 95.14% |    **666** |


So tri2 reduced WER from 124.46% → 103.57%.

That's a reduction of:

- 20.89 absolute WER points
- about 16.8% relative WER reduction
- Insertions dropped from 1,852 → 666, about 64.0% reduction

That's good evidence that LDA+MLLT improved the acoustic modeling.

Improvement တွေတော့ရှိလာတယ်

```
REF: နံပါတ် ၁ ပါ
HYP: ၂၁ ရက်ပါ

REF: ၆
HYP: ၆

REF: ၇
HYP: ၇

REF: နံပါတ် ၈ ပါ
HYP: နံပါတ် ၈ ၁၁ ပါ

REF: နံပါတ် ၉ ပါ
HYP: နံပါတ် ၇၉ ရက်ပါ

REF: နံပါတ် ၇ ပါ
HYP: ၇ ၃
```

> The LDA+MLLT triphone model improved the Bigram-LM decoding performance from 124.46% WER to 103.57% WER. The main improvement was a substantial reduction in insertion errors, from 1,852 to 666, indicating that the LDA+MLLT acoustic transformation produced more constrained and accurate hypotheses. However, deletion errors increased from 531 to 1,225, suggesting that further speaker or acoustic adaptation is needed.


### Train SAT/fMLLR model

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 41A: TRAIN SAT/fMLLR MODEL"
echo "========================================"

echo
echo "=== 1. Check required script ==="

if [ ! -x steps/train_sat.sh ]; then
    echo "ERROR: steps/train_sat.sh not found or not executable"
    exit 1
fi

echo "steps/train_sat.sh: OK"

echo
echo "=== 2. Check tri2 alignment ==="

if [ ! -f exp/tri2_ali/final.mdl ]; then
    echo "ERROR: exp/tri2_ali/final.mdl not found"
    exit 1
fi

echo "tri2 alignment: OK"

echo
echo "=== 3. Remove previous SAT model ==="

rm -rf exp/tri3

echo
echo "=== 4. Train SAT/fMLLR model ==="

steps/train_sat.sh \
    --cmd run.pl \
    2500 \
    15000 \
    data/train \
    data/lang \
    exp/tri2_ali \
    exp/tri3

echo
echo "=== 5. Check trained model ==="

ls -lh exp/tri3/final.mdl

echo
echo "=== 6. Model information ==="

gmm-info exp/tri3/final.mdl | \
    grep -E "number of phones|number of pdfs|feature dimension|number of gaussians"

echo
echo "========================================"
echo "STEP 41A COMPLETE"
echo "========================================"

STEP 41A: TRAIN SAT/fMLLR MODEL

=== 1. Check required script ===
steps/train_sat.sh: OK

=== 2. Check tri2 alignment ===
tri2 alignment: OK

=== 3. Remove previous SAT model ===

=== 4. Train SAT/fMLLR model ===
steps/train_sat.sh --cmd run.pl 2500 15000 data/train data/lang exp/tri2_ali exp/tri3
steps/train_sat.sh: feature type is lda
steps/train_sat.sh: obtaining initial fMLLR transforms since not present in exp/tri2_ali
steps/train_sat.sh: Accumulating tree stats
steps/train_sat.sh: Getting questions for tree clustering.
steps/train_sat.sh: Building the tree
steps/train_sat.sh: Initializing the model
WARNING (gmm-init-model[5.5.1182~1-e02e3]:InitAmGmm():gmm-init-model.cc:55) Tree has pdf-id 49 with no stats; corresponding phone list: 198 199 200 201 
This is a bad warning.
steps/train_sat.sh: Converting alignments from exp/tri2_ali to use current tree
steps/train_sat.sh: Compiling graphs of transcripts
Pass 1
Pass 2
Estimating fMLLR transforms
Pass 3
Pass 4
Estimating fMLLR transfo

gmm-info exp/tri3/final.mdl 


number of phones 201
number of pdfs 1129
feature dimension 40
number of gaussians 15033

STEP 41A COMPLETE


```python
tri1
  ↓
Delta features
  ↓
tri2
  ↓
LDA + MLLT
  ↓
tri3
  ↓
SAT + fMLLR
```

#### Align training data with SAT

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 41B: ALIGN TRAINING DATA WITH SAT"
echo "========================================"

echo
echo "=== 1. Remove previous SAT alignment ==="
rm -rf exp/tri3_ali

echo
echo "=== 2. Align training data ==="

steps/align_si.sh \
    --nj 4 \
    --cmd run.pl \
    data/train \
    data/lang \
    exp/tri3 \
    exp/tri3_ali

echo
echo "=== 3. Check alignment ==="

ls -lh exp/tri3_ali/final.mdl

echo
echo "Number of alignment files:"
ls exp/tri3_ali/ali.*.gz | wc -l

echo
echo "=== 4. Check fMLLR transforms ==="

find exp/tri3_ali -name "*.trans.*" -o -name "trans.*" | head

echo
echo "========================================"
echo "STEP 41B COMPLETE"
echo "========================================"

STEP 41B: ALIGN TRAINING DATA WITH SAT

=== 1. Remove previous SAT alignment ===

=== 2. Align training data ===
steps/align_si.sh --nj 4 --cmd run.pl data/train data/lang exp/tri3 exp/tri3_ali
steps/align_si.sh: feature type is lda
steps/align_si.sh: aligning data in data/train using model from exp/tri3, putting alignments in exp/tri3_ali
steps/diagnostic/analyze_alignments.sh --cmd run.pl data/lang exp/tri3_ali
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 74.60794127460794% of the time at utterance end.  This may not be optimal.
steps/diagnostic/analyze_alignments.sh: see stats in exp/tri3_ali/log/analyze_alignments.log
steps/align_si.sh: done aligning data.

=== 3. Check alignment ===
-rw-r--r-- 1 thant_syn thant_syn 5.0M Sep 10 09:21 exp/tri3_ali/final.mdl

Number of alignment files:
4

=== 4. Check fMLLR transforms ===

STEP 41B COMPLETE


#### Build the SAT + Bigram graph

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 41C: BUILD TRI3 SAT BIGRAM GRAPH"
echo "========================================"

echo
echo "=== 1. Remove old graph ==="
rm -rf exp/tri3/graph_bigram

echo
echo "=== 2. Build HCLG graph ==="

utils/mkgraph.sh \
    data/lang \
    exp/tri3 \
    exp/tri3/graph_bigram

echo
echo "=== 3. Check graph files ==="

ls -lh exp/tri3/graph_bigram/HCLG.fst
ls -lh exp/tri3/graph_bigram/words.txt
ls -lh exp/tri3/graph_bigram/phones.txt

echo
echo "=== 4. Graph information ==="

fstinfo exp/tri3/graph_bigram/HCLG.fst | \
    grep -E "states|arcs|initial state|# of final states"

echo
echo "========================================"
echo "STEP 41C COMPLETE"
echo "========================================"

STEP 41C: BUILD TRI3 SAT BIGRAM GRAPH

=== 1. Remove old graph ===

=== 2. Build HCLG graph ===


tree-info exp/tri3/tree 
tree-info exp/tri3/tree 
make-h-transducer --disambig-syms-out=exp/tri3/graph_bigram/disambig_tid.int --transition-scale=1.0 data/lang/tmp/ilabels_3_1 exp/tri3/tree exp/tri3/final.mdl 
fstrmsymbols exp/tri3/graph_bigram/disambig_tid.int 
fstminimizeencoded 
fstrmepslocal 
fstdeterminizestar --use-log=true 
fsttablecompose exp/tri3/graph_bigram/Ha.fst data/lang/tmp/CLG_3_1.fst 
fstisstochastic exp/tri3/graph_bigram/HCLGa.fst 


0.000461894 -0.375128
HCLGa is not stochastic


add-self-loops --self-loop-scale=0.1 --reorder=true exp/tri3/final.mdl exp/tri3/graph_bigram/HCLGa.fst 



=== 3. Check graph files ===
-rw-r--r-- 1 thant_syn thant_syn 372K Sep 10 09:22 exp/tri3/graph_bigram/HCLG.fst
-rw-r--r-- 1 thant_syn thant_syn 2.4K Sep 10 09:22 exp/tri3/graph_bigram/words.txt
-rw-r--r-- 1 thant_syn thant_syn 1.9K Sep 10 09:22 exp/tri3/graph_bigram/phones.txt

=== 4. Graph information ===
# of states                                       5591
# of arcs                                         16778
initial state                                     0
# of final states                                 50
# of accessible states                            5591
# of coaccessible states                          5591
# of connected states                             5591
cyclic at initial state                           n

STEP 41C COMPLETE


In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 41D: TRI3 SAT BIGRAM TEST DECODING"
echo "========================================"

echo
echo "=== 1. Remove previous decoding ==="
rm -rf exp/tri3/decode_test_bigram

echo
echo "=== 2. Decode test data ==="

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri3/graph_bigram \
    data/test \
    exp/tri3/decode_test_bigram

echo
echo "=== 3. Check decoding output ==="

echo "Lattice files:"
ls exp/tri3/decode_test_bigram/lat.*.gz | wc -l

echo
echo "Log files:"
ls exp/tri3/decode_test_bigram/log/decode.*.log | wc -l

echo
echo "=== 4. Lattice depth ==="

grep -h "Overall, lattice depth" \
    exp/tri3/decode_test_bigram/log/analyze_alignments.log \
    2>/dev/null || true

echo
echo "========================================"
echo "STEP 41D COMPLETE"
echo "========================================"

STEP 41D: TRI3 SAT BIGRAM TEST DECODING

=== 1. Remove previous decoding ===

=== 2. Decode test data ===
steps/decode.sh --nj 4 --cmd run.pl exp/tri3/graph_bigram data/test exp/tri3/decode_test_bigram



steps/decode.sh WARNING: Running speaker independent system decoding using a SAT model!
steps/decode.sh WARNING: This is OK if you know what you are doing...



decode.sh: feature type is lda
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri3/graph_bigram exp/tri3/decode_test_bigram
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_bigram/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,3,10) and mean=4.9
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_bigram/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.

=== 3. Check decoding output ===
Lattice files:
4

Log files:
4

=== 4. Lattice depth ===

STEP 41D COMPLETE


#### Tri3 + SAT Bigram WER

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "========================================"
echo "STEP 41E: TRI3 SAT WER EVALUATION"
echo "========================================"

echo
echo "=== 1. Extract 1-best hypotheses ==="

rm -f exp/tri3/decode_test_bigram/tra.1best.txt

for job in 1 2 3 4; do
    lattice-best-path \
        "ark:gunzip -c exp/tri3/decode_test_bigram/lat.${job}.gz|" \
        "ark,t:exp/tri3/decode_test_bigram/tra.${job}.txt"
done

cat exp/tri3/decode_test_bigram/tra.*.txt | \
    sort -k1,1 > exp/tri3/decode_test_bigram/tra.1best.txt

echo
echo "=== 2. Convert word IDs to words ==="

utils/int2sym.pl \
    -f 2- \
    exp/tri3/graph_bigram/words.txt \
    exp/tri3/decode_test_bigram/tra.1best.txt \
    > exp/tri3/decode_test_bigram/hyp.txt

echo
echo "=== 3. Number of hypotheses ==="

wc -l exp/tri3/decode_test_bigram/hyp.txt

echo
echo "=== 4. Calculate WER ==="

python3 - <<'PY'
from pathlib import Path

def get_data(path):
    data = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        p = line.split(maxsplit=1)
        if len(p) == 2:
            data[p[0]] = p[1].split()
        elif len(p) == 1:
            data[p[0]] = []
    return data

refs = get_data("data/test/text")
hyps = get_data("exp/tri3/decode_test_bigram/hyp.txt")

S = D = I = N = SER = missing = 0

for utt, r in refs.items():
    if utt not in hyps:
        missing += 1
        h = []
    else:
        h = hyps[utt]

    n, m = len(r), len(h)

    # dp[i][j] = (errors, insertions, deletions, substitutions)
    dp = [[(0,0,0,0) for _ in range(m+1)] for _ in range(n+1)]

    for i in range(1, n+1):
        dp[i][0] = (i, 0, i, 0)

    for j in range(1, m+1):
        dp[0][j] = (j, j, 0, 0)

    for i in range(1, n+1):
        for j in range(1, m+1):
            candidates = []

            # Match/substitution
            prev = dp[i-1][j-1]
            if r[i-1] == h[j-1]:
                candidates.append(prev)
            else:
                candidates.append(
                    (prev[0]+1, prev[1], prev[2], prev[3]+1)
                )

            # Deletion
            prev = dp[i-1][j]
            candidates.append(
                (prev[0]+1, prev[1], prev[2]+1, prev[3])
            )

            # Insertion
            prev = dp[i][j-1]
            candidates.append(
                (prev[0]+1, prev[1]+1, prev[2], prev[3])
            )

            dp[i][j] = min(candidates, key=lambda x: x[0])

    errors, ins, dele, sub = dp[n][m]

    I += ins
    D += dele
    S += sub
    N += n

    if errors > 0:
        SER += 1

total_errors = S + D + I

print(f"Reference words : {N}")
print(f"Substitutions   : {S}")
print(f"Deletions       : {D}")
print(f"Insertions      : {I}")
print(f"Total errors    : {total_errors}")
print(f"WER             : {total_errors/N*100:.2f}%")
print(f"SER             : {SER/len(refs)*100:.2f}%")
print(f"Sentences       : {len(refs)}")
print(f"Missing hyps    : {missing}")
PY

echo
echo "========================================"
echo "STEP 41E COMPLETE"
echo "========================================"

STEP 41E: TRI3 SAT WER EVALUATION

=== 1. Extract 1-best hypotheses ===


lattice-best-path 'ark:gunzip -c exp/tri3/decode_test_bigram/lat.1.gz|' ark,t:exp/tri3/decode_test_bigram/tra.1.txt 
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232253, best cost 11.0174 + 12715.9 = 12726.9 over 246 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232313, best cost 20.2969 + 11747.2 = 11767.5 over 235 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232326, best cost 11.0212 + 11643.8 = 11654.8 over 234 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232345, best cost 16.2914 + 10410.5 = 10426.8 over 204 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232406, best cost 16.2461 + 11328.4 = 11344.6 over 223 f


=== 2. Convert word IDs to words ===

=== 3. Number of hypotheses ===
1502 exp/tri3/decode_test_bigram/hyp.txt

=== 4. Calculate WER ===
Reference words : 3504
Substitutions   : 1563
Deletions       : 1339
Insertions      : 494
Total errors    : 3396
WER             : 96.92%
SER             : 95.21%
Sentences       : 1502
Missing hyps    : 0

STEP 41E COMPLETE


Excellent. Step 41E is a successful improvement. 🎉

Your SAT/fMLLR model achieved:

WER = 96.92%

This is now your best result so far.

| System                        |         WER |       Sub |   Del |     Ins |
| ----------------------------- | ----------: | --------: | ----: | ------: |
| Tri1 + Unigram                | **373.06%** |     2,530 |    52 |  10,490 |
| Tri1 + Bigram                 | **124.46%** |     1,978 |   531 |   1,852 |
| Tri2 + LDA/MLLT + Bigram      | **103.57%** |     1,738 | 1,225 |     666 |
| **Tri3 + SAT/fMLLR + Bigram** |  **96.92%** | **1,563** | 1,339 | **494** |

From 103.57% → 96.92%:

- 6.65 absolute WER points
- 6.42% relative WER reduction
- substitutions: 1,738 → 1,563
- insertions: 666 → 494
- deletions increased slightly: 1,225 → 1,339

**The most encouraging part is that both substitutions and insertions decreased.**

#### One important limitation

- Remember the warning during decoding:

- Running speaker independent system decoding using a SAT model!
- This is OK if you know what you are doing.

- So this is SAT-trained, but we're not doing speaker-specific fMLLR adaptation on the test speakers.

- Therefore, don't claim that this is the maximum possible SAT performance.

- For your report, I'd describe it as:

- SAT/fMLLR-trained GMM-HMM acoustic model evaluated in speaker-independent decoding.

#### Tri3 Error Analysis

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41F: TRI3 ERROR ANALYSIS"
echo "=================================================="

HYP=exp/tri3/decode_test_bigram/hyp.txt
REF=data/test/text

echo
echo "1. First 30 REF -> HYP examples"
echo "--------------------------------------------------"

python3 - <<'PY'
from pathlib import Path

ref_file = Path("data/test/text")
hyp_file = Path("exp/tri3/decode_test_bigram/hyp.txt")

refs = {}
hyps = {}

for line in ref_file.read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    refs[p[0]] = p[1] if len(p) > 1 else ""

for line in hyp_file.read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    hyps[p[0]] = p[1] if len(p) > 1 else ""

count = 0

for utt in refs:
    if count >= 30:
        break

    print(f"UTT: {utt}")
    print(f"REF: {refs[utt]}")
    print(f"HYP: {hyps.get(utt, '')}")
    print()

    count += 1
PY

echo
echo "2. Length statistics"
echo "--------------------------------------------------"

python3 - <<'PY'
from pathlib import Path
from collections import Counter

refs = {}
hyps = {}

for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    refs[p[0]] = p[1].split() if len(p) > 1 else []

for line in Path("exp/tri3/decode_test_bigram/hyp.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    hyps[p[0]] = p[1].split() if len(p) > 1 else []

ref_lengths = Counter()
hyp_lengths = Counter()

total_ref = 0
total_hyp = 0

for utt, ref in refs.items():
    hyp = hyps.get(utt, [])

    ref_lengths[len(ref)] += 1
    hyp_lengths[len(hyp)] += 1

    total_ref += len(ref)
    total_hyp += len(hyp)

print("Reference length distribution:")
for n, c in sorted(ref_lengths.items()):
    print(f"  {n} words: {c}")

print()
print("Hypothesis length distribution:")
for n, c in sorted(hyp_lengths.items()):
    print(f"  {n} words: {c}")

print()
print(f"Total reference words   : {total_ref}")
print(f"Total hypothesis words  : {total_hyp}")
print(f"HYP/REF ratio           : {total_hyp/total_ref:.2f}x")
print(f"Average reference length: {total_ref/len(refs):.2f}")
print(f"Average hypothesis length: {total_hyp/len(refs):.2f}")
PY

echo
echo "3. Most frequent hypothesis words"
echo "--------------------------------------------------"

python3 - <<'PY'
from pathlib import Path
from collections import Counter

c = Counter()

for line in Path("exp/tri3/decode_test_bigram/hyp.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) > 1:
        c.update(p[1].split())

for word, count in c.most_common(30):
    print(f"{count:5d}  {word}")
PY

echo
echo "4. Most frequent reference words"
echo "--------------------------------------------------"

python3 - <<'PY'
from pathlib import Path
from collections import Counter

c = Counter()

for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) > 1:
        c.update(p[1].split())

for word, count in c.most_common(30):
    print(f"{count:5d}  {word}")
PY

echo
echo "=================================================="
echo "ERROR ANALYSIS COMPLETE"
echo "=================================================="

STEP 41F: TRI3 ERROR ANALYSIS

1. First 30 REF -> HYP examples
--------------------------------------------------
UTT: AungKhantMyat_Rec1_20260907_232253
REF: ၀
HYP: 

UTT: AungKhantMyat_Rec1_20260907_232313
REF: ၁
HYP: ၂၀၀၁ 

UTT: AungKhantMyat_Rec1_20260907_232326
REF: ၂
HYP: 

UTT: AungKhantMyat_Rec1_20260907_232345
REF: ၃
HYP: ၃ 

UTT: AungKhantMyat_Rec1_20260907_232406
REF: ၄
HYP: ပါ 

UTT: AungKhantMyat_Rec1_20260907_232432
REF: ၅
HYP: ငွေက ရက်ပါ 

UTT: AungKhantMyat_Rec1_20260907_232450
REF: ၆
HYP: ၆ 

UTT: AungKhantMyat_Rec1_20260907_232502
REF: ၇
HYP: ၇ ၂ 

UTT: AungKhantMyat_Rec1_20260907_232539
REF: ၈
HYP: ၃ ၈ 

UTT: AungKhantMyat_Rec1_20260907_232610
REF: ၉
HYP: ၃ 

UTT: AungKhantMyat_Rec1_20260907_232627
REF: နံပါတ် ၀ ပါ
HYP: နံပါတ် ၁၆ ပါ 

UTT: AungKhantMyat_Rec1_20260907_232641
REF: နံပါတ် ၁ ပါ
HYP: နံပါတ် ၁၁ 

UTT: AungKhantMyat_Rec1_20260907_232654
REF: နံပါတ် ၂ ပါ
HYP: နံပါတ် ပါ 

UTT: AungKhantMyat_Rec1_20260907_232744
REF: နံပါတ် ၃ ပါ
HYP: 

UTT: AungKhantMyat_Rec1_

This output is very useful. It tells us something important: the tri3 model is no longer suffering mainly from over-generation.

It has shifted toward deletions + acoustic confusions.

| Metric         | Tri1 + Bigram | Tri2 + Bigram | **Tri3 + Bigram** |
| -------------- | ------------: | ------------: | ----------------: |
| WER            |       124.46% |       103.57% |        **96.92%** |
| Insertions     |         1,852 |           666 |           **494** |
| Deletions      |           531 |         1,225 |         **1,339** |
| Substitutions  |         1,978 |         1,738 |         **1,563** |
| Hyp/Ref length |             — |             — |         **0.76×** |


The key observation is:

> Tri3 is producing too few words.
>
> The reference has 3,504 words, but the hypothesis has only 2,659, and there are 161 completely empty hypotheses.

So the current bottleneck is primarily deletion / weak acoustic recognition, rather than the Bigram LM.

##### Quantify the Error Types

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41G: DETAILED ERROR TYPE ANALYSIS"
echo "=================================================="

python3 - <<'PY'
from pathlib import Path
from collections import Counter

ref_file = Path("data/test/text")
hyp_file = Path("exp/tri3/decode_test_bigram/hyp.txt")

refs = {}
hyps = {}

for line in ref_file.read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    refs[p[0]] = p[1].split() if len(p) > 1 else []

for line in hyp_file.read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    hyps[p[0]] = p[1].split() if len(p) > 1 else []


# --------------------------------------------------
# Levenshtein alignment with backtrace
# --------------------------------------------------

def align(ref, hyp):
    n = len(ref)
    m = len(hyp)

    dp = [[0] * (m + 1) for _ in range(n + 1)]
    bt = [[None] * (m + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        dp[i][0] = i
        bt[i][0] = "D"

    for j in range(1, m + 1):
        dp[0][j] = j
        bt[0][j] = "I"

    for i in range(1, n + 1):
        for j in range(1, m + 1):

            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
                bt[i][j] = "C"
            else:
                sub = dp[i-1][j-1] + 1
                delete = dp[i-1][j] + 1
                insert = dp[i][j-1] + 1

                # deterministic tie-breaking
                best = min(sub, delete, insert)
                dp[i][j] = best

                if best == sub:
                    bt[i][j] = "S"
                elif best == delete:
                    bt[i][j] = "D"
                else:
                    bt[i][j] = "I"

    # Backtrace
    i = n
    j = m
    operations = []

    while i > 0 or j > 0:

        op = bt[i][j]

        if op == "C":
            operations.append(("C", ref[i-1], hyp[j-1]))
            i -= 1
            j -= 1

        elif op == "S":
            operations.append(("S", ref[i-1], hyp[j-1]))
            i -= 1
            j -= 1

        elif op == "D":
            operations.append(("D", ref[i-1], "<del>"))
            i -= 1

        elif op == "I":
            operations.append(("I", "<ins>", hyp[j-1]))
            j -= 1

    return operations[::-1]


subs = Counter()
deletions = Counter()
insertions = Counter()

correct = 0
sub_count = 0
del_count = 0
ins_count = 0

for utt, ref in refs.items():

    hyp = hyps.get(utt, [])

    for op, r, h in align(ref, hyp):

        if op == "C":
            correct += 1

        elif op == "S":
            sub_count += 1
            subs[(r, h)] += 1

        elif op == "D":
            del_count += 1
            deletions[r] += 1

        elif op == "I":
            ins_count += 1
            insertions[h] += 1


print()
print("SUMMARY")
print("--------------------------------------------------")
print(f"Correct      : {correct}")
print(f"Substitutions: {sub_count}")
print(f"Deletions    : {del_count}")
print(f"Insertions   : {ins_count}")
print(f"Total errors : {sub_count + del_count + ins_count}")

print()
print("TOP 30 SUBSTITUTION PAIRS")
print("--------------------------------------------------")

for (r, h), c in subs.most_common(30):
    print(f"{c:4d}  REF={r:<15} -> HYP={h}")

print()
print("TOP 30 DELETED REFERENCE WORDS")
print("--------------------------------------------------")

for word, c in deletions.most_common(30):
    print(f"{c:4d}  {word}")

print()
print("TOP 30 INSERTED HYPOTHESIS WORDS")
print("--------------------------------------------------")

for word, c in insertions.most_common(30):
    print(f"{c:4d}  {word}")

print()
print("==================================================")
print("END OF STEP 41G")
print("==================================================")
PY

STEP 41G: DETAILED ERROR TYPE ANALYSIS

SUMMARY
--------------------------------------------------
Correct      : 602
Substitutions: 1563
Deletions    : 1339
Insertions   : 494
Total errors : 3396

TOP 30 SUBSTITUTION PAIRS
--------------------------------------------------
  69  REF=ပါ              -> HYP=ရက်ပါ
  37  REF=ပါ              -> HYP=၃
  22  REF=ပါ              -> HYP=ခုပါ
  21  REF=ရက်ပါ           -> HYP=ပါ
  17  REF=ခုပါ            -> HYP=ပါ
  15  REF=နှိပ်ပါ         -> HYP=ပါ
  15  REF=ကို             -> HYP=၂
  13  REF=ပါ              -> HYP=၀
  11  REF=ခုပါ            -> HYP=၃
  11  REF=ရွေးပါ          -> HYP=ရက်ပါ
  11  REF=ကျပ်            -> HYP=၂
  10  REF=ရွေးပါ          -> HYP=ပါ
  10  REF=အော်ဒါနံပါတ်    -> HYP=၃
  10  REF=ပါ              -> HYP=၅
   9  REF=အရေအတွက်        -> HYP=ငွေက
   8  REF=နံပါတ်          -> HYP=ငွေက
   8  REF=ပါ              -> HYP=ငွေက
   7  REF=၁၀၀၁            -> HYP=၁၀၀၀၁
   7  REF=ပါ              -> HYP=၁
   7  REF=ကျပ်            -> HYP

- ၃ is strongly over-recognized
- နံပါတ် is heavily deleted
- Some words are confused with semantically/template-related words

```
ပါ     → ရက်ပါ       69
ပါ     → ခုပါ         22
ရက်ပါ → ပါ            21
ရွေးပါ → ရက်ပါ        11
```

```
အရေအတွက် → ငွေက       9
နံပါတ်   → ငွေက       8
ပါ       → ငွေက       8
```

> The Bigram language model substantially reduced insertion errors, while LDA+MLLT and SAT/fMLLR progressively improved the acoustic model. However, the remaining errors are dominated by deletions and substitutions, particularly for frequent template words and Burmese numerical expressions. The frequent recognition of ၃ and deletion of words such as နံပါတ်, ရက်စွဲက, ငွေက, and အရေအတွက် indicate that acoustic discrimination remains the primary bottleneck.

##### Analyze performance by utterance length

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41H: WER BY UTTERANCE LENGTH"
echo "=================================================="

python3 - <<'PY'
from pathlib import Path

refs = {}
hyps = {}

for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    refs[p[0]] = p[1].split() if len(p) > 1 else []

for line in Path("exp/tri3/decode_test_bigram/hyp.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    hyps[p[0]] = p[1].split() if len(p) > 1 else []


def edit_distance(ref, hyp):
    n = len(ref)
    m = len(hyp)

    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i

    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):

            if ref[i-1] == hyp[j-1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[n][m]


groups = {}

for utt, ref in refs.items():

    length = len(ref)
    hyp = hyps.get(utt, [])

    if length not in groups:
        groups[length] = {
            "sentences": 0,
            "words": 0,
            "errors": 0,
            "correct_sentences": 0
        }

    g = groups[length]

    g["sentences"] += 1
    g["words"] += len(ref)

    errors = edit_distance(ref, hyp)
    g["errors"] += errors

    if ref == hyp:
        g["correct_sentences"] += 1


print()
print(f"{'REF LEN':<10} {'SENTENCES':>10} {'REF WORDS':>10} "
      f"{'ERRORS':>10} {'WER':>10} {'SER':>10}")
print("-" * 65)

total_words = 0
total_errors = 0
total_sentences = 0

for length in sorted(groups):

    g = groups[length]

    wer = 100 * g["errors"] / g["words"]
    ser = 100 * (g["sentences"] - g["correct_sentences"]) / g["sentences"]

    print(
        f"{length:<10} "
        f"{g['sentences']:>10} "
        f"{g['words']:>10} "
        f"{g['errors']:>10} "
        f"{wer:>9.2f}% "
        f"{ser:>9.2f}%"
    )

    total_words += g["words"]
    total_errors += g["errors"]
    total_sentences += g["sentences"]

print("-" * 65)

print(
    f"{'TOTAL':<10} "
    f"{total_sentences:>10} "
    f"{total_words:>10} "
    f"{total_errors:>10} "
    f"{100*total_errors/total_words:>9.2f}%"
)

print()
print("Number of correctly recognized complete sentences:")
for length in sorted(groups):
    g = groups[length]
    print(
        f"  {length}-word: "
        f"{g['correct_sentences']} / {g['sentences']}"
    )

print()
print("==================================================")
print("END OF STEP 41H")
print("==================================================")
PY

STEP 41H: WER BY UTTERANCE LENGTH

REF LEN     SENTENCES  REF WORDS     ERRORS        WER        SER
-----------------------------------------------------------------
1                 701        701        890    126.96%     90.87%
3                 501       1503       1369     91.08%     98.60%
4                 200        800        695     86.88%     99.50%
5                 100        500        442     88.40%    100.00%
-----------------------------------------------------------------
TOTAL            1502       3504       3396     96.92%

Number of correctly recognized complete sentences:
  1-word: 64 / 701
  3-word: 7 / 501
  4-word: 1 / 200
  5-word: 0 / 100

END OF STEP 41H


| Reference length |         WER | Correct complete sentences |
| ---------------- | ----------: | -------------------------: |
| 1 word           | **126.96%** |                   64 / 701 |
| 3 words          |  **91.08%** |                    7 / 501 |
| 4 words          |  **86.88%** |                    1 / 200 |
| 5 words          |  **88.40%** |                    0 / 100 |
| **Overall**      |  **96.92%** |             **72 / 1,502** |


The 1-word utterances are especially bad, but the problem is not limited to numbers.

And only 72/1,502 = 4.8% of complete test utterances are recognized perfectly.

So we should not simply build a special number LM. The acoustic model needs deeper investigation.

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41I: SPEAKER / DATA DISTRIBUTION ANALYSIS"
echo "=================================================="

python3 - <<'PY'
from pathlib import Path
from collections import Counter, defaultdict

train_utt2spk = {}
test_utt2spk = {}

# -------------------------------
# Read utt2spk
# -------------------------------

for line in Path("data/train/utt2spk").read_text(encoding="utf-8").splitlines():
    p = line.split()
    if len(p) >= 2:
        train_utt2spk[p[0]] = p[1]

for line in Path("data/test/utt2spk").read_text(encoding="utf-8").splitlines():
    p = line.split()
    if len(p) >= 2:
        test_utt2spk[p[0]] = p[1]


# -------------------------------
# Basic speaker counts
# -------------------------------

train_counts = Counter(train_utt2spk.values())
test_counts = Counter(test_utt2spk.values())

print()
print("TRAINING SPEAKERS")
print("----------------------------------------")

for spk, count in sorted(train_counts.items()):
    print(f"{spk:<30} {count:>5} utterances")

print()
print("TEST SPEAKERS")
print("----------------------------------------")

for spk, count in sorted(test_counts.items()):
    print(f"{spk:<30} {count:>5} utterances")


# -------------------------------
# Word counts by training speaker
# -------------------------------

speaker_words = defaultdict(Counter)

for line in Path("data/train/text").read_text(encoding="utf-8").splitlines():

    p = line.split(maxsplit=1)

    if len(p) < 2:
        continue

    utt = p[0]
    words = p[1].split()

    spk = train_utt2spk.get(utt)

    if spk:
        speaker_words[spk].update(words)


print()
print("TOP WORDS PER TRAINING SPEAKER")
print("----------------------------------------")

for spk in sorted(speaker_words):

    print()
    print(f"Speaker: {spk}")

    for word, count in speaker_words[spk].most_common(10):
        print(f"  {count:>4}  {word}")


# -------------------------------
# Number coverage per speaker
# -------------------------------

numbers = [
    "၀", "၁", "၂", "၃", "၄",
    "၅", "၆", "၇", "၈", "၉",
    "၁၀", "၁၁", "၁၂", "၁၃",
    "၁၄", "၁၅", "၁၆", "၁၇",
    "၁၈", "၁၉", "၂၀"
]

print()
print("NUMBER COVERAGE BY TRAINING SPEAKER")
print("----------------------------------------")

for spk in sorted(speaker_words):

    print()
    print(f"Speaker: {spk}")

    for number in numbers:
        count = speaker_words[spk][number]

        if count > 0:
            print(f"  {number:<6} {count:>4}")


print()
print("==================================================")
print("END OF STEP 41I")
print("==================================================")
PY

STEP 41I: SPEAKER / DATA DISTRIBUTION ANALYSIS

TRAINING SPEAKERS
----------------------------------------
MyintThuSoe_Rec1                 150 utterances
MyintThuSoe_Rec2                 150 utterances
MyintThuSoe_Rec3                 150 utterances
MyintThuSoe_Rec4                 150 utterances
MyintThuSoe_Rec5                 150 utterances
SoeThandarTint_Rec1              150 utterances
SoeThandarTint_Rec2              149 utterances
SoeThandarTint_Rec3              150 utterances
SoeThandarTint_Rec4              150 utterances
SoeThandarTint_Rec5              150 utterances
ThantSinTun                      750 utterances
WaiYanHtetAung                   750 utterances

TEST SPEAKERS
----------------------------------------
AungKhantMyat_Rec1               150 utterances
AungKhantMyat_Rec2               151 utterances
AungKhantMyat_Rec3               150 utterances
AungKhantMyat_Rec4               151 utterances
AungKhantMyat_Rec5               151 utterances
ThidaAye             

**WER by Test Speaker**

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41J: WER BY TEST SPEAKER"
echo "=================================================="

python3 - <<'PY'
from pathlib import Path
from collections import defaultdict

# ----------------------------------------
# Read test speaker mapping
# ----------------------------------------

utt2spk = {}

for line in Path("data/test/utt2spk").read_text(encoding="utf-8").splitlines():
    p = line.split()

    if len(p) >= 2:
        utt2spk[p[0]] = p[1]


# ----------------------------------------
# Read reference and hypothesis
# ----------------------------------------

refs = {}
hyps = {}

for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)

    refs[p[0]] = p[1].split() if len(p) > 1 else []


for line in Path("exp/tri3/decode_test_bigram/hyp.txt").read_text(
    encoding="utf-8"
).splitlines():

    p = line.split(maxsplit=1)

    hyps[p[0]] = p[1].split() if len(p) > 1 else []


# ----------------------------------------
# Levenshtein distance
# ----------------------------------------

def edit_distance(ref, hyp):

    n = len(ref)
    m = len(hyp)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            if ref[i-1] == hyp[j-1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[n][m]


# ----------------------------------------
# Collect statistics
# ----------------------------------------

stats = defaultdict(lambda: {
    "sentences": 0,
    "words": 0,
    "errors": 0,
    "perfect": 0,
    "empty": 0
})


for utt, ref in refs.items():

    spk = utt2spk.get(utt, "UNKNOWN")
    hyp = hyps.get(utt, [])

    stats[spk]["sentences"] += 1
    stats[spk]["words"] += len(ref)
    stats[spk]["errors"] += edit_distance(ref, hyp)

    if ref == hyp:
        stats[spk]["perfect"] += 1

    if len(hyp) == 0:
        stats[spk]["empty"] += 1


# ----------------------------------------
# Print results
# ----------------------------------------

print()
print(
    f"{'SPEAKER':<30}"
    f"{'SENTS':>8}"
    f"{'WORDS':>8}"
    f"{'ERRORS':>9}"
    f"{'WER':>10}"
    f"{'SER':>10}"
    f"{'PERFECT':>10}"
    f"{'EMPTY':>8}"
)

print("-" * 95)

total_words = 0
total_errors = 0
total_sentences = 0
total_perfect = 0
total_empty = 0

for spk in sorted(stats):

    s = stats[spk]

    wer = 100 * s["errors"] / s["words"]

    ser = 100 * (
        s["sentences"] - s["perfect"]
    ) / s["sentences"]

    print(
        f"{spk:<30}"
        f"{s['sentences']:>8}"
        f"{s['words']:>8}"
        f"{s['errors']:>9}"
        f"{wer:>9.2f}%"
        f"{ser:>9.2f}%"
        f"{s['perfect']:>10}"
        f"{s['empty']:>8}"
    )

    total_words += s["words"]
    total_errors += s["errors"]
    total_sentences += s["sentences"]
    total_perfect += s["perfect"]
    total_empty += s["empty"]


print("-" * 95)

print(
    f"{'TOTAL':<30}"
    f"{total_sentences:>8}"
    f"{total_words:>8}"
    f"{total_errors:>9}"
    f"{100*total_errors/total_words:>9.2f}%"
    f"{100*(total_sentences-total_perfect)/total_sentences:>9.2f}%"
    f"{total_perfect:>10}"
    f"{total_empty:>8}"
)

print()
print("==================================================")
print("INTERPRETATION")
print("==================================================")

for spk in sorted(stats):

    s = stats[spk]

    wer = 100 * s["errors"] / s["words"]

    print(f"{spk}: WER={wer:.2f}%, empty={s['empty']}, perfect={s['perfect']}")

print()
print("==================================================")
print("END OF STEP 41J")
print("==================================================")
PY

STEP 41J: WER BY TEST SPEAKER

SPEAKER                          SENTS   WORDS   ERRORS       WER       SER   PERFECT   EMPTY
-----------------------------------------------------------------------------------------------
AungKhantMyat_Rec1                 150     348      354   101.72%    97.33%         4      15
AungKhantMyat_Rec2                 151     353      339    96.03%    97.35%         4      33
AungKhantMyat_Rec3                 150     350      329    94.00%    96.00%         6      25
AungKhantMyat_Rec4                 151     353      330    93.48%    92.72%        11      29
AungKhantMyat_Rec5                 151     351      340    96.87%    97.35%         4      56
ThidaAye                           749    1749     1704    97.43%    94.26%        43       3
-----------------------------------------------------------------------------------------------
TOTAL                             1502    3504     3396    96.92%    95.21%        72     161

INTERPRETATION
AungKhant

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41K: CHECK SAT ADAPTATION SUPPORT"
echo "=================================================="

echo
echo "1. Decode script:"
ls -l steps/decode_fmllr.sh 2>/dev/null || echo "decode_fmllr.sh NOT FOUND"

echo
echo "2. Speaker adaptation utilities:"
which align_fmllr.sh 2>/dev/null || true
which ali-to-post 2>/dev/null || true
which gmm-est-fmllr 2>/dev/null || true
which gmm-est-fmllr-gpost 2>/dev/null || true

echo
echo "3. SAT model:"
ls -lh exp/tri3/final.mdl
ls -lh exp/tri3/final.mat 2>/dev/null || true

echo
echo "4. Existing tri3 alignment:"
ls -lh exp/tri3_ali/final.mdl
ls exp/tri3_ali/ali.*.gz | wc -l

echo
echo "5. Test speaker mapping:"
head -20 data/test/utt2spk

echo
echo "=================================================="
echo "END OF STEP 41K"
echo "=================================================="

STEP 41K: CHECK SAT ADAPTATION SUPPORT

1. Decode script:
-rwxr-xr-x 1 thant_syn thant_syn 10334 Aug 29 05:52 steps/decode_fmllr.sh

2. Speaker adaptation utilities:
/home/thant_syn/kaldi/egs/burmese_asr/steps/align_fmllr.sh
/home/thant_syn/kaldi/src/bin/ali-to-post
/home/thant_syn/kaldi/src/gmmbin/gmm-est-fmllr
/home/thant_syn/kaldi/src/gmmbin/gmm-est-fmllr-gpost

3. SAT model:
lrwxrwxrwx 1 thant_syn thant_syn 6 Sep 10 09:20 exp/tri3/final.mdl -> 35.mdl
-rw-r--r-- 1 thant_syn thant_syn 15K Sep 10 09:16 exp/tri3/final.mat

4. Existing tri3 alignment:
-rw-r--r-- 1 thant_syn thant_syn 5.0M Sep 10 09:21 exp/tri3_ali/final.mdl
4

5. Test speaker mapping:
AungKhantMyat_Rec1_20260907_232253 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232313 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232326 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232345 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232406 AungKhantMyat_Rec1
AungKhantMyat_Rec1_20260907_232432 AungKhantMyat_Rec1
AungKhantMyat_Rec

##### Speaker-adapted SAT decoding

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41L: SAT + fMLLR SPEAKER-ADAPTED DECODING"
echo "=================================================="

# Remove any incomplete previous attempt
rm -rf exp/tri3/decode_test_bigram_fmllr

# Show command usage first
echo
echo "Checking decode_fmllr.sh..."
steps/decode_fmllr.sh --help 2>&1 | head -40

echo
echo "=================================================="
echo "Starting speaker-adapted decoding..."
echo "=================================================="

steps/decode_fmllr.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri3/graph_bigram \
    data/test \
    exp/tri3/decode_test_bigram_fmllr

echo
echo "=================================================="
echo "STEP 41L FINISHED"
echo "=================================================="

echo
echo "Decode directory:"
ls -lh exp/tri3/decode_test_bigram_fmllr | head -30

echo
echo "Number of lattices:"
ls exp/tri3/decode_test_bigram_fmllr/lat.*.gz 2>/dev/null | wc -l

echo
echo "Speaker transforms:"
find exp/tri3/decode_test_bigram_fmllr \
    -type f \
    \( -name "*.trans" -o -name "*.trans.gz" -o -name "trans.*" \) \
    -print

STEP 41L: SAT + fMLLR SPEAKER-ADAPTED DECODING

Checking decode_fmllr.sh...
steps/decode_fmllr.sh --help
No help found.

Starting speaker-adapted decoding...
steps/decode_fmllr.sh --nj 4 --cmd run.pl exp/tri3/graph_bigram data/test exp/tri3/decode_test_bigram_fmllr
steps/decode.sh --scoring-opts  --num-threads 1 --skip-scoring false --acwt 0.083333 --nj 4 --cmd run.pl --beam 10.0 --model exp/tri3/final.alimdl --max-active 2000 exp/tri3/graph_bigram data/test exp/tri3/decode_test_bigram_fmllr.si
decode.sh: feature type is lda
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri3/graph_bigram exp/tri3/decode_test_bigram_fmllr.si
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_bigram_fmllr.si/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,3,8) and mean=4.1
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_bigram_fmllr.si/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist 

#### Tri3 Training-set Diagnostic

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 41M: TRI3 TRAINING-SET DIAGNOSTIC"
echo "=================================================="

rm -rf exp/tri3/decode_train_bigram

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri3/graph_bigram \
    data/train \
    exp/tri3/decode_train_bigram

echo
echo "=================================================="
echo "EXTRACTING TRAINING HYPOTHESES"
echo "=================================================="

lattice-best-path \
    --word-symbol-table=data/lang/words.txt \
    "ark:gunzip -c exp/tri3/decode_train_bigram/lat.*.gz|" \
    ark,t:- \
    2>/dev/null > /tmp/tri3_train_hyp.txt

utils/int2sym.pl \
    -f 2- \
    data/lang/words.txt \
    /tmp/tri3_train_hyp.txt \
    > /tmp/tri3_train_hyp_words.txt

echo
echo "=================================================="
echo "CALCULATING TRAINING WER"
echo "=================================================="

python3 - <<'PY'
from pathlib import Path

ref = {}
for line in Path("data/train/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        ref[p[0]] = p[1].split()

hyp = {}
for line in Path("/tmp/tri3_train_hyp_words.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        hyp[p[0]] = p[1].split()
    elif len(p) == 1:
        hyp[p[0]] = []

def wer_counts(r, h):
    n = len(r)
    m = len(h)

    d = [[0]*(m+1) for _ in range(n+1)]
    op = [[None]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        d[i][0] = i
        if i:
            op[i][0] = "D"

    for j in range(m+1):
        d[0][j] = j
        if j:
            op[0][j] = "I"

    for i in range(1, n+1):
        for j in range(1, m+1):
            if r[i-1] == h[j-1]:
                d[i][j] = d[i-1][j-1]
                op[i][j] = "C"
            else:
                choices = [
                    (d[i-1][j-1] + 1, "S"),
                    (d[i-1][j] + 1, "D"),
                    (d[i][j-1] + 1, "I")
                ]
                d[i][j], op[i][j] = min(choices)

    i, j = n, m
    S = D = I = 0

    while i > 0 or j > 0:
        x = op[i][j]
        if x == "C" or x == "S":
            if x == "S":
                S += 1
            i -= 1
            j -= 1
        elif x == "D":
            D += 1
            i -= 1
        elif x == "I":
            I += 1
            j -= 1

    return S, D, I

S = D = I = 0
total_words = 0
sentences = 0
perfect = 0

for utt, r in ref.items():
    h = hyp.get(utt, [])
    s, d, ins = wer_counts(r, h)
    S += s
    D += d
    I += ins
    total_words += len(r)
    sentences += 1

    if s == d == ins == 0:
        perfect += 1

errors = S + D + I
wer = 100 * errors / total_words
ser = 100 * (sentences - perfect) / sentences

print(f"Training sentences : {sentences}")
print(f"Reference words     : {total_words}")
print(f"Correct             : {total_words - errors}")
print(f"Substitutions       : {S}")
print(f"Deletions           : {D}")
print(f"Insertions          : {I}")
print(f"Total errors        : {errors}")
print(f"WER                 : {wer:.2f}%")
print(f"SER                 : {ser:.2f}%")
print(f"Perfect sentences   : {perfect}")
print(f"Missing hypotheses  : {len(set(ref)-set(hyp))}")

print("\nFirst 20 examples:")
for utt in list(ref)[:20]:
    print("REF:", " ".join(ref[utt]))
    print("HYP:", " ".join(hyp.get(utt, [])))
    print()
PY

STEP 41M: TRI3 TRAINING-SET DIAGNOSTIC
steps/decode.sh --nj 4 --cmd run.pl exp/tri3/graph_bigram data/train exp/tri3/decode_train_bigram



steps/decode.sh WARNING: Running speaker independent system decoding using a SAT model!
steps/decode.sh WARNING: This is OK if you know what you are doing...



decode.sh: feature type is lda
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri3/graph_bigram exp/tri3/decode_train_bigram
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 75.25841947315772% of the time at utterance begin.  This may not be optimal.
analyze_phone_length_stats.py: WARNING: optional-silence SIL is seen only 62.71695594125501% of the time at utterance end.  This may not be optimal.
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_train_bigram/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,1,3) and mean=1.6
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_train_bigram/log/analyze_lattice_depth_

**WER**               : 45.54%

**SER**               : 57.89%

### Build Trigram LM

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 42A: BUILD TRIGRAM LANGUAGE MODEL"
echo "=================================================="

mkdir -p data/local/lm

python3 - <<'PY'
from collections import Counter
from pathlib import Path
import math

sentences = []

for line in Path("data/train/text").read_text(encoding="utf-8").splitlines():
    parts = line.split(maxsplit=1)

    if len(parts) == 2:
        words = parts[1].split()
        sentences.append(["<s>"] + words + ["</s>"])

unigrams = Counter()
bigrams = Counter()
trigrams = Counter()

for sent in sentences:
    for w in sent:
        unigrams[w] += 1

    for i in range(len(sent) - 1):
        bigrams[(sent[i], sent[i+1])] += 1

    for i in range(len(sent) - 2):
        trigrams[(sent[i], sent[i+1], sent[i+2])] += 1

total_tokens = sum(unigrams.values())

print(f"Sentences       : {len(sentences)}")
print(f"Tokens          : {total_tokens}")
print(f"Unigrams        : {len(unigrams)}")
print(f"Bigrams         : {len(bigrams)}")
print(f"Trigrams        : {len(trigrams)}")

# Write ARPA
out = Path("data/local/lm/word.3gram.arpa")

with out.open("w", encoding="utf-8") as f:

    f.write("\\data\\\n")
    f.write(f"ngram 1={len(unigrams)}\n")
    f.write(f"ngram 2={len(bigrams)}\n")
    f.write(f"ngram 3={len(trigrams)}\n\n")

    # Unigrams
    f.write("\\1-grams:\n")

    for w, count in sorted(unigrams.items()):
        prob = count / total_tokens
        log10prob = math.log10(prob)

        f.write(f"{log10prob:.6f}\t{w}\n")

    f.write("\n")

    # Bigrams
    f.write("\\2-grams:\n")

    for (w1, w2), count in sorted(bigrams.items()):
        prob = count / unigrams[w1]
        log10prob = math.log10(prob)

        f.write(f"{log10prob:.6f}\t{w1} {w2}\n")

    f.write("\n")

    # Trigrams
    f.write("\\3-grams:\n")

    for (w1, w2, w3), count in sorted(trigrams.items()):
        prob = count / bigrams[(w1, w2)]
        log10prob = math.log10(prob)

        f.write(f"{log10prob:.6f}\t{w1} {w2} {w3}\n")

    f.write("\n\\end\\\n")

print(f"\nCreated: {out}")
PY

echo
echo "=================================================="
echo "ARPA PREVIEW"
echo "=================================================="

head -35 data/local/lm/word.3gram.arpa

echo
echo "=================================================="
echo "TRIGRAM EXAMPLES"
echo "=================================================="

grep -A15 '^\\3-grams:' data/local/lm/word.3gram.arpa | head -16

STEP 42A: BUILD TRIGRAM LANGUAGE MODEL
Sentences       : 2999
Tokens          : 12992
Unigrams        : 138
Bigrams         : 324
Trigrams        : 319

Created: data/local/lm/word.3gram.arpa

ARPA PREVIEW
\data\
ngram 1=138
ngram 2=324
ngram 3=319

\1-grams:
-0.636700	</s>
-0.636700	<s>
-1.814823	ကို
-1.812646	ကျပ်
-1.812646	ခုပါ
-1.812646	ငွေက
-2.812646	စက်တင်ဘာ
-2.812646	ဇန်နဝါရီ
-2.812646	ဇူလိုင်
-2.812646	ဇွန်လ
-2.812646	တစ်ကြိမ်
-2.812646	ထပ်ပြောပါ
-1.415575	နံပါတ်
-2.511616	နံပါတ်ကို
-2.113676	နှိပ်ပါ
-1.131857	ပါ
-2.812646	ပြောပါ
-1.814823	ဖုန်းနံပါတ်
-2.812646	ဖေဖော်ဝါရီ
-2.812646	ဖြည်းဖြည်း
-2.812646	မတ်လ
-2.812646	မေလ
-1.812646	ရက်စွဲက
-1.812646	ရက်ပါ
-2.113676	ရွေးချယ်မှု
-2.118041	ရွေးပါ
-1.812646	အရေအတွက်
-2.812646	အောက်တိုဘာ
-1.909556	အော်ဒါနံပါတ်

TRIGRAM EXAMPLES
\3-grams:
-1.000000	<s> ငွေက ၁,၀၀၀
-1.000000	<s> ငွေက ၁,၀၀၀,၀၀၀
-1.000000	<s> ငွေက ၁၀,၀၀၀
-1.000000	<s> ငွေက ၁၀၀,၀၀၀
-1.000000	<s> ငွေက ၂,၀၀၀
-1.000000	<s> ငွေက ၂၀,၀၀၀
-1.000000	<s> ငွေက ၅,၀၀၀
-1.000000	<s> ငွ

#### Convert Trigram ARPA TO G.fst

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 42B: CONVERT TRIGRAM ARPA TO G.FST"
echo "=================================================="

# Backup the current Bigram G.fst
cp data/lang/G.fst data/lang/G_bigram_backup.fst

# Build Trigram G.fst
arpa2fst \
    --disambig-symbol=#0 \
    --read-symbol-table=data/lang/words.txt \
    data/local/lm/word.3gram.arpa \
    data/lang/G.fst

echo
echo "=================================================="
echo "TRIGRAM G.FST INFORMATION"
echo "=================================================="

fstinfo data/lang/G.fst | grep -E \
    "num states|num arcs|num final states|input label sorted|output label sorted"

echo
echo "=================================================="
echo "STOCHASTICITY CHECK"
echo "=================================================="

fstisstochastic data/lang/G.fst || true

echo
echo "=================================================="
echo "G.FST CREATED"
echo "=================================================="

ls -lh data/lang/G.fst

STEP 42B: CONVERT TRIGRAM ARPA TO G.FST


arpa2fst --disambig-symbol=#0 --read-symbol-table=data/lang/words.txt data/local/lm/word.3gram.arpa data/lang/G.fst 
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:94) Reading \data\ section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \1-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \2-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:Read():arpa-file-parser.cc:149) Reading \3-grams: section.
LOG (arpa2fst[5.5.1182~1-e02e3]:RemoveRedundantStates():arpa-lm-compiler.cc:359) Reduced num-states from 385 to 385



TRIGRAM G.FST INFORMATION
input label sorted                                y
output label sorted                               n

STOCHASTICITY CHECK


fstisstochastic data/lang/G.fst 


0.262449 -0.693148

G.FST CREATED
-rw-r--r-- 1 thant_syn thant_syn 20K Sep 10 10:13 data/lang/G.fst


#### Trigram LG.fst

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 42C: BUILD TRIGRAM LG.FST"
echo "=================================================="

rm -f /tmp/LG_tri_compose.fst
rm -f /tmp/LG_tri_determinized.fst
rm -f /tmp/LG_tri_minimized.fst
rm -f /tmp/LG_tri_pushed.fst
rm -f data/lang/LG.fst

echo
echo "1. Compose L_disambig.fst + Trigram G.fst..."

fsttablecompose \
    data/lang/L_disambig.fst \
    data/lang/G.fst \
    /tmp/LG_tri_compose.fst

echo "Compose completed."

echo
echo "2. Determinize..."

fstdeterminizestar \
    --use-log=true \
    /tmp/LG_tri_compose.fst \
    /tmp/LG_tri_determinized.fst

echo "Determinization completed."

echo
echo "3. Minimize..."

fstminimizeencoded \
    /tmp/LG_tri_determinized.fst \
    /tmp/LG_tri_minimized.fst

echo "Minimization completed."

echo
echo "4. Push weights..."

fstpushspecial \
    /tmp/LG_tri_minimized.fst \
    /tmp/LG_tri_pushed.fst

echo "Weight pushing completed."

echo
echo "5. Sort by input labels..."

fstarcsort \
    --sort_type=ilabel \
    /tmp/LG_tri_pushed.fst \
    data/lang/LG.fst

echo
echo "=================================================="
echo "TRIGRAM LG.FST INFORMATION"
echo "=================================================="

fstinfo data/lang/LG.fst | grep -E \
    "num states|num arcs|num final states|input label sorted|output label sorted"

echo
echo "=================================================="
echo "FILE SIZE"
echo "=================================================="

ls -lh data/lang/LG.fst

echo
echo "=================================================="
echo "STEP 42C COMPLETE"
echo "=================================================="

STEP 42C: BUILD TRIGRAM LG.FST

1. Compose L_disambig.fst + Trigram G.fst...


fsttablecompose data/lang/L_disambig.fst data/lang/G.fst /tmp/LG_tri_compose.fst 


Compose completed.

2. Determinize...


fstdeterminizestar --use-log=true /tmp/LG_tri_compose.fst /tmp/LG_tri_determinized.fst 


Determinization completed.

3. Minimize...


fstminimizeencoded /tmp/LG_tri_determinized.fst /tmp/LG_tri_minimized.fst 


Minimization completed.

4. Push weights...


fstpushspecial /tmp/LG_tri_minimized.fst /tmp/LG_tri_pushed.fst 


Weight pushing completed.

5. Sort by input labels...

TRIGRAM LG.FST INFORMATION
input label sorted                                y
output label sorted                               n

FILE SIZE
-rw-r--r-- 1 thant_syn thant_syn 35K Sep 10 10:14 data/lang/LG.fst

STEP 42C COMPLETE


#### Trigram HCLG GRAPH

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 42D: BUILD TRIGRAM HCLG GRAPH"
echo "=================================================="

rm -rf exp/tri3/graph_trigram

utils/mkgraph.sh \
    data/lang \
    exp/tri3 \
    exp/tri3/graph_trigram

echo
echo "=================================================="
echo "TRIGRAM HCLG INFORMATION"
echo "=================================================="

fstinfo exp/tri3/graph_trigram/HCLG.fst | grep -E \
    "num states|num arcs|num final states|input label sorted|output label sorted"

echo
echo "=================================================="
echo "GRAPH FILES"
echo "=================================================="

ls -lh exp/tri3/graph_trigram/

echo
echo "=================================================="
echo "STEP 42D COMPLETE"
echo "=================================================="

STEP 42D: BUILD TRIGRAM HCLG GRAPH


tree-info exp/tri3/tree 
tree-info exp/tri3/tree 
fsttablecompose data/lang/L_disambig.fst data/lang/G.fst 
fstminimizeencoded 
fstdeterminizestar --use-log=true 
fstpushspecial 
fstisstochastic data/lang/tmp/LG.fst 


-0.176893 -0.177497
[info]: LG not stochastic.


fstcomposecontext --context-size=3 --central-position=1 --read-disambig-syms=data/lang/phones/disambig.int --write-disambig-syms=data/lang/tmp/disambig_ilabels_3_1.int data/lang/tmp/ilabels_3_1.34946 data/lang/tmp/LG.fst 
fstisstochastic data/lang/tmp/CLG_3_1.fst 


0 -0.177497
[info]: CLG not stochastic.


make-h-transducer --disambig-syms-out=exp/tri3/graph_trigram/disambig_tid.int --transition-scale=1.0 data/lang/tmp/ilabels_3_1 exp/tri3/tree exp/tri3/final.mdl 
fstdeterminizestar --use-log=true 
fsttablecompose exp/tri3/graph_trigram/Ha.fst data/lang/tmp/CLG_3_1.fst 
fstminimizeencoded 
fstrmepslocal 
fstrmsymbols exp/tri3/graph_trigram/disambig_tid.int 
fstisstochastic exp/tri3/graph_trigram/HCLGa.fst 


0.272755 -0.487846
HCLGa is not stochastic


add-self-loops --self-loop-scale=0.1 --reorder=true exp/tri3/final.mdl exp/tri3/graph_trigram/HCLGa.fst 



TRIGRAM HCLG INFORMATION
input label sorted                                n
output label sorted                               n

GRAPH FILES
total 508K
-rw-r--r-- 1 thant_syn thant_syn 487K Sep 10 10:14 HCLG.fst
-rw-r--r-- 1 thant_syn thant_syn   15 Sep 10 10:14 disambig_tid.int
-rw-r--r-- 1 thant_syn thant_syn    5 Sep 10 10:14 num_pdfs
drwxr-xr-x 2 thant_syn thant_syn 4.0K Sep 10 10:14 phones
-rw-r--r-- 1 thant_syn thant_syn 1.9K Sep 10 10:14 phones.txt
-rw-r--r-- 1 thant_syn thant_syn 2.4K Sep 10 10:14 words.txt

STEP 42D COMPLETE


In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 42E: TRI3 + TRIGRAM TEST DECODING"
echo "=================================================="

rm -rf exp/tri3/decode_test_trigram

steps/decode.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri3/graph_trigram \
    data/test \
    exp/tri3/decode_test_trigram

echo
echo "=================================================="
echo "DECODING CHECK"
echo "=================================================="

echo "Lattice files:"
ls exp/tri3/decode_test_trigram/lat.*.gz | wc -l

echo
echo "Log files:"
ls exp/tri3/decode_test_trigram/log/decode.*.log | wc -l

echo
echo "Decode directory:"
ls -lh exp/tri3/decode_test_trigram | head -20

echo
echo "=================================================="
echo "STEP 42E COMPLETE"
echo "=================================================="

STEP 42E: TRI3 + TRIGRAM TEST DECODING
steps/decode.sh --nj 4 --cmd run.pl exp/tri3/graph_trigram data/test exp/tri3/decode_test_trigram



steps/decode.sh WARNING: Running speaker independent system decoding using a SAT model!
steps/decode.sh WARNING: This is OK if you know what you are doing...



decode.sh: feature type is lda
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri3/graph_trigram exp/tri3/decode_test_trigram
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_trigram/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,4,10) and mean=5.0
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_trigram/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.

DECODING CHECK
Lattice files:
4

Log files:
4

Decode directory:
total 420K
-rw-r--r-- 1 thant_syn thant_syn  65K Sep 10 10:15 lat.1.gz
-rw-r--r-- 1 thant_syn thant_syn  51K Sep 10 10:15 lat.2.gz
-rw-r

#### Tri3 + trigram WER

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 42F: TRI3 + TRIGRAM WER"
echo "=================================================="

rm -f /tmp/tri3_trigram_hyp.txt
rm -f /tmp/tri3_trigram_hyp_words.txt

echo
echo "1. Extracting 1-best hypotheses..."

lattice-best-path \
    --word-symbol-table=data/lang/words.txt \
    "ark:gunzip -c exp/tri3/decode_test_trigram/lat.*.gz|" \
    ark,t:- \
    2>/dev/null > /tmp/tri3_trigram_hyp.txt

echo "Hypotheses extracted."

echo
echo "2. Converting word IDs to words..."

utils/int2sym.pl \
    -f 2- \
    data/lang/words.txt \
    /tmp/tri3_trigram_hyp.txt \
    > /tmp/tri3_trigram_hyp_words.txt

echo "Conversion completed."

echo
echo "3. Calculating WER..."

python3 - <<'PY'
from pathlib import Path

ref = {}
for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        ref[p[0]] = p[1].split()

hyp = {}
for line in Path("/tmp/tri3_trigram_hyp_words.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        hyp[p[0]] = p[1].split()
    elif len(p) == 1:
        hyp[p[0]] = []

def wer_counts(r, h):
    n = len(r)
    m = len(h)

    d = [[0]*(m+1) for _ in range(n+1)]
    op = [[None]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        d[i][0] = i
        if i:
            op[i][0] = "D"

    for j in range(m+1):
        d[0][j] = j
        if j:
            op[0][j] = "I"

    for i in range(1, n+1):
        for j in range(1, m+1):
            if r[i-1] == h[j-1]:
                d[i][j] = d[i-1][j-1]
                op[i][j] = "C"
            else:
                choices = [
                    (d[i-1][j-1] + 1, "S"),
                    (d[i-1][j] + 1, "D"),
                    (d[i][j-1] + 1, "I")
                ]
                d[i][j], op[i][j] = min(choices)

    i, j = n, m
    S = D = I = 0

    while i > 0 or j > 0:
        x = op[i][j]

        if x == "C":
            i -= 1
            j -= 1

        elif x == "S":
            S += 1
            i -= 1
            j -= 1

        elif x == "D":
            D += 1
            i -= 1

        elif x == "I":
            I += 1
            j -= 1

    return S, D, I

S = D = I = 0
total_words = 0
sentences = 0
perfect = 0

for utt, r in ref.items():
    h = hyp.get(utt, [])

    s, d, ins = wer_counts(r, h)

    S += s
    D += d
    I += ins
    total_words += len(r)
    sentences += 1

    if s == 0 and d == 0 and ins == 0:
        perfect += 1

errors = S + D + I

wer = 100 * errors / total_words
ser = 100 * (sentences - perfect) / sentences

print()
print("==================================================")
print("TRI3 + TRIGRAM RESULTS")
print("==================================================")
print(f"Reference words : {total_words}")
print(f"Correct         : {total_words - errors}")
print(f"Substitutions   : {S}")
print(f"Deletions       : {D}")
print(f"Insertions      : {I}")
print(f"Total errors    : {errors}")
print(f"WER             : {wer:.2f}%")
print(f"SER             : {ser:.2f}%")
print(f"Sentences       : {sentences}")
print(f"Perfect         : {perfect}")
print(f"Missing hyps    : {len(set(ref) - set(hyp))}")

print()
print("==================================================")
print("FIRST 20 EXAMPLES")
print("==================================================")

for utt in list(ref)[:20]:
    print(f"REF: {' '.join(ref[utt])}")
    print(f"HYP: {' '.join(hyp.get(utt, []))}")
    print()
PY

STEP 42F: TRI3 + TRIGRAM WER

1. Extracting 1-best hypotheses...
Hypotheses extracted.

2. Converting word IDs to words...
Conversion completed.

3. Calculating WER...

TRI3 + TRIGRAM RESULTS
Reference words : 3504
Correct         : 138
Substitutions   : 1554
Deletions       : 1345
Insertions      : 467
Total errors    : 3366
WER             : 96.06%
SER             : 94.87%
Sentences       : 1502
Perfect         : 77
Missing hyps    : 0

FIRST 20 EXAMPLES
REF: ၀
HYP: 

REF: ၁
HYP: ၂၀၀၁

REF: ၂
HYP: 

REF: ၃
HYP: ၃

REF: ၄
HYP: ပါ

REF: ၅
HYP: ငွေက ရက်ပါ

REF: ၆
HYP: ၆

REF: ၇
HYP: ၇ ၂

REF: ၈
HYP: ၃ ၈

REF: ၉
HYP: ၃

REF: နံပါတ် ၀ ပါ
HYP: နံပါတ် ၁၆ ပါ

REF: နံပါတ် ၁ ပါ
HYP: နံပါတ် ၁၁

REF: နံပါတ် ၂ ပါ
HYP: နံပါတ် ပါ

REF: နံပါတ် ၃ ပါ
HYP: 

REF: နံပါတ် ၄ ပါ
HYP: နံပါတ် ပါ

REF: နံပါတ် ၅ ပါ
HYP: ၃

REF: နံပါတ် ၆ ပါ
HYP: ၃ ပါ

REF: နံပါတ် ၇ ပါ
HYP: ရက်ပါ

REF: နံပါတ် ၈ ပါ
HYP: ၃ ပါ

REF: နံပါတ် ၉ ပါ
HYP: ၃



> Increasing the language-model context from Bigram to Trigram produced only a marginal improvement, s
>
> uggesting that the main performance bottleneck is likely the acoustic modeling/data rather than the language model.

#### Trigram Error Analysis

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 43: TRIGRAM ERROR ANALYSIS"
echo "=================================================="

python3 - <<'PY'
from pathlib import Path
from collections import Counter

ref = {}
for line in Path("data/test/text").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        ref[p[0]] = p[1].split()

hyp = {}
for line in Path("/tmp/tri3_trigram_hyp_words.txt").read_text(encoding="utf-8").splitlines():
    p = line.split(maxsplit=1)
    if len(p) == 2:
        hyp[p[0]] = p[1].split()
    elif len(p) == 1:
        hyp[p[0]] = []

def align(r, h):
    n, m = len(r), len(h)

    d = [[0]*(m+1) for _ in range(n+1)]
    op = [[None]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        d[i][0] = i
        if i:
            op[i][0] = "D"

    for j in range(m+1):
        d[0][j] = j
        if j:
            op[0][j] = "I"

    for i in range(1, n+1):
        for j in range(1, m+1):
            if r[i-1] == h[j-1]:
                d[i][j] = d[i-1][j-1]
                op[i][j] = "C"
            else:
                choices = [
                    (d[i-1][j-1]+1, "S"),
                    (d[i-1][j]+1, "D"),
                    (d[i][j-1]+1, "I")
                ]
                d[i][j], op[i][j] = min(choices)

    i, j = n, m
    result = []

    while i > 0 or j > 0:
        x = op[i][j]

        if x == "C":
            result.append(("C", r[i-1], h[j-1]))
            i -= 1
            j -= 1

        elif x == "S":
            result.append(("S", r[i-1], h[j-1]))
            i -= 1
            j -= 1

        elif x == "D":
            result.append(("D", r[i-1], None))
            i -= 1

        elif x == "I":
            result.append(("I", None, h[j-1]))
            j -= 1

    return result[::-1]

subs = Counter()
deletions = Counter()
insertions = Counter()
correct = 0

for utt, r in ref.items():
    h = hyp.get(utt, [])

    for typ, rw, hw in align(r, h):
        if typ == "C":
            correct += 1
        elif typ == "S":
            subs[(rw, hw)] += 1
        elif typ == "D":
            deletions[rw] += 1
        elif typ == "I":
            insertions[hw] += 1

print()
print("Correct:", correct)

print()
print("==================================================")
print("TOP 25 SUBSTITUTIONS")
print("==================================================")

for (r, h), n in subs.most_common(25):
    print(f"{n:4d}  REF={r}  ->  HYP={h}")

print()
print("==================================================")
print("TOP 25 DELETIONS")
print("==================================================")

for word, n in deletions.most_common(25):
    print(f"{n:4d}  {word}")

print()
print("==================================================")
print("TOP 25 INSERTIONS")
print("==================================================")

for word, n in insertions.most_common(25):
    print(f"{n:4d}  {word}")

print()
print("==================================================")
print("TOP 20 MOST FREQUENT HYPOTHESIS WORDS")
print("==================================================")

all_hyp = Counter()
for words in hyp.values():
    all_hyp.update(words)

for word, n in all_hyp.most_common(20):
    print(f"{n:4d}  {word}")
PY

echo
echo "=================================================="
echo "STEP 43 COMPLETE"
echo "=================================================="

STEP 43: TRIGRAM ERROR ANALYSIS

Correct: 605

TOP 25 SUBSTITUTIONS
  38  REF=နံပါတ်  ->  HYP=ရက်ပါ
  35  REF=နံပါတ်  ->  HYP=၃
  23  REF=ငွေက  ->  HYP=၃
  23  REF=အော်ဒါနံပါတ်  ->  HYP=ရက်ပါ
  20  REF=အရေအတွက်  ->  HYP=ငွေက
  18  REF=နံပါတ်  ->  HYP=ငွေက
  18  REF=အော်ဒါနံပါတ်  ->  HYP=၃
  17  REF=အရေအတွက်  ->  HYP=၃
  17  REF=ရက်စွဲက  ->  HYP=ရက်ပါ
  15  REF=ဖုန်းနံပါတ်  ->  HYP=၃
  15  REF=ရက်စွဲက  ->  HYP=၃
  15  REF=ရွေးချယ်မှု  ->  HYP=၂
  15  REF=အရေအတွက်  ->  HYP=၁၂၃၄
  13  REF=ငွေက  ->  HYP=ရက်ပါ
  12  REF=ဖုန်းနံပါတ်  ->  HYP=ရက်ပါ
  12  REF=နံပါတ်  ->  HYP=၂၂
  11  REF=ရက်စွဲက  ->  HYP=၂၈
  10  REF=ပါ  ->  HYP=၂
   8  REF=၃၉၉  ->  HYP=၃
   8  REF=ငွေက  ->  HYP=၂
   8  REF=၂၀၂၆  ->  HYP=ပါ
   8  REF=ရွေးချယ်မှု  ->  HYP=၃
   8  REF=နံပါတ်  ->  HYP=၂၀၀၁
   7  REF=အရေအတွက်  ->  HYP=၁
   7  REF=၂၀၂၆  ->  HYP=၂၁

TOP 25 DELETIONS
 227  ပါ
  78  ကျပ်
  75  ကို
  56  ခုပါ
  46  နှိပ်ပါ
  44  ရက်ပါ
  44  ရွေးပါ
  39  ၂၀၂၆
  35  နံပါတ်
  26  ၅
  22  ၇
  21  ၂
  21  ၁၂
  21  အရေအတွက်


---

**Use your best acoustic model (tri3) and your best-performing Trigram graph (graph_trigram):**

### SAT + fMLLR Trigram Decoding

In [ ]:
%%bash
cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 44: SAT + fMLLR TRIGRAM DECODING"
echo "=================================================="

rm -rf exp/tri3/decode_test_trigram_fmllr

steps/decode_fmllr.sh \
    --nj 4 \
    --cmd run.pl \
    exp/tri3/graph_trigram \
    data/test \
    exp/tri3/decode_test_trigram_fmllr

echo
echo "=================================================="
echo "DECODING CHECK"
echo "=================================================="

echo "Lattice files:"
ls exp/tri3/decode_test_trigram_fmllr/lat.*.gz 2>/dev/null | wc -l

echo
echo "Log files:"
ls exp/tri3/decode_test_trigram_fmllr/log/decode.*.log 2>/dev/null | wc -l

echo
echo "Adaptation files:"
find exp/tri3/decode_test_trigram_fmllr \
    -type f \
    \( -name "*.trans" -o -name "*.trans.gz" -o -name "trans.*" \) \
    -print

echo
echo "Decode directory:"
ls -lh exp/tri3/decode_test_trigram_fmllr | head -30

echo
echo "=================================================="
echo "STEP 44 COMPLETE"
echo "=================================================="

STEP 44: SAT + fMLLR TRIGRAM DECODING
steps/decode_fmllr.sh --nj 4 --cmd run.pl exp/tri3/graph_trigram data/test exp/tri3/decode_test_trigram_fmllr
steps/decode.sh --scoring-opts  --num-threads 1 --skip-scoring false --acwt 0.083333 --nj 4 --cmd run.pl --beam 10.0 --model exp/tri3/final.alimdl --max-active 2000 exp/tri3/graph_trigram data/test exp/tri3/decode_test_trigram_fmllr.si
decode.sh: feature type is lda
steps/diagnostic/analyze_lats.sh --cmd run.pl exp/tri3/graph_trigram exp/tri3/decode_test_trigram_fmllr.si
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_trigram_fmllr.si/log/analyze_alignments.log
Overall, lattice depth (10,50,90-percentile)=(1,3,8) and mean=3.9
steps/diagnostic/analyze_lats.sh: see stats in exp/tri3/decode_test_trigram_fmllr.si/log/analyze_lattice_depth_stats.log
steps/decode.sh: Not scoring because local/score.sh does not exist or not executable.

DECODING CHECK
Lattice files:
0

Log files:
0

Adaptation files:

Decode directory:
total 8.

In [ ]:
%%bash

cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 45: CHECK fMLLR DECODING OUTPUT"
echo "=================================================="

decode_dir=exp/tri3/decode_test_trigram_fmllr.si

echo
echo "Decode directory:"
ls -lah "$decode_dir"

echo
echo "Lattice files:"
find "$decode_dir" -maxdepth 1 -name "lat.*.gz" -print | sort

echo
echo "Number of lattice files:"
find "$decode_dir" -maxdepth 1 -name "lat.*.gz" | wc -l

echo
echo "Log files:"
find "$decode_dir/log" -maxdepth 1 -type f -print | sort

echo
echo "Adaptation-related files:"
find "$decode_dir" -maxdepth 1 -type f \
    \( -name "*trans*" -o -name "*fmllr*" -o -name "*spk*" \) \
    -print | sort

echo
echo "Number of jobs:"
cat "$decode_dir/num_jobs"

echo
echo "=================================================="
echo "STEP 45 COMPLETE"
echo "=================================================="

STEP 45: CHECK fMLLR DECODING OUTPUT

Decode directory:
total 436K
drwxr-xr-x  3 thant_syn thant_syn 4.0K Sep 10 10:21 .
drwxr-xr-x 12 thant_syn thant_syn 4.0K Sep 10 10:20 ..
-rw-r--r--  1 thant_syn thant_syn  76K Sep 10 10:20 lat.1.gz
-rw-r--r--  1 thant_syn thant_syn  57K Sep 10 10:20 lat.2.gz
-rw-r--r--  1 thant_syn thant_syn  22K Sep 10 10:20 lat.3.gz
-rw-r--r--  1 thant_syn thant_syn 258K Sep 10 10:21 lat.4.gz
drwxr-xr-x  2 thant_syn thant_syn 4.0K Sep 10 10:21 log
-rw-r--r--  1 thant_syn thant_syn    2 Sep 10 10:20 num_jobs

Lattice files:
exp/tri3/decode_test_trigram_fmllr.si/lat.1.gz
exp/tri3/decode_test_trigram_fmllr.si/lat.2.gz
exp/tri3/decode_test_trigram_fmllr.si/lat.3.gz
exp/tri3/decode_test_trigram_fmllr.si/lat.4.gz

Number of lattice files:
4

Log files:
exp/tri3/decode_test_trigram_fmllr.si/log/analyze_alignments.log
exp/tri3/decode_test_trigram_fmllr.si/log/analyze_lattice_depth_stats.log
exp/tri3/decode_test_trigram_fmllr.si/log/decode.1.log
exp/tri3/decode_test_trig

#### fMLLR vs Previous systems

In [ ]:
%%bash

cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 47: fMLLR vs PREVIOUS SYSTEMS"
echo "=================================================="

python3 - <<'PY'
systems = [
    ("tri3 + Bigram", 96.92, 1563, 1339, 494, 72),
    ("tri3 + Trigram", 96.06, 1554, 1345, 467, 77),
]

print()
print("Previous results:")
print("-" * 70)
print(f"{'System':<25} {'WER':>8} {'S':>8} {'D':>8} {'I':>8} {'Perfect':>8}")
print("-" * 70)

for name, wer, s, d, i, perfect in systems:
    print(f"{name:<25} {wer:>7.2f}% {s:>8} {d:>8} {i:>8} {perfect:>8}")

print()
print("Now check the Step 46 output above.")
print("Use the fMLLR WER to determine whether speaker adaptation helped.")
PY

echo
echo "=================================================="
echo "STEP 47 COMPLETE"
echo "=================================================="

STEP 47: fMLLR vs PREVIOUS SYSTEMS

Previous results:
----------------------------------------------------------------------
System                         WER        S        D        I  Perfect
----------------------------------------------------------------------
tri3 + Bigram               96.92%     1563     1339      494       72
tri3 + Trigram              96.06%     1554     1345      467       77

Now check the Step 46 output above.
Use the fMLLR WER to determine whether speaker adaptation helped.

STEP 47 COMPLETE


#### fMLLR WER

In [ ]:
%%bash

cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 48: ACTUAL fMLLR WER SCORE"
echo "=================================================="

decode_dir=exp/tri3/decode_test_trigram_fmllr.si

echo
echo "[1/4] Extracting best paths..."

lattice-best-path \
    "ark:gunzip -c $decode_dir/lat.*.gz|" \
    "ark,t:$decode_dir/tra.ark"

echo "Best-path extraction complete."

echo
echo "[2/4] Converting word IDs to words..."

utils/int2sym.pl -f 2- \
    exp/tri3/graph_trigram/words.txt \
    "$decode_dir/tra.ark" \
    > "$decode_dir/hyp.txt"

echo "Hypotheses:"
wc -l "$decode_dir/hyp.txt"

echo
echo "[3/4] Creating reference..."

cut -d' ' -f1- data/test/text > "$decode_dir/ref.txt"

echo "References:"
wc -l "$decode_dir/ref.txt"

echo
echo "[4/4] Calculating WER..."

python3 - "$decode_dir/ref.txt" "$decode_dir/hyp.txt" <<'PY'
import sys

ref_file = sys.argv[1]
hyp_file = sys.argv[2]

refs = {}
hyps = {}

with open(ref_file, encoding="utf-8") as f:
    for line in f:
        p = line.rstrip("\n").split(maxsplit=1)
        if len(p) == 2:
            refs[p[0]] = p[1].split()
        else:
            refs[p[0]] = []

with open(hyp_file, encoding="utf-8") as f:
    for line in f:
        p = line.rstrip("\n").split(maxsplit=1)
        if len(p) == 2:
            hyps[p[0]] = p[1].split()
        else:
            hyps[p[0]] = []

def edit_distance(ref, hyp):
    n = len(ref)
    m = len(hyp)

    dp = [[(0,0,0,0) for _ in range(m+1)]
          for _ in range(n+1)]

    for j in range(1, m+1):
        dp[0][j] = (0,0,0,j)

    for i in range(1, n+1):
        dp[i][0] = (0,0,i,0)

    for i in range(1, n+1):
        for j in range(1, m+1):

            if ref[i-1] == hyp[j-1]:
                c,s,d,ins = dp[i-1][j-1]
                dp[i][j] = (c+1,s,d,ins)
            else:
                c,s,d,ins = dp[i-1][j-1]
                sub = (c,s+1,d,ins)

                c,s,d,ins = dp[i-1][j]
                dele = (c,s,d+1,ins)

                c,s,d,ins = dp[i][j-1]
                insert = (c,s,d,ins+1)

                dp[i][j] = min(
                    [sub, dele, insert],
                    key=lambda x: x[1]+x[2]+x[3]
                )

    return dp[n][m]

C = S = D = I = N = 0
perfect = 0
empty = 0

for utt, ref in refs.items():
    hyp = hyps.get(utt, [])

    if not hyp:
        empty += 1

    c,s,d,ins = edit_distance(ref, hyp)

    C += c
    S += s
    D += d
    I += ins
    N += len(ref)

    if ref == hyp:
        perfect += 1

errors = S + D + I

wer = 100 * errors / N
ser = 100 * (len(refs) - perfect) / len(refs)

print()
print("==================================================")
print("SAT + fMLLR + TRIGRAM RESULTS")
print("==================================================")
print(f"Sentences       : {len(refs)}")
print(f"Reference words : {N}")
print(f"Correct         : {C}")
print(f"Substitutions   : {S}")
print(f"Deletions       : {D}")
print(f"Insertions      : {I}")
print(f"Total errors    : {errors}")
print(f"WER             : {wer:.2f}%")
print(f"SER             : {ser:.2f}%")
print(f"Perfect         : {perfect}")
print(f"Empty hyps      : {empty}")
print("==================================================")
PY

echo
echo "First 20 REF/HYP pairs:"
paste "$decode_dir/ref.txt" "$decode_dir/hyp.txt" | head -20

echo
echo "=================================================="
echo "STEP 48 COMPLETE"
echo "=================================================="

STEP 48: ACTUAL fMLLR WER SCORE

[1/4] Extracting best paths...


lattice-best-path 'ark:gunzip -c exp/tri3/decode_test_trigram_fmllr.si/lat.*.gz|' ark,t:exp/tri3/decode_test_trigram_fmllr.si/tra.ark 
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232253, best cost 23.6009 + 12834.3 = 12857.9 over 246 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232313, best cost 15.0571 + 11955.9 = 11971 over 235 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232326, best cost 16.9673 + 11917.2 = 11934.2 over 234 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232345, best cost 16.3351 + 10737.7 = 10754 over 204 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232406, best cost 11.6299 + 11532.4 = 1154

Best-path extraction complete.

[2/4] Converting word IDs to words...
Hypotheses:
1502 exp/tri3/decode_test_trigram_fmllr.si/hyp.txt

[3/4] Creating reference...
References:
1502 exp/tri3/decode_test_trigram_fmllr.si/ref.txt

[4/4] Calculating WER...

SAT + fMLLR + TRIGRAM RESULTS
Sentences       : 1502
Reference words : 3504
Correct         : 748
Substitutions   : 1800
Deletions       : 956
Insertions      : 902
Total errors    : 3658
WER             : 104.39%
SER             : 92.48%
Perfect         : 113
Empty hyps      : 83

First 20 REF/HYP pairs:
AungKhantMyat_Rec1_20260907_232253 ၀	AungKhantMyat_Rec1_20260907_232253 ၀ 
AungKhantMyat_Rec1_20260907_232313 ၁	AungKhantMyat_Rec1_20260907_232313 ၁ 
AungKhantMyat_Rec1_20260907_232326 ၂	AungKhantMyat_Rec1_20260907_232326 ပါ 
AungKhantMyat_Rec1_20260907_232345 ၃	AungKhantMyat_Rec1_20260907_232345 ၃ 
AungKhantMyat_Rec1_20260907_232406 ၄	AungKhantMyat_Rec1_20260907_232406 
AungKhantMyat_Rec1_20260907_232432 ၅	AungKhantMyat_Rec1_20260907_23

So fMLLR reduced deletions and increased the number of completely correct sentences,

but it caused a large increase in substitutions and insertions, resulting in a worse overall WER.

| System                     |         WER |   Sub |   Del | Ins | Perfect |
| -------------------------- | ----------: | ----: | ----: | --: | ------: |
| tri3 + Bigram              |  **96.92%** | 1,563 | 1,339 | 494 |      72 |
| tri3 + Trigram             |  **96.06%** | 1,554 | 1,345 | 467 |      77 |
| **tri3 + fMLLR + Trigram** | **104.39%** | 1,800 |   956 | 902 |     113 |


#### Detailed Error Analysis

In [ ]:
%%bash

cd /home/thant_syn/kaldi/egs/burmese_asr
source ./path.sh

echo "=================================================="
echo "STEP 49: DETAILED ERROR ANALYSIS"
echo "=================================================="

decode_dir=exp/tri3/decode_test_trigram

# --------------------------------------------------
# 1. Make sure 1-best hypothesis exists
# --------------------------------------------------

if [ ! -f "$decode_dir/hyp.txt" ]; then

    echo "[1] Extracting 1-best hypotheses..."

    lattice-best-path \
        "ark:gunzip -c $decode_dir/lat.*.gz|" \
        "ark,t:$decode_dir/tra.ark"

    utils/int2sym.pl -f 2- \
        exp/tri3/graph_trigram/words.txt \
        "$decode_dir/tra.ark" \
        > "$decode_dir/hyp.txt"
else
    echo "[1] Existing hyp.txt found."
fi

# --------------------------------------------------
# 2. Build reference
# --------------------------------------------------

cut -d' ' -f1- data/test/text > "$decode_dir/ref.txt"

# --------------------------------------------------
# 3. Python error analysis
# --------------------------------------------------

python3 - "$decode_dir/ref.txt" "$decode_dir/hyp.txt" <<'PY'
import sys
from collections import Counter, defaultdict

ref_file = sys.argv[1]
hyp_file = sys.argv[2]

refs = {}
hyps = {}

with open(ref_file, encoding="utf-8") as f:
    for line in f:
        p = line.rstrip("\n").split(maxsplit=1)
        refs[p[0]] = p[1].split() if len(p) == 2 else []

with open(hyp_file, encoding="utf-8") as f:
    for line in f:
        p = line.rstrip("\n").split(maxsplit=1)
        hyps[p[0]] = p[1].split() if len(p) == 2 else []

# --------------------------------------------------
# Alignment
# --------------------------------------------------

def align(ref, hyp):
    n, m = len(ref), len(hyp)

    # dp stores:
    # (total_error, substitutions, deletions, insertions, operations)
    dp = [[None] * (m + 1) for _ in range(n + 1)]

    dp[0][0] = (0, 0, 0, 0, [])

    for j in range(1, m + 1):
        prev = dp[0][j-1]
        dp[0][j] = (
            j,
            0,
            0,
            j,
            prev[4] + [("I", None, hyp[j-1])]
        )

    for i in range(1, n + 1):
        prev = dp[i-1][0]
        dp[i][0] = (
            i,
            0,
            i,
            0,
            prev[4] + [("D", ref[i-1], None)]
        )

    for i in range(1, n + 1):
        for j in range(1, m + 1):

            candidates = []

            # Match
            if ref[i-1] == hyp[j-1]:
                c,s,d,ins,ops = dp[i-1][j-1]
                candidates.append(
                    (c,s,d,ins,ops+[("C",ref[i-1],hyp[j-1])])
                )

            # Substitution
            c,s,d,ins,ops = dp[i-1][j-1]
            candidates.append(
                (c+1,s+1,d,ins,ops+[("S",ref[i-1],hyp[j-1])])
            )

            # Deletion
            c,s,d,ins,ops = dp[i-1][j]
            candidates.append(
                (c+1,s,d+1,ins,ops+[("D",ref[i-1],None)])
            )

            # Insertion
            c,s,d,ins,ops = dp[i][j-1]
            candidates.append(
                (c+1,s,d,ins+1,ops+[("I",None,hyp[j-1])])
            )

            dp[i][j] = min(
                candidates,
                key=lambda x: x[0]
            )

    return dp[n][m]

subs = Counter()
dels = Counter()
ins = Counter()

length_stats = defaultdict(lambda: {
    "sent": 0,
    "words": 0,
    "errors": 0,
    "perfect": 0
})

total_words = 0
total_errors = 0
total_perfect = 0

# --------------------------------------------------
# Process utterances
# --------------------------------------------------

for utt, ref in refs.items():

    hyp = hyps.get(utt, [])

    total_words += len(ref)

    result = align(ref, hyp)
    errors, s, d, i, ops = result

    total_errors += errors

    if ref == hyp:
        total_perfect += 1

    for op, r, h in ops:

        if op == "S":
            subs[(r,h)] += 1

        elif op == "D":
            dels[r] += 1

        elif op == "I":
            ins[h] += 1

    length = len(ref)

    length_stats[length]["sent"] += 1
    length_stats[length]["words"] += len(ref)
    length_stats[length]["errors"] += errors

    if ref == hyp:
        length_stats[length]["perfect"] += 1

# --------------------------------------------------
# Overall
# --------------------------------------------------

print()
print("==================================================")
print("BEST SYSTEM: tri3 + TRIGRAM")
print("==================================================")

print(f"Sentences       : {len(refs)}")
print(f"Reference words : {total_words}")
print(f"Total errors    : {total_errors}")
print(f"WER             : {100*total_errors/total_words:.2f}%")
print(f"Perfect         : {total_perfect}")
print(f"SER             : {100*(len(refs)-total_perfect)/len(refs):.2f}%")

# --------------------------------------------------
# Substitutions
# --------------------------------------------------

print()
print("--------------------------------------------------")
print("TOP 20 SUBSTITUTIONS")
print("--------------------------------------------------")

for (r,h),count in subs.most_common(20):
    print(f"{r} -> {h} : {count}")

# --------------------------------------------------
# Deletions
# --------------------------------------------------

print()
print("--------------------------------------------------")
print("TOP 20 DELETIONS")
print("--------------------------------------------------")

for word,count in dels.most_common(20):
    print(f"{word} : {count}")

# --------------------------------------------------
# Insertions
# --------------------------------------------------

print()
print("--------------------------------------------------")
print("TOP 20 INSERTIONS")
print("--------------------------------------------------")

for word,count in ins.most_common(20):
    print(f"{word} : {count}")

# --------------------------------------------------
# Utterance length
# --------------------------------------------------

print()
print("--------------------------------------------------")
print("WER BY UTTERANCE LENGTH")
print("--------------------------------------------------")

print(
    f"{'Words':>8} "
    f"{'Sentences':>10} "
    f"{'RefWords':>10} "
    f"{'Errors':>10} "
    f"{'WER':>10} "
    f"{'Perfect':>10}"
)

for length in sorted(length_stats):

    x = length_stats[length]

    wer = 100*x["errors"]/x["words"] if x["words"] else 0

    print(
        f"{length:>8} "
        f"{x['sent']:>10} "
        f"{x['words']:>10} "
        f"{x['errors']:>10} "
        f"{wer:>9.2f}% "
        f"{x['perfect']:>10}"
    )

# --------------------------------------------------
# Summary of error types
# --------------------------------------------------

print()
print("--------------------------------------------------")
print("ERROR TYPE SUMMARY")
print("--------------------------------------------------")

print(f"Substitutions : {sum(subs.values())}")
print(f"Deletions     : {sum(dels.values())}")
print(f"Insertions    : {sum(ins.values())}")

print()
print("==================================================")
print("STEP 49 COMPLETE")
print("==================================================")
PY

STEP 49: DETAILED ERROR ANALYSIS
[1] Extracting 1-best hypotheses...


lattice-best-path 'ark:gunzip -c exp/tri3/decode_test_trigram/lat.*.gz|' ark,t:exp/tri3/decode_test_trigram/tra.ark 
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232253, best cost 11.0184 + 12715.9 = 12726.9 over 246 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232313, best cost 19.6016 + 11747.2 = 11766.8 over 235 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232326, best cost 11.0221 + 11643.8 = 11654.8 over 234 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232345, best cost 15.1937 + 10410.5 = 10425.7 over 204 frames.
LOG (lattice-best-path[5.5.1182~1-e02e3]:main():lattice-best-path.cc:99) For utterance AungKhantMyat_Rec1_20260907_232406, best cost 16.2461 + 11328.4 = 11344.6 over 223 f


BEST SYSTEM: tri3 + TRIGRAM
Sentences       : 1502
Reference words : 3504
Total errors    : 3366
WER             : 96.06%
Perfect         : 77
SER             : 94.87%

--------------------------------------------------
TOP 20 SUBSTITUTIONS
--------------------------------------------------
ပါ -> ရက်ပါ : 65
ပါ -> ၃ : 48
ပါ -> ခုပါ : 24
ရက်ပါ -> ပါ : 21
ခုပါ -> ပါ : 17
နှိပ်ပါ -> ပါ : 15
ကို -> ၂ : 14
ပါ -> ၀ : 12
ခုပါ -> ၃ : 11
ရွေးပါ -> ရက်ပါ : 11
ပါ -> ၅ : 10
ကျပ် -> ၂ : 10
ပါ -> ၁ : 9
ရွေးပါ -> ပါ : 9
အရေအတွက် -> ငွေက : 9
နံပါတ် -> ငွေက : 8
၁၀၀၁ -> ၁၀၀၀၁ : 8
ပါ -> ၁၀၀၀၁ : 8
ရက်ပါ -> ၃ : 8
ပါ -> ၂ : 7

--------------------------------------------------
TOP 20 DELETIONS
--------------------------------------------------
နံပါတ် : 168
ရက်စွဲက : 87
ငွေက : 78
အရေအတွက် : 75
၂၀၂၆ : 72
အော်ဒါနံပါတ် : 55
ကို : 53
ကျပ် : 50
ပါ : 49
ရွေးချယ်မှု : 43
ဖုန်းနံပါတ် : 42
၅ : 25
၂ : 18
၆ : 18
၄ : 16
ခုပါ : 16
၈ : 16
နံပါတ်ကို : 16
၇ : 15
၉ : 14

--------------------------------------------------
TOP

# Final ASR Experiment Comparison

In [ ]:
%%bash

cd /home/thant_syn/kaldi/egs/burmese_asr

echo "=================================================="
echo "STEP 50: FINAL ASR EXPERIMENT COMPARISON"
echo "=================================================="

python3 - <<'PY'

experiments = [
    {
        "system": "Tri1 + Bigram",
        "wer": 124.46,
        "ser": 92.01,
        "s": 1978,
        "d": 531,
        "i": 1852,
        "perfect": "-"
    },
    {
        "system": "Tri2 + Bigram",
        "wer": 103.57,
        "ser": 95.14,
        "s": 1738,
        "d": 1225,
        "i": 666,
        "perfect": "-"
    },
    {
        "system": "Tri3 + Bigram",
        "wer": 96.92,
        "ser": 95.21,
        "s": 1563,
        "d": 1339,
        "i": 494,
        "perfect": 72
    },
    {
        "system": "Tri3 + Trigram",
        "wer": 96.06,
        "ser": 94.87,
        "s": 1554,
        "d": 1345,
        "i": 467,
        "perfect": 77
    },
    {
        "system": "Tri3 + fMLLR + Trigram",
        "wer": 104.39,
        "ser": 92.48,
        "s": 1800,
        "d": 956,
        "i": 902,
        "perfect": 113
    }
]

print()
print("=" * 100)
print("FINAL BURMESE ASR RESULTS")
print("=" * 100)

print(
    f"{'System':<27}"
    f"{'WER':>10}"
    f"{'SER':>10}"
    f"{'Sub':>10}"
    f"{'Del':>10}"
    f"{'Ins':>10}"
    f"{'Perfect':>10}"
)

print("-" * 100)

for x in experiments:
    print(
        f"{x['system']:<27}"
        f"{x['wer']:>9.2f}%"
        f"{x['ser']:>9.2f}%"
        f"{x['s']:>10}"
        f"{x['d']:>10}"
        f"{x['i']:>10}"
        f"{str(x['perfect']):>10}"
    )

print("=" * 100)

# Best WER
best = min(experiments, key=lambda x: x["wer"])

print()
print(f"BEST SYSTEM BY WER : {best['system']}")
print(f"BEST WER           : {best['wer']:.2f}%")

# Improvements
print()
print("KEY IMPROVEMENTS")
print("-" * 60)

print(
    f"Tri1 → Tri2 WER improvement: "
    f"{124.46 - 103.57:.2f} percentage points"
)

print(
    f"Tri2 → Tri3 WER improvement: "
    f"{103.57 - 96.92:.2f} percentage points"
)

print(
    f"Bigram → Trigram improvement: "
    f"{96.92 - 96.06:.2f} percentage points"
)

print(
    f"Trigram → fMLLR change: "
    f"{104.39 - 96.06:+.2f} percentage points"
)

print()
print("=" * 100)
print("INTERPRETATION")
print("=" * 100)

print("""
1. Increasing acoustic-model complexity improved WER:
   Tri1 → Tri2 → Tri3.

2. The largest language-model benefit came from moving from
   the unigram baseline to the bigram model.

3. Trigram modeling produced only a small additional improvement
   over bigram modeling.

4. fMLLR reduced deletions but substantially increased
   substitutions and insertions, resulting in worse WER.

5. Therefore, Tri3 + Trigram is the best-performing system
   among the experiments completed so far.

6. The very high WER indicates that the main remaining challenge
   is acoustic recognition, particularly Burmese words and numbers,
   rather than only language-model complexity.
""")

print("=" * 100)
print("STEP 50 COMPLETE")
print("=" * 100)

PY

STEP 50: FINAL ASR EXPERIMENT COMPARISON

FINAL BURMESE ASR RESULTS
System                            WER       SER       Sub       Del       Ins   Perfect
----------------------------------------------------------------------------------------------------
Tri1 + Bigram                 124.46%    92.01%      1978       531      1852         -
Tri2 + Bigram                 103.57%    95.14%      1738      1225       666         -
Tri3 + Bigram                  96.92%    95.21%      1563      1339       494        72
Tri3 + Trigram                 96.06%    94.87%      1554      1345       467        77
Tri3 + fMLLR + Trigram        104.39%    92.48%      1800       956       902       113

BEST SYSTEM BY WER : Tri3 + Trigram
BEST WER           : 96.06%

KEY IMPROVEMENTS
------------------------------------------------------------
Tri1 → Tri2 WER improvement: 20.89 percentage points
Tri2 → Tri3 WER improvement: 6.65 percentage points
Bigram → Trigram improvement: 0.86 percentage points
T

# References

- https://github.com/ye-kyaw-thu/AIE-F-B2/tree/main/assignment/assignment-4/recording_tool

- [Technical Report: A Practical Guide to Kaldi ASR Optimization](https://arxiv.org/abs/2506.07149)

---